# Extended Pipeline v260430

In [1]:
#!/usr/bin/env python3
"""
=============================================================================
 PSYCHIATRIC GENE-SET DIFFERENTIAL & PROXIMITY PIPELINE (TWAS / S-PrediXcan)
=============================================================================
 User-supplied axis space: any number of diseases / conditions.
 User supplies ONE custom gene set (txt file or comma-separated list),
 OR MULTIPLE gene sets.

 Analyses:
   1. Per-disease enrichment (Stouffer Z, mean |Z|, Wilcoxon vs 0,
      permutation p, bootstrap CIs)
   2. Differential tests across diseases (Kruskal-Wallis + pairwise
      Mann-Whitney + FDR)
   3. Profile proximity / distances between diseases (restricted to
      gene-set genes) + hierarchical clustering dendrogram
   4. Per-gene detail table with Z-scores across all diseases
   5. Pairwise disease statistical tests (gene-set-restricted):
        Unpaired : Mann-Whitney U, KS 2-sample, Welch's t, Levene,
                   Brunner-Munzel, Cohen's d
        Paired   : paired t, Wilcoxon signed-rank, Cohen's d, sign-flip
                   permutation
        Concordance: sign concordance rate + binomial test, Lin's CCC,
                     Kendall's tau, concordance/discordance counts
      with BH-FDR correction across all pairs.
   6. Gene-level leave-one-out (LOO) influence analysis to flag
      results driven by 1-2 outlier genes.
   7. Z-score QQ-plot diagnostics per disease.

 Output: CSVs, summary, boxplot, enrichment bar, proximity heatmap,
         scatter matrix, pairwise-stats heatmap, QQ-plot, dendrogram,
         gene-influence plot, input-provenance manifest.

 CLI usage:
   python geneset_pipeline.py \
       --dirs data/mdd/ data/bip/ data/ocd/ \
       --labels MDD BIP OCD \
       --geneset my_genes.txt --label "Dopamine_Receptors" --out results/

   python geneset_pipeline.py \
       --dirs data/mdd/ data/bip/ data/ocd/ \
       --labels MDD BIP OCD \
       --genesets_file genesets.py --out results/ \
       --random_seed 123 --robust --bootstrap

 Programmatic usage:
   from geneset_pipeline import run_pipeline, run_multi_geneset_pipeline

   results = run_pipeline(
       dirs=["data/mdd/", "data/bip/", "data/ocd/"],
       labels=["MDD", "BIP", "OCD"],
       geneset="DRD1,DRD2,DRD3,DRD4,DRD5,COMT,SLC6A3",
       label="Dopamine_System",
       out="results/",
       random_seed=42,
       robust=False,
       bootstrap=True,
       n_boot=2000,
   )
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from itertools import combinations
import argparse, warnings, os, sys, textwrap, traceback
import hashlib
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime
import ast

warnings.filterwarnings("ignore")


# ===================================================================
#  0.  SMALL UTILITIES (provenance, validation, robust stats)
# ===================================================================

def _sha256_file(path, blocksize=65536):
    """Compute SHA256 of a file (short-form, first 16 hex chars)."""
    try:
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(blocksize), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return "ERROR"


def compute_input_provenance(dirs, geneset_input=None):
    """
    Build a provenance manifest containing SHA256 hashes for every
    input file plus timestamps.  Used for auditability/reproducibility.
    """
    manifest = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "python_version": sys.version.split()[0],
        "platform": sys.platform,
        "input_dirs": {},
        "geneset_input": None,
    }

    for d in dirs:
        d_path = Path(d)
        entry = {"path": str(d_path.resolve()), "exists": d_path.exists(), "files": {}}
        if d_path.is_dir():
            for f in sorted(d_path.iterdir()):
                if f.is_file():
                    entry["files"][f.name] = {
                        "sha256": _sha256_file(f),
                        "size_bytes": f.stat().st_size,
                        "mtime": datetime.fromtimestamp(f.stat().st_mtime).isoformat(timespec="seconds"),
                    }
        manifest["input_dirs"][str(d)] = entry

    if geneset_input is not None:
        try:
            if isinstance(geneset_input, (str, Path)) and Path(geneset_input).is_file():
                p = Path(geneset_input)
                manifest["geneset_input"] = {
                    "type": "file",
                    "path": str(p.resolve()),
                    "sha256": _sha256_file(p),
                    "size_bytes": p.stat().st_size,
                    "mtime": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
                }
            elif isinstance(geneset_input, (list, tuple, set)):
                manifest["geneset_input"] = {
                    "type": "iterable",
                    "n_items": len(list(geneset_input)),
                }
            elif isinstance(geneset_input, dict):
                manifest["geneset_input"] = {
                    "type": "dict",
                    "n_sets": len(geneset_input),
                    "labels": list(map(str, geneset_input.keys())),
                }
            else:
                manifest["geneset_input"] = {"type": "string", "preview": str(geneset_input)[:120]}
        except Exception as e:
            manifest["geneset_input"] = {"type": "unknown", "error": str(e)}
    return manifest


def validate_zscores(meta_z_dict, threshold_extreme=8.0, pct_warn=30.0):
    """
    Sanity check Z-score distributions per disease.

    Flags:
      * very high proportion of |Z| > threshold_extreme
      * duplicated keys (already deduped on load – noted for completeness)

    Returns list of warning strings.
    """
    warnings_list = []
    for d, mz in meta_z_dict.items():
        if not mz:
            warnings_list.append(f"{d}: empty meta-Z dictionary")
            continue
        zs = np.asarray(list(mz.values()), dtype=float)
        zs = zs[np.isfinite(zs)]
        if len(zs) == 0:
            warnings_list.append(f"{d}: no finite Z-scores")
            continue
        n_extreme = int(np.sum(np.abs(zs) > threshold_extreme))
        pct_extreme = 100 * n_extreme / len(zs)
        if pct_extreme > pct_warn:
            warnings_list.append(
                f"{d}: {pct_extreme:.1f}% of |Z| > {threshold_extreme} "
                f"(n={n_extreme}/{len(zs)}) — distribution may be inflated"
            )
        n_huge = int(np.sum(np.abs(zs) > 50))
        if n_huge > 0:
            warnings_list.append(
                f"{d}: {n_huge} extremely large |Z| > 50 detected — verify input"
            )
    return warnings_list


def _winsorize(arr, lower_pct=1.0, upper_pct=99.0):
    """Clip values to the [lower_pct, upper_pct] percentile range."""
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 3:
        return arr
    lo = np.percentile(arr, lower_pct)
    hi = np.percentile(arr, upper_pct)
    return np.clip(arr, lo, hi)


def _trimmed_mean(arr, proportion=0.1):
    """Symmetric trimmed mean."""
    return float(stats.trim_mean(arr, proportion))


# ===================================================================
#  1.  GENERALISED LOADER  (returns meta_z dict, Ensembl↔symbol map,
#                            AND per-tissue Z dicts)
# ===================================================================

def load_spredixcan_folder(folder_path):
    """
    Load S-PrediXcan results from a folder.  Handles CSV / TSV / TXT.
    Reads BOTH gene-symbol and Ensembl-ID columns when available.
    Computes meta-Z across tissue files via Stouffer's method.

    Returns
    -------
    meta_z        : dict  {GENE_SYMBOL: meta_z_score}
    id_map        : dict  {ENSEMBL_ID: GENE_SYMBOL, ENSEMBL_NO_VER: GENE_SYMBOL, …}
    tissue_gene_z : dict  {tissue_filename_stem: {GENE_SYMBOL: z_score, …}, …}
    """
    folder = Path(folder_path)
    files = []
    for ext in ("*.csv", "*.txt", "*.tsv", "*.dat"):
        files.extend(folder.glob(ext))
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError(f"No result files found in {folder_path}")

    print(f"    {len(files)} file(s) in {folder.name}/")

    gene_z = defaultdict(list)   # canonical_name → [z, z, …]
    tissue_gene_z = {}           # tissue_stem   → {canonical_name: z}
    id_map = {}                  # alternative_id  → canonical_name

    for f in files:
        tissue_name = f.stem
        tissue_dict = {}
        parsed = False
        for sep in [",", "\t", r"\s+"]:
            try:
                engine = "python" if sep == r"\s+" else "c"
                df = pd.read_csv(f, sep=sep, engine=engine)
                cols_lower = {c.lower(): c for c in df.columns}

                symbol_col = None
                for candidate in ["gene_name", "genename", "symbol", "gene_symbol"]:
                    if candidate in cols_lower:
                        symbol_col = cols_lower[candidate]
                        break

                id_col = None
                for candidate in ["gene", "gene_id", "geneid", "ensembl_id", "ensembl"]:
                    if candidate in cols_lower:
                        actual = cols_lower[candidate]
                        if actual != symbol_col:
                            id_col = actual
                            break

                primary_col = symbol_col or id_col

                z_col = None
                for candidate in ["zscore", "z_score", "z", "zstat", "z_stat"]:
                    if candidate in cols_lower:
                        z_col = cols_lower[candidate]
                        break

                if not (primary_col and z_col):
                    continue

                keep = list(dict.fromkeys(
                    [c for c in [primary_col, z_col, symbol_col, id_col] if c is not None]
                ))
                sub = df[keep].dropna(subset=[primary_col, z_col])

                # Detect duplicates BEFORE deduping
                n_dup = int(sub.duplicated(subset=primary_col).sum())
                if n_dup > 0:
                    print(f"    ⚠ {f.name}: {n_dup} duplicate '{primary_col}' rows "
                          f"(kept first occurrence)")
                sub = sub.drop_duplicates(subset=primary_col, keep="first")

                for _, row in sub.iterrows():
                    sym, ens = None, None
                    if symbol_col:
                        s = str(row[symbol_col]).strip().upper()
                        if s and s not in ("NAN", "NA", "NONE", ""):
                            sym = s
                    if id_col:
                        e = str(row[id_col]).strip().upper()
                        if e and e not in ("NAN", "NA", "NONE", ""):
                            ens = e

                    canonical = sym or ens
                    if not canonical:
                        continue
                    try:
                        z_val = float(row[z_col])
                        gene_z[canonical].append(z_val)
                        tissue_dict[canonical] = z_val
                    except (ValueError, TypeError):
                        continue

                    if ens and sym:
                        id_map[ens] = sym
                        if "." in ens:
                            id_map[ens.split(".")[0]] = sym

                parsed = True
                break
            except Exception:
                continue

        if parsed and tissue_dict:
            tissue_gene_z[tissue_name] = tissue_dict

        if not parsed:
            print(f"    ⚠ Could not parse {f.name}")

    meta = {}
    for g, zs in gene_z.items():
        if len(zs) > 0:
            meta[g] = np.sum(zs) / np.sqrt(len(zs))

    print(f"    → meta-Z computed for {len(meta):,} genes")
    print(f"    → Ensembl aliases mapped: {len(id_map):,}")
    print(f"    → {len(tissue_gene_z)} tissue file(s) indexed")
    return meta, id_map, tissue_gene_z


# ===================================================================
#  2.  GENE-SET I/O  (tabular-aware loader + Ensembl↔symbol harmoniser)
# ===================================================================

def _try_load_tabular_geneset(filepath):
    for sep in ["\t", ",", r"\s+"]:
        try:
            engine = "python" if sep == r"\s+" else "c"
            df = pd.read_csv(filepath, sep=sep, engine=engine)

            if len(df.columns) < 2:
                continue

            cols_lower = {c.lower(): c for c in df.columns}

            known_cols = {
                "gene_name", "genename", "symbol", "gene_symbol",
                "gene", "gene_id", "zscore", "z_score", "pvalue",
                "effect_size", "n_snps_used", "pred_perf_r2",
            }
            if not any(k in cols_lower for k in known_cols):
                continue

            symbol_col = None
            for c in ["gene_name", "genename", "symbol", "gene_symbol"]:
                if c in cols_lower:
                    symbol_col = cols_lower[c]
                    break

            id_col = None
            for c in ["gene", "gene_id", "geneid", "ensembl_id"]:
                if c in cols_lower:
                    if cols_lower[c] != symbol_col:
                        id_col = cols_lower[c]
                        break

            primary = symbol_col or id_col
            if primary is None:
                continue

            genes = []
            fallback_col = id_col if (symbol_col and id_col) else None

            for _, row in df.iterrows():
                val = str(row[primary]).strip().upper() if pd.notna(row[primary]) else ""
                if val and val not in ("NAN", "NA", "NONE", ""):
                    genes.append(val)
                elif fallback_col is not None:
                    val = str(row[fallback_col]).strip().upper() if pd.notna(row[fallback_col]) else ""
                    if val and val not in ("NAN", "NA", "NONE", ""):
                        genes.append(val)

            if genes:
                print(f"  Loaded {len(genes)} genes from tabular file "
                      f"(column: '{primary}'): {Path(filepath).name}")
                return genes

        except Exception:
            continue
    return None


def load_geneset(geneset_input):
    if isinstance(geneset_input, (list, tuple, set)):
        raw = [str(g).strip().upper() for g in geneset_input if str(g).strip()]
        print(f"  Loaded {len(raw)} genes from Python iterable")

    elif Path(geneset_input).is_file():
        raw = _try_load_tabular_geneset(geneset_input)
        if raw is None:
            with open(geneset_input) as fh:
                raw = [line.strip().upper() for line in fh
                       if line.strip() and not line.startswith("#")]
            print(f"  Loaded {len(raw)} genes from plain-text file: "
                  f"{Path(geneset_input).name}")
    else:
        raw = [g.strip().upper() for g in geneset_input.split(",") if g.strip()]
        print(f"  Loaded {len(raw)} genes from inline string")

    seen = set()
    unique = []
    for g in raw:
        if g not in seen:
            seen.add(g)
            unique.append(g)
    return unique


def load_multiple_genesets(genesets_input):
    if isinstance(genesets_input, dict):
        out = {}
        for label, gs in genesets_input.items():
            out[str(label)] = load_geneset(gs)
        return out

    path = Path(genesets_input)
    if not path.is_file():
        raise FileNotFoundError(f"Gene-set collection file not found: {genesets_input}")

    for sep in ["\t", ","]:
        try:
            df = pd.read_csv(path, sep=sep)
            cols = {c.lower(): c for c in df.columns}
            if "label" in cols and "genes" in cols:
                out = {}
                for _, row in df.iterrows():
                    label = str(row[cols["label"]]).strip()
                    genes = str(row[cols["genes"]]).strip()
                    if label and genes and label.upper() != "NAN" and genes.upper() != "NAN":
                        out[label] = load_geneset(genes)
                if out:
                    return out
        except Exception:
            pass

    text = path.read_text()

    try:
        tree = ast.parse(text, filename=str(path))
        for node in tree.body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == "gene_sets":
                        value = ast.literal_eval(node.value)
                        if isinstance(value, dict):
                            out = {}
                            for label, gs in value.items():
                                out[str(label)] = load_geneset(gs)
                            return out
    except Exception:
        pass

    out = {}
    namespace = {}
    try:
        exec(text, {}, namespace)
        for k, v in namespace.items():
            if k.startswith("_"):
                continue
            if isinstance(v, (list, tuple, set, str)):
                try:
                    out[k] = load_geneset(v)
                except Exception:
                    pass
        if out:
            return out
    except Exception:
        pass

    raise ValueError(
        "Could not parse multiple gene sets. "
        "Use a dict, a Python file with assignments, or a TSV/CSV with columns: label, genes."
    )


def harmonize_geneset(gene_list, meta_z_dict, id_maps, verbose=True):
    _print = print if verbose else lambda *a, **k: None

    all_canonical = set()
    for mz in meta_z_dict.values():
        all_canonical.update(mz.keys())

    merged = {}
    for im in id_maps.values():
        merged.update(im)

    harmonized = []
    direct = 0
    aliased = 0
    missed = []

    for g in gene_list:
        if g in all_canonical:
            harmonized.append(g)
            direct += 1
            continue
        if g in merged and merged[g] in all_canonical:
            harmonized.append(merged[g])
            aliased += 1
            continue
        if "." in g:
            base = g.split(".")[0]
            if base in all_canonical:
                harmonized.append(base)
                aliased += 1
                continue
            if base in merged and merged[base] in all_canonical:
                harmonized.append(merged[base])
                aliased += 1
                continue
        harmonized.append(g)
        missed.append(g)

    seen = set()
    unique = []
    for g in harmonized:
        if g not in seen:
            seen.add(g)
            unique.append(g)

    n_dup = len(harmonized) - len(unique)
    _print(f"  Gene-name harmonisation:")
    _print(f"    direct matches   : {direct}")
    _print(f"    alias-resolved   : {aliased}")
    _print(f"    still unmatched  : {len(missed)}")
    if n_dup:
        _print(f"    duplicates removed after aliasing: {n_dup}")
    if missed and len(missed) <= 20:
        _print(f"    unmatched genes  : {', '.join(missed)}")

    return unique


# ===================================================================
#  3.  ENRICHMENT ANALYSIS  (with bootstrap CIs + robust option)
# ===================================================================

def bootstrap_enrichment(zv, n_boot=2000, rng=None):
    """
    Bootstrap percentile 95% CIs for Stouffer Z, mean Z, mean |Z|.

    Returns dict with *_ci_low / *_ci_high keys (rounded to 4 d.p.).
    Returns NaNs for samples smaller than 3.
    """
    out = {
        "stouffer_ci_low":   np.nan, "stouffer_ci_high":   np.nan,
        "mean_z_ci_low":     np.nan, "mean_z_ci_high":     np.nan,
        "mean_abs_z_ci_low": np.nan, "mean_abs_z_ci_high": np.nan,
        "n_boot": n_boot,
    }
    zv = np.asarray(zv, dtype=float)
    zv = zv[np.isfinite(zv)]
    n = len(zv)
    if n < 3 or n_boot <= 0:
        return out

    if rng is None:
        rng = np.random.default_rng(42)

    boot_stouffer = np.empty(n_boot)
    boot_mean     = np.empty(n_boot)
    boot_mean_abs = np.empty(n_boot)
    sqrtn = np.sqrt(n)
    for i in range(n_boot):
        sample = rng.choice(zv, size=n, replace=True)
        boot_stouffer[i] = np.sum(sample) / sqrtn
        boot_mean[i]     = np.mean(sample)
        boot_mean_abs[i] = np.mean(np.abs(sample))

    out["stouffer_ci_low"]   = round(float(np.percentile(boot_stouffer, 2.5)), 4)
    out["stouffer_ci_high"]  = round(float(np.percentile(boot_stouffer, 97.5)), 4)
    out["mean_z_ci_low"]     = round(float(np.percentile(boot_mean, 2.5)), 4)
    out["mean_z_ci_high"]    = round(float(np.percentile(boot_mean, 97.5)), 4)
    out["mean_abs_z_ci_low"] = round(float(np.percentile(boot_mean_abs, 2.5)), 4)
    out["mean_abs_z_ci_high"]= round(float(np.percentile(boot_mean_abs, 97.5)), 4)
    return out


def gene_set_enrichment(meta_z, gene_list, n_perm=10000,
                        random_seed=42, robust=False,
                        bootstrap=False, n_boot=2000):
    """
    Enrichment statistics for one disease:
      - Stouffer Z
      - Mean |Z|, mean Z, median Z
      - Trimmed mean (always reported), winsorized Stouffer (if robust=True)
      - Wilcoxon signed-rank test vs 0
      - Permutation p-value (skipped if n_perm == 0)
      - Bootstrap 95% CIs (if bootstrap=True)
      - Reliability flags (low_n_warning, low_coverage_warning)
    """
    zv = np.array([meta_z.get(g, np.nan) for g in gene_list])
    zv = zv[~np.isnan(zv)]
    n = len(zv)
    n_total = len(gene_list)
    coverage = round(100 * n / n_total, 1) if n_total else 0

    result = {
        "n_genes_found": n,
        "n_genes_total": n_total,
        "coverage_pct": coverage,
        "stouffer_z": np.nan,
        "stouffer_z_winsorized": np.nan,
        "mean_abs_z": np.nan,
        "mean_z": np.nan,
        "trimmed_mean_z": np.nan,
        "median_z": np.nan,
        "wilcoxon_p": np.nan,
        "permutation_p": np.nan,
        "z_scores": [],
        # Bootstrap fields
        "stouffer_ci_low":   np.nan, "stouffer_ci_high":   np.nan,
        "mean_z_ci_low":     np.nan, "mean_z_ci_high":     np.nan,
        "mean_abs_z_ci_low": np.nan, "mean_abs_z_ci_high": np.nan,
        # Reliability flags
        "low_n_warning":         bool(n < 5),
        "low_coverage_warning":  bool(coverage < 60.0 and n_total > 0),
        "wilcoxon_unreliable":   bool(n < 6),
        "robust_used":           bool(robust),
    }

    if n < 3:
        return result

    # Optional winsorization for robust mode
    zv_for_stats = _winsorize(zv) if robust else zv

    stouffer = np.sum(zv_for_stats) / np.sqrt(n)
    result["stouffer_z"] = round(float(stouffer), 4)

    # Always compute winsorized Stouffer for reference
    zv_w = _winsorize(zv)
    result["stouffer_z_winsorized"] = round(float(np.sum(zv_w) / np.sqrt(n)), 4)

    result["mean_abs_z"]    = round(float(np.mean(np.abs(zv_for_stats))), 4)
    result["mean_z"]        = round(float(np.mean(zv_for_stats)), 4)
    result["median_z"]      = round(float(np.median(zv)), 4)
    try:
        result["trimmed_mean_z"] = round(_trimmed_mean(zv, proportion=0.1), 4)
    except Exception:
        pass
    result["z_scores"] = zv.tolist()

    if n >= 6:
        try:
            _, wil_p = stats.wilcoxon(zv, alternative="two-sided")
            result["wilcoxon_p"] = round(float(wil_p), 6)
        except Exception:
            pass

    all_z = np.array(list(meta_z.values()))
    if len(all_z) > n and n_perm > 0:
        obs_stat = abs(stouffer)
        rng_perm = np.random.default_rng(random_seed)
        count = 0
        for _ in range(n_perm):
            perm_z = rng_perm.choice(all_z, size=n, replace=False)
            if robust:
                perm_z = _winsorize(perm_z)
            if abs(np.sum(perm_z) / np.sqrt(n)) >= obs_stat:
                count += 1
        result["permutation_p"] = round((count + 1) / (n_perm + 1), 6)

    if bootstrap and n_boot > 0:
        rng_boot = np.random.default_rng(random_seed + 1)
        ci = bootstrap_enrichment(zv_for_stats, n_boot=n_boot, rng=rng_boot)
        for k in ("stouffer_ci_low", "stouffer_ci_high",
                  "mean_z_ci_low", "mean_z_ci_high",
                  "mean_abs_z_ci_low", "mean_abs_z_ci_high"):
            result[k] = ci[k]

    return result


# ===================================================================
#  3b.  GENE-LEVEL INFLUENCE ANALYSIS (LOO / jackknife)
# ===================================================================

def gene_influence_analysis(meta_z_dict, gene_list, threshold_pct=20.0):
    """
    Leave-one-out (LOO) influence analysis on the per-disease Stouffer Z
    and mean |Z|.

    For each disease and each gene present in the data, recompute the
    enrichment statistic without that gene and record the delta.

    A gene is flagged as "Influential" if removing it:
      - flips the sign of the Stouffer Z, OR
      - changes |Stouffer Z| by more than `threshold_pct` percent.

    Returns a tidy DataFrame (one row per gene × disease).
    """
    rows = []
    diseases = list(meta_z_dict.keys())
    for d in diseases:
        # Build the in-data Z vector for this disease
        zv = []
        gene_in_data = []
        for g in gene_list:
            z = meta_z_dict[d].get(g, np.nan)
            if not np.isnan(z):
                zv.append(z)
                gene_in_data.append(g)
        zv = np.asarray(zv, dtype=float)
        n = len(zv)
        if n < 4:
            continue

        full_stouffer = np.sum(zv) / np.sqrt(n)
        full_mean_abs = np.mean(np.abs(zv))

        for i, g in enumerate(gene_in_data):
            zv_loo = np.delete(zv, i)
            n_loo = len(zv_loo)
            loo_stouffer = np.sum(zv_loo) / np.sqrt(n_loo)
            loo_mean_abs = np.mean(np.abs(zv_loo))

            d_stouffer = full_stouffer - loo_stouffer
            d_mean_abs = full_mean_abs - loo_mean_abs

            sign_change = (np.sign(full_stouffer) != np.sign(loo_stouffer)
                           and full_stouffer != 0 and loo_stouffer != 0)
            pct_change = (abs(d_stouffer) / abs(full_stouffer) * 100
                          if full_stouffer != 0 else np.nan)
            influential = bool(sign_change or
                               (not np.isnan(pct_change) and pct_change > threshold_pct))

            rows.append({
                "Disease": d,
                "Gene": g,
                "Z_score": round(float(zv[i]), 4),
                "Full_Stouffer_Z": round(float(full_stouffer), 4),
                "LOO_Stouffer_Z":  round(float(loo_stouffer), 4),
                "Delta_Stouffer":  round(float(d_stouffer), 4),
                "Pct_Change_Stouffer": (round(float(pct_change), 2)
                                        if not np.isnan(pct_change) else np.nan),
                "Sign_Change": bool(sign_change),
                "Full_Mean_AbsZ":  round(float(full_mean_abs), 4),
                "LOO_Mean_AbsZ":   round(float(loo_mean_abs), 4),
                "Delta_Mean_AbsZ": round(float(d_mean_abs), 4),
                "Influential": influential,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["Disease", "Pct_Change_Stouffer"],
                            ascending=[True, False]).reset_index(drop=True)
    return df


# ===================================================================
#  4.  DIFFERENTIAL TESTS
# ===================================================================

def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.array([])
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    adjusted = np.empty(n)
    cum_min = 1.0
    for i in range(n - 1, -1, -1):
        adj = sorted_p[i] * n / (i + 1)
        cum_min = min(cum_min, adj)
        adjusted[order[i]] = min(cum_min, 1.0)
    return adjusted


def differential_tests(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    common = [g for g in gene_list if all(g in meta_z_dict[d] for d in diseases)]

    result = {
        "n_common_genes": len(common),
        "common_genes": common,
        "kruskal_h": np.nan,
        "kruskal_p": np.nan,
        "pairwise_df": pd.DataFrame()
    }

    if len(common) < 3:
        print("  ⚠ Fewer than 3 genes overlap across all diseases – skipping differential tests")
        return result

    z_by_disease = {d: np.array([meta_z_dict[d][g] for g in common]) for d in diseases}

    h_stat, kw_p = stats.kruskal(*z_by_disease.values())
    result["kruskal_h"] = round(h_stat, 4)
    result["kruskal_p"] = round(kw_p, 6)

    rows = []
    for d1, d2 in combinations(diseases, 2):
        v1, v2 = z_by_disease[d1], z_by_disease[d2]
        u_stat, mwu_p = stats.mannwhitneyu(v1, v2, alternative="two-sided")
        n1, n2 = len(v1), len(v2)
        rbc = 1 - (2 * u_stat) / (n1 * n2)

        try:
            _, paired_p = stats.wilcoxon(v1 - v2, alternative="two-sided")
            paired_p = round(paired_p, 6)
        except Exception:
            paired_p = np.nan

        rows.append({
            "Disease_1": d1,
            "Disease_2": d2,
            "MWU_U": int(u_stat),
            "MWU_p": round(mwu_p, 6),
            "Rank_Biserial_r": round(rbc, 4),
            "Paired_Wilcoxon_p": paired_p,
            "Mean_Z_diff": round(np.mean(v1) - np.mean(v2), 4)
        })

    pw_df = pd.DataFrame(rows)
    if len(pw_df) > 0:
        pw_df["MWU_FDR"] = benjamini_hochberg(pw_df["MWU_p"].values)
        pw_p = pw_df["Paired_Wilcoxon_p"].values
        if not np.any(np.isnan(pw_p)):
            pw_df["Paired_FDR"] = benjamini_hochberg(pw_p)
    result["pairwise_df"] = pw_df
    return result


# ===================================================================
#  4b.  PAIRWISE DISEASE STATISTICAL TESTS  (gene-set-restricted)
# ===================================================================

def _lins_ccc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3:
        return np.nan
    mx, my = np.mean(x), np.mean(y)
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    if sx == 0 or sy == 0:
        return np.nan
    r, _ = stats.pearsonr(x, y)
    numerator = 2 * r * sx * sy
    denominator = sx ** 2 + sy ** 2 + (mx - my) ** 2
    if denominator == 0:
        return np.nan
    return numerator / denominator


def pairwise_disease_geneset_tests(meta_z_dict, gene_list,
                                   n_perm=5000, random_seed=42, robust=False):
    """
    Comprehensive pairwise statistical tests between diseases,
    restricted to gene-set genes.

    With `robust=True`, Z-vectors are winsorized (1st/99th percentile)
    before parametric (t-test, Cohen's d, Levene) computations.
    Non-parametric tests are unaffected by winsorization.
    """
    diseases = list(meta_z_dict.keys())
    if len(diseases) < 2:
        return pd.DataFrame()

    rows = []
    for d1, d2 in combinations(diseases, 2):
        z1_all = np.array([meta_z_dict[d1][g] for g in gene_list
                           if g in meta_z_dict[d1]
                           and np.isfinite(meta_z_dict[d1][g])])
        z2_all = np.array([meta_z_dict[d2][g] for g in gene_list
                           if g in meta_z_dict[d2]
                           and np.isfinite(meta_z_dict[d2][g])])

        common = [g for g in gene_list
                  if g in meta_z_dict[d1] and g in meta_z_dict[d2]
                  and np.isfinite(meta_z_dict[d1][g])
                  and np.isfinite(meta_z_dict[d2][g])]
        z1_p = np.array([meta_z_dict[d1][g] for g in common])
        z2_p = np.array([meta_z_dict[d2][g] for g in common])

        # Optional robust transformation for parametric tests
        z1_all_r = _winsorize(z1_all) if robust else z1_all
        z2_all_r = _winsorize(z2_all) if robust else z2_all
        z1_p_r   = _winsorize(z1_p)   if robust else z1_p
        z2_p_r   = _winsorize(z2_p)   if robust else z2_p

        row = {
            "Disease_1": d1,
            "Disease_2": d2,
            "n_genes_D1": len(z1_all),
            "n_genes_D2": len(z2_all),
            "n_genes_paired": len(common),
            "mean_Z_D1":   round(float(np.mean(z1_all)),   4) if len(z1_all) else np.nan,
            "mean_Z_D2":   round(float(np.mean(z2_all)),   4) if len(z2_all) else np.nan,
            "sd_Z_D1":     round(float(np.std(z1_all, ddof=1)), 4) if len(z1_all) > 1 else np.nan,
            "sd_Z_D2":     round(float(np.std(z2_all, ddof=1)), 4) if len(z2_all) > 1 else np.nan,
            "median_Z_D1": round(float(np.median(z1_all)), 4) if len(z1_all) else np.nan,
            "median_Z_D2": round(float(np.median(z2_all)), 4) if len(z2_all) else np.nan,
        }

        if len(common) >= 1:
            diffs = z1_p - z2_p
            diffs_r = z1_p_r - z2_p_r
            row["mean_diff"]   = round(float(np.mean(diffs)),   4)
            row["median_diff"] = round(float(np.median(diffs)), 4)
            row["sd_diff"]     = round(float(np.std(diffs, ddof=1)), 4) if len(common) > 1 else np.nan
        else:
            diffs = np.array([])
            diffs_r = np.array([])
            row["mean_diff"]   = np.nan
            row["median_diff"] = np.nan
            row["sd_diff"]     = np.nan

        _enough_unpaired = len(z1_all) >= 3 and len(z2_all) >= 3

        row["MWU_U"] = np.nan; row["MWU_p"] = np.nan; row["Rank_Biserial_r"] = np.nan
        if _enough_unpaired:
            try:
                u, p = stats.mannwhitneyu(z1_all, z2_all, alternative="two-sided")
                n1, n2 = len(z1_all), len(z2_all)
                row["MWU_U"] = int(u)
                row["MWU_p"] = round(p, 6)
                row["Rank_Biserial_r"] = round(1 - 2 * u / (n1 * n2), 4)
            except Exception:
                pass

        row["KS_stat"] = np.nan; row["KS_p"] = np.nan
        if _enough_unpaired:
            try:
                ks, p = stats.ks_2samp(z1_all, z2_all)
                row["KS_stat"] = round(ks, 4)
                row["KS_p"]    = round(p, 6)
            except Exception:
                pass

        row["Welch_t"] = np.nan; row["Welch_p"] = np.nan
        if _enough_unpaired:
            try:
                t, p = stats.ttest_ind(z1_all_r, z2_all_r, equal_var=False)
                row["Welch_t"] = round(t, 4)
                row["Welch_p"] = round(p, 6)
            except Exception:
                pass

        row["Levene_W"] = np.nan; row["Levene_p"] = np.nan
        if _enough_unpaired:
            try:
                w, p = stats.levene(z1_all_r, z2_all_r)
                row["Levene_W"] = round(w, 4)
                row["Levene_p"] = round(p, 6)
            except Exception:
                pass

        row["BM_W"] = np.nan; row["BM_p"] = np.nan
        if _enough_unpaired:
            try:
                bm, p = stats.brunnermunzel(z1_all, z2_all)
                row["BM_W"] = round(bm, 4)
                row["BM_p"] = round(p, 6)
            except Exception:
                pass

        row["Cohens_d_unpaired"] = np.nan
        if _enough_unpaired:
            try:
                s1 = np.var(z1_all_r, ddof=1)
                s2 = np.var(z2_all_r, ddof=1)
                pooled = np.sqrt((s1 + s2) / 2)
                if pooled > 0:
                    row["Cohens_d_unpaired"] = round(
                        float((np.mean(z1_all_r) - np.mean(z2_all_r)) / pooled), 4
                    )
            except Exception:
                pass

        row["Paired_t_stat"] = np.nan; row["Paired_t_p"] = np.nan
        row["Paired_Wilcoxon_W"] = np.nan; row["Paired_Wilcoxon_p"] = np.nan
        row["Cohens_d_paired"] = np.nan
        row["Permutation_p"] = np.nan

        if len(common) >= 3:
            try:
                t, p = stats.ttest_rel(z1_p_r, z2_p_r)
                row["Paired_t_stat"] = round(t, 4)
                row["Paired_t_p"]    = round(p, 6)
            except Exception:
                pass

            if len(common) >= 6:
                try:
                    w, p = stats.wilcoxon(diffs, alternative="two-sided")
                    row["Paired_Wilcoxon_W"] = int(w)
                    row["Paired_Wilcoxon_p"] = round(p, 6)
                except Exception:
                    pass

            try:
                sd_d = np.std(diffs_r, ddof=1)
                if sd_d > 0:
                    row["Cohens_d_paired"] = round(float(np.mean(diffs_r) / sd_d), 4)
            except Exception:
                pass

            if n_perm > 0:
                try:
                    obs = abs(np.mean(diffs))
                    rng = np.random.default_rng(random_seed)
                    count = 0
                    for _ in range(n_perm):
                        signs = rng.choice([-1, 1], size=len(diffs))
                        if abs(np.mean(diffs * signs)) >= obs:
                            count += 1
                    row["Permutation_p"] = round((count + 1) / (n_perm + 1), 6)
                except Exception:
                    pass

        row["n_concordant"]          = np.nan
        row["n_discordant"]          = np.nan
        row["n_zero_either"]         = np.nan
        row["sign_concordance_rate"] = np.nan
        row["sign_concordance_p"]    = np.nan
        row["Lin_CCC"]               = np.nan
        row["Kendall_tau"]           = np.nan
        row["Kendall_p"]             = np.nan

        if len(common) >= 3:
            try:
                signs1 = np.sign(z1_p)
                signs2 = np.sign(z2_p)
                zero_mask = (signs1 == 0) | (signs2 == 0)
                n_zero = int(np.sum(zero_mask))
                row["n_zero_either"] = n_zero
                nz_mask = ~zero_mask
                s1_nz = signs1[nz_mask]
                s2_nz = signs2[nz_mask]
                n_nz = len(s1_nz)
                if n_nz >= 1:
                    concordant_mask = (s1_nz == s2_nz)
                    n_conc = int(np.sum(concordant_mask))
                    n_disc = n_nz - n_conc
                    row["n_concordant"] = n_conc
                    row["n_discordant"] = n_disc
                    row["sign_concordance_rate"] = round(n_conc / n_nz, 4)
                    if n_nz >= 3:
                        try:
                            binom_p = stats.binomtest(n_conc, n_nz, 0.5).pvalue
                        except AttributeError:
                            binom_p = stats.binom_test(n_conc, n_nz, 0.5)
                        row["sign_concordance_p"] = round(float(binom_p), 6)
            except Exception:
                pass

            try:
                ccc = _lins_ccc(z1_p, z2_p)
                if not np.isnan(ccc):
                    row["Lin_CCC"] = round(float(ccc), 4)
            except Exception:
                pass

            try:
                tau, tau_p = stats.kendalltau(z1_p, z2_p)
                row["Kendall_tau"] = round(float(tau), 4)
                row["Kendall_p"]   = round(float(tau_p), 6)
            except Exception:
                pass

        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    p_cols = [c for c in df.columns if c.endswith("_p")]
    for pc in p_cols:
        vals = df[pc].values.astype(float)
        mask = ~np.isnan(vals)
        if mask.sum() > 1:
            fdr_vals = np.full(len(vals), np.nan)
            fdr_vals[mask] = benjamini_hochberg(vals[mask])
            df[pc.replace("_p", "_FDR")] = fdr_vals
        elif mask.sum() == 1:
            df[pc.replace("_p", "_FDR")] = vals

    return df


# ===================================================================
#  5.  PROXIMITY / DISTANCE
# ===================================================================

def profile_proximity(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    common = [g for g in gene_list if all(g in meta_z_dict[d] for d in diseases)]

    if len(common) < 5:
        print("  ⚠ Fewer than 5 overlapping genes – proximity metrics unreliable")
    if len(common) < 3:
        return pd.DataFrame(), pd.DataFrame()

    profile_matrix = pd.DataFrame(
        {d: [meta_z_dict[d][g] for g in common] for d in diseases},
        index=common
    )

    rows = []
    for d1, d2 in combinations(diseases, 2):
        v1 = profile_matrix[d1].values
        v2 = profile_matrix[d2].values

        pr, pr_p = stats.pearsonr(v1, v2)
        sr, sr_p = stats.spearmanr(v1, v2)
        cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-15)
        eucl = np.linalg.norm(v1 - v2)
        manhattan = np.sum(np.abs(v1 - v2))

        rows.append({
            "Disease_1": d1,
            "Disease_2": d2,
            "Pearson_r": round(pr, 4),
            "Pearson_p": round(pr_p, 6),
            "Spearman_rho": round(sr, 4),
            "Spearman_p": round(sr_p, 6),
            "Cosine_sim": round(cos_sim, 4),
            "Euclidean_dist": round(eucl, 4),
            "Manhattan_dist": round(manhattan, 4),
            "n_genes": len(common)
        })

    return pd.DataFrame(rows), profile_matrix


# ===================================================================
#  6.  PER-GENE DETAIL TABLE
# ===================================================================

def build_gene_detail_table(meta_z_dict, gene_list):
    diseases = list(meta_z_dict.keys())
    rows = []
    for g in gene_list:
        row = {"Gene": g}
        zvals = []
        for d in diseases:
            z = meta_z_dict[d].get(g, np.nan)
            row[f"Z_{d}"] = round(z, 4) if not np.isnan(z) else np.nan
            if not np.isnan(z):
                zvals.append(z)
        row["N_diseases"] = len(zvals)
        if len(zvals) >= 2:
            row["Z_range"] = round(max(zvals) - min(zvals), 4)
            row["Z_mean"] = round(np.mean(zvals), 4)
        else:
            row["Z_range"] = np.nan
            row["Z_mean"] = np.nan
        rows.append(row)

    df = pd.DataFrame(rows)
    sort_cols = [f"Z_{d}" for d in diseases]
    df["_sort"] = df[sort_cols].abs().mean(axis=1)
    df = df.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index(drop=True)
    return df


# ===================================================================
#  7.  VISUALISATION
# ===================================================================

def _make_disease_colors(diseases):
    _LEGACY = {"MDD": "#e74c3c", "BIP": "#3498db", "OCD": "#2ecc71"}
    pal = sns.color_palette("husl", max(len(diseases), 3))
    colors = {}
    pal_idx = 0
    for d in diseases:
        if d in _LEGACY:
            colors[d] = _LEGACY[d]
        else:
            colors[d] = pal[pal_idx]
            pal_idx += 1
    return colors


def plot_zscore_boxplot(enrichment_dict, label, out_dir):
    data = []
    for dis, info in enrichment_dict.items():
        for z in info["z_scores"]:
            data.append({"Disease": dis, "meta-Z": z})
    if not data:
        return
    df = pd.DataFrame(data)

    order = list(enrichment_dict.keys())
    dcol = _make_disease_colors(order)
    colors = [dcol.get(d, "#999") for d in order]

    fig, ax = plt.subplots(figsize=(max(7, len(order) * 2), 5))

    sns.boxplot(data=df, x="Disease", y="meta-Z", order=order,
                palette=colors, width=0.5, ax=ax)
    sns.stripplot(data=df, x="Disease", y="meta-Z", order=order,
                  color=".25", size=3, alpha=0.6, ax=ax)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"Gene-set meta-Z Distribution – {label}", fontsize=13)
    ax.set_ylabel("meta-Z (TWAS association)")

    for i, d in enumerate(order):
        n = enrichment_dict.get(d, {}).get("n_genes_found", 0)
        ax.text(i, ax.get_ylim()[0], f"n={n}", ha="center", va="top",
                fontsize=9, color="gray")

    plt.tight_layout()
    path = os.path.join(out_dir, f"zscore_boxplot_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_enrichment_bar(enrichment_dict, label, out_dir):
    order = list(enrichment_dict.keys())
    dcol = _make_disease_colors(order)

    rows = []
    for d in order:
        rows.append({
            "Disease": d,
            "Stouffer Z": enrichment_dict[d].get("stouffer_z", 0) or 0,
            "Mean |Z|":   enrichment_dict[d].get("mean_abs_z", 0) or 0,
            "Stouffer_lo": enrichment_dict[d].get("stouffer_ci_low", np.nan),
            "Stouffer_hi": enrichment_dict[d].get("stouffer_ci_high", np.nan),
            "MeanAbs_lo":  enrichment_dict[d].get("mean_abs_z_ci_low", np.nan),
            "MeanAbs_hi":  enrichment_dict[d].get("mean_abs_z_ci_high", np.nan),
        })
    if not rows:
        return
    df = pd.DataFrame(rows)
    colors = [dcol.get(d, "#999") for d in df["Disease"]]

    fig, axes = plt.subplots(1, 2, figsize=(max(10, len(order) * 2.5), 5))

    bars0 = axes[0].bar(df["Disease"], df["Stouffer Z"], color=colors, edgecolor="white")
    # Draw bootstrap CI error bars when available
    if df["Stouffer_lo"].notna().any():
        for b, lo, hi, val in zip(bars0, df["Stouffer_lo"], df["Stouffer_hi"], df["Stouffer Z"]):
            if not (np.isnan(lo) or np.isnan(hi)):
                axes[0].plot([b.get_x() + b.get_width() / 2] * 2, [lo, hi],
                             color="black", lw=1.2)
                axes[0].plot([b.get_x() + b.get_width() / 2 - 0.08,
                              b.get_x() + b.get_width() / 2 + 0.08], [lo, lo],
                             color="black", lw=1.2)
                axes[0].plot([b.get_x() + b.get_width() / 2 - 0.08,
                              b.get_x() + b.get_width() / 2 + 0.08], [hi, hi],
                             color="black", lw=1.2)

    axes[0].axhline(0, color="gray", ls="--", lw=0.8)
    axes[0].axhline(1.96, color="red", ls=":", lw=0.7, alpha=0.5)
    axes[0].axhline(-1.96, color="red", ls=":", lw=0.7, alpha=0.5)
    axes[0].set_title("Stouffer Z (with 95% bootstrap CI if available)")
    axes[0].set_ylabel("Combined Z-score")

    bars1 = axes[1].bar(df["Disease"], df["Mean |Z|"], color=colors, edgecolor="white")
    if df["MeanAbs_lo"].notna().any():
        for b, lo, hi in zip(bars1, df["MeanAbs_lo"], df["MeanAbs_hi"]):
            if not (np.isnan(lo) or np.isnan(hi)):
                axes[1].plot([b.get_x() + b.get_width() / 2] * 2, [lo, hi],
                             color="black", lw=1.2)
    axes[1].set_title("Mean |Z|")
    axes[1].set_ylabel("Mean absolute Z-score")

    fig.suptitle(f"Gene-set Enrichment – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"enrichment_bar_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_proximity_heatmap(prox_df, diseases, label, out_dir):
    if prox_df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(max(12, len(diseases) * 3), max(5, len(diseases) * 2)))
    for ax, metric, title in zip(axes,
                                  ["Pearson_r", "Cosine_sim"],
                                  ["Pearson r", "Cosine Similarity"]):
        mat = pd.DataFrame(np.eye(len(diseases)), index=diseases, columns=diseases)
        for _, row in prox_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[metric]
            if d1 in diseases and d2 in diseases:
                mat.loc[d1, d2] = val
                mat.loc[d2, d1] = val

        sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap="RdBu_r",
                    center=0, vmin=-1, vmax=1, square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(title)

    fig.suptitle(f"Profile Proximity – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"proximity_heatmap_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_proximity_dendrogram(prox_df, diseases, label, out_dir):
    """
    Hierarchical clustering of diseases based on (1 - Pearson r) distance.
    Requires at least 3 diseases.
    """
    if prox_df.empty or len(diseases) < 3:
        return
    try:
        from scipy.cluster.hierarchy import linkage, dendrogram
        from scipy.spatial.distance import squareform
    except Exception as e:
        print(f"  ⚠ scipy clustering unavailable: {e}")
        return

    n = len(diseases)
    dmat = np.zeros((n, n))
    idx = {d: i for i, d in enumerate(diseases)}
    for _, row in prox_df.iterrows():
        d1, d2 = row["Disease_1"], row["Disease_2"]
        if d1 in idx and d2 in idx:
            dist = max(0.0, 1 - float(row["Pearson_r"]))
            dmat[idx[d1], idx[d2]] = dist
            dmat[idx[d2], idx[d1]] = dist

    try:
        condensed = squareform(dmat, checks=False)
        Z = linkage(condensed, method="average")
        fig, ax = plt.subplots(figsize=(max(6, n * 1.3), 5))
        dendrogram(Z, labels=diseases, ax=ax, color_threshold=0)
        ax.set_ylabel("Distance (1 − Pearson r)")
        ax.set_title(f"Disease Hierarchical Clustering – {label}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        path = os.path.join(out_dir, f"proximity_dendrogram_{label}.png")
        plt.savefig(path, dpi=300)
        plt.close()
        print(f"  → {path}")
    except Exception as e:
        print(f"  ⚠ Dendrogram failed: {e}")


def plot_zscore_qq(meta_z_dict, gene_list, label, out_dir):
    """
    Per-disease QQ-plot: gene-set Z-scores vs N(0,1) theoretical quantiles.
    Useful sanity check for calibration.
    """
    diseases = list(meta_z_dict.keys())
    n = len(diseases)
    if n == 0:
        return
    fig, axes = plt.subplots(1, n, figsize=(max(5 * n, 6), 5))
    if n == 1:
        axes = [axes]
    for ax, d in zip(axes, diseases):
        gs_z = np.array([meta_z_dict[d].get(g, np.nan) for g in gene_list])
        gs_z = gs_z[~np.isnan(gs_z)]
        if len(gs_z) < 3:
            ax.text(0.5, 0.5, f"{d}: n<3", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11)
            ax.set_xticks([]); ax.set_yticks([])
            continue
        observed = np.sort(gs_z)
        theoretical = stats.norm.ppf((np.arange(1, len(observed) + 1) - 0.5) / len(observed))
        ax.scatter(theoretical, observed, alpha=0.7, s=30,
                   c="steelblue", edgecolor="black", lw=0.3)
        lims = [min(theoretical.min(), observed.min()),
                max(theoretical.max(), observed.max())]
        ax.plot(lims, lims, "r--", lw=1, label="y = x (N(0,1))")
        ax.axhline(0, color="gray", lw=0.5, ls=":")
        ax.axvline(0, color="gray", lw=0.5, ls=":")
        ax.set_xlabel("Theoretical N(0,1) quantile")
        ax.set_ylabel("Observed gene-set meta-Z")
        ax.set_title(f"{d} (n={len(observed)})", fontsize=11)
        ax.legend(fontsize=8, loc="upper left")
    fig.suptitle(f"Z-score QQ-plots (gene set vs N(0,1)) – {label}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"qq_plot_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_gene_influence(influence_df, label, out_dir, top_n=15):
    """
    Per-disease lollipop / bar plot of the top-|ΔStouffer| genes.
    Influential genes (sign-flip or > threshold% change) are highlighted in red.
    """
    if influence_df.empty:
        return
    diseases = list(influence_df["Disease"].unique())
    n = len(diseases)
    fig, axes = plt.subplots(1, n,
        figsize=(max(6 * n, 8), max(5, top_n * 0.35)))
    if n == 1:
        axes = [axes]
    for ax, d in zip(axes, diseases):
        sub = influence_df[influence_df["Disease"] == d].copy()
        sub = sub.reindex(sub["Delta_Stouffer"].abs().sort_values(ascending=False).index)
        sub = sub.head(top_n).iloc[::-1]
        if sub.empty:
            ax.text(0.5, 0.5, f"{d}: insufficient data",
                    ha="center", va="center", transform=ax.transAxes)
            continue
        colors = ["#e74c3c" if x else "#3498db" for x in sub["Influential"]]
        ax.barh(sub["Gene"], sub["Delta_Stouffer"], color=colors,
                edgecolor="black", lw=0.3)
        ax.axvline(0, color="gray", ls="--", lw=0.8)
        ax.set_xlabel("Δ Stouffer Z (full − LOO)")
        ax.set_title(f"{d}: top influential genes", fontsize=11)
    fig.suptitle(f"Gene-level Leave-One-Out Influence – {label}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"gene_influence_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_scatter_matrix(profile_matrix, label, out_dir):
    if profile_matrix.empty:
        return

    diseases = list(profile_matrix.columns)
    pairs = list(combinations(diseases, 2))
    n_pairs = len(pairs)
    if n_pairs == 0:
        return

    fig, axes = plt.subplots(1, n_pairs, figsize=(5 * n_pairs, 5))
    if n_pairs == 1:
        axes = [axes]

    for ax, (d1, d2) in zip(axes, pairs):
        v1 = profile_matrix[d1].values
        v2 = profile_matrix[d2].values
        ax.scatter(v1, v2, alpha=0.5, s=20, color="#555")

        slope, intercept = np.polyfit(v1, v2, 1)
        xr = np.linspace(v1.min(), v1.max(), 100)
        ax.plot(xr, slope * xr + intercept, color="red", lw=1.5, ls="--")

        r, _ = stats.pearsonr(v1, v2)
        ax.set_xlabel(f"{d1} meta-Z")
        ax.set_ylabel(f"{d2} meta-Z")
        ax.set_title(f"{d1} vs {d2}  (r = {r:.3f})")
        ax.axhline(0, color="gray", ls=":", lw=0.5)
        ax.axvline(0, color="gray", ls=":", lw=0.5)

    fig.suptitle(f"Cross-Disease Z-score Scatter – {label}", fontsize=14)
    plt.tight_layout()
    path = os.path.join(out_dir, f"scatter_matrix_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


def plot_pairwise_stats_heatmap(pw_df, diseases, label, out_dir):
    if pw_df.empty or len(diseases) < 2:
        return

    p_candidates = [
        ("KS_p",               "KS 2-sample"),
        ("Welch_p",            "Welch t"),
        ("Paired_Wilcoxon_p",  "Paired Wilcoxon"),
        ("Paired_t_p",         "Paired t"),
        ("Permutation_p",      "Permutation"),
        ("sign_concordance_p", "Sign Concordance"),
        ("Kendall_p",          "Kendall τ"),
    ]
    p_metrics = [(col, lbl) for col, lbl in p_candidates
                 if col in pw_df.columns and pw_df[col].notna().any()]

    d_candidates = [
        ("Cohens_d_unpaired", "Cohen's d (unpaired)"),
        ("Cohens_d_paired",   "Cohen's d (paired)"),
    ]
    d_metrics = [(col, lbl) for col, lbl in d_candidates
                 if col in pw_df.columns and pw_df[col].notna().any()]

    conc_candidates = [
        ("sign_concordance_rate", "Sign Concordance Rate"),
        ("Lin_CCC",               "Lin's CCC"),
        ("Kendall_tau",            "Kendall τ"),
    ]
    conc_metrics = [(col, lbl) for col, lbl in conc_candidates
                    if col in pw_df.columns and pw_df[col].notna().any()]

    n_panels = len(p_metrics) + len(d_metrics) + len(conc_metrics)
    if n_panels == 0:
        return

    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
    )
    if n_panels == 1:
        axes = [axes]

    panel_idx = 0

    for col, lbl in p_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                neg_log = -np.log10(max(val, 1e-300))
                mat.loc[d1, d2] = neg_log
                mat.loc[d2, d1] = neg_log
        for d in diseases:
            mat.loc[d, d] = 0.0

        sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                    square=True, ax=ax, linewidths=0.5, linecolor="white",
                    vmin=0)
        ax.set_title(f"−log₁₀(p)\n{lbl}", fontsize=10)

    for col, lbl in d_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        mat = pd.DataFrame(0.0, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                mat.loc[d1, d2] =  val
                mat.loc[d2, d1] = -val

        vabs = max(abs(mat.values.min()), abs(mat.values.max()), 0.01)
        sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
                    center=0, vmin=-vabs, vmax=vabs, square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(lbl, fontsize=10)

    for col, lbl in conc_metrics:
        ax = axes[panel_idx]; panel_idx += 1
        if col == "sign_concordance_rate":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = 0.0, 1.0, 0.5
            cmap = "YlGn"
        elif col == "Lin_CCC":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"
        elif col == "Kendall_tau":
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"
        else:
            diag_val = 1.0
            vmin_val, vmax_val, center_val = -1.0, 1.0, 0.0
            cmap = "RdBu_r"

        mat = pd.DataFrame(diag_val, index=diseases, columns=diseases)
        for _, row in pw_df.iterrows():
            d1, d2 = row["Disease_1"], row["Disease_2"]
            val = row[col]
            if d1 in diseases and d2 in diseases and not np.isnan(val):
                mat.loc[d1, d2] = val
                mat.loc[d2, d1] = val

        sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap=cmap,
                    center=center_val, vmin=vmin_val, vmax=vmax_val,
                    square=True, ax=ax,
                    linewidths=0.5, linecolor="white")
        ax.set_title(lbl, fontsize=10)

    fig.suptitle(f"Pairwise Disease Tests – {label}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(out_dir, f"pairwise_stats_heatmap_{label}.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"  → {path}")


# ===================================================================
#  8.  CORE PIPELINE FUNCTION (callable directly)
# ===================================================================

# Static block appended to every summary file
_LIMITATIONS_BLOCK = textwrap.dedent("""\
    LIMITATIONS & ASSUMPTIONS
    -------------------------
    * Stouffer's method assumes (approximately) independent gene-level
      associations.  Linkage disequilibrium and shared eQTLs in TWAS can
      induce gene–gene correlations, slightly inflating Type I error.
    * Permutation p-values are computed against a random gene-set null
      and capture competitive (vs. genome) enrichment; they do not test
      self-containment vs. biological pathway annotation.
    * Wilcoxon/paired tests with n < 6 are flagged as unreliable.
      Coverage < 60% should be interpreted with caution.
    * Bootstrap CIs reflect resampling uncertainty in the gene-set
      members, not in the upstream TWAS Z-scores themselves.
    * Concordance metrics treat each gene independently and do not
      account for cross-tissue Z-score correlation.

    HOW TO INTERPRET
    ----------------
    * |Stouffer Z| > 1.96 ↔ two-sided p < 0.05 (uncorrected); compare
      against permutation p for a competitive null.
    * Cohen's d benchmarks (between-disease):
        small ≈ 0.2 | medium ≈ 0.5 | large ≈ 0.8
    * Rank-biserial r benchmarks:
        small ≈ 0.1 | medium ≈ 0.3 | large ≈ 0.5
    * Lin's CCC: 0 = no concordance, 1 = perfect agreement on identity line.
    * Sign concordance rate = 0.5 expected under independence; values
      ≫ 0.5 indicate shared directional effects.
""")


def run_pipeline(
    dirs,
    labels,
    geneset,
    label="CustomGeneSet",
    out="results",
    n_perm=10000,
    save_outputs=True,
    plot=True,
    verbose=True,
    # ── New robustness parameters (all backward-compatible) ─────────
    random_seed=42,
    robust=False,
    bootstrap=False,
    n_boot=2000,
    family_wise_fdr=False,
    influence_threshold_pct=20.0,
):
    """
    Run the full gene-set differential & proximity pipeline.

    New parameters
    --------------
    random_seed : int
        Master RNG seed used for permutation, sign-flip, and bootstrap
        steps.  Logged into every summary for reproducibility.
    robust : bool
        If True, winsorize Z-scores at 1st/99th percentile before
        parametric statistics (Stouffer Z, Welch t, Cohen's d, ...).
        Non-parametric tests are unaffected.
    bootstrap : bool
        If True, compute 95% bootstrap percentile CIs for Stouffer Z,
        mean Z, and mean |Z| (added to the enrichment table and plots).
    n_boot : int
        Number of bootstrap resamples.
    family_wise_fdr : bool
        If True, additionally compute a single global BH-FDR across
        ALL p-values produced by enrichment + differential + pairwise
        + proximity tests within this run.  Result saved as
        family_wise_fdr_<label>.csv.
    influence_threshold_pct : float
        Percent change in |Stouffer Z| at which a single gene removal
        is flagged as "influential" in the LOO analysis.

    Returns dict (existing keys preserved, new keys added):
        ... (existing) ...
        "influence_df"  : per-disease LOO influence table
        "provenance"    : input-file SHA256 manifest dict
        "warnings"      : list of pipeline-level warnings
    """
    _print = print if verbose else lambda *a, **k: None
    step_errors = {}
    pipeline_warnings = []

    # ── Validate dirs / labels ────────────────────────────────────
    if isinstance(dirs, (str, Path)):
        dirs = [dirs]
    if isinstance(labels, str):
        labels = [labels]
    dirs = list(dirs)
    labels = list(labels)
    if len(dirs) != len(labels):
        raise ValueError(
            f"dirs (length {len(dirs)}) and labels (length {len(labels)}) "
            f"must have the same length."
        )

    if save_outputs or plot:
        os.makedirs(out, exist_ok=True)

    # Master RNG (children spawned for sub-steps for reproducibility)
    master_rng = np.random.default_rng(random_seed)

    _print(f"\n{'='*72}")
    _print(f"  PSYCHIATRIC GENE-SET DIFFERENTIAL & PROXIMITY PIPELINE")
    _print(f"  Gene set         : {label}")
    _print(f"  Conditions       : {', '.join(labels)}")
    _print(f"  Output           : {out}/")
    _print(f"  Random seed      : {random_seed}")
    _print(f"  Robust mode      : {robust}")
    _print(f"  Bootstrap CIs    : {bootstrap}  (n_boot={n_boot})")
    _print(f"  n_perm           : {n_perm}")
    _print(f"  Family-wise FDR  : {family_wise_fdr}")
    _print(f"{'='*72}\n")

    # ── Compute provenance early ───────────────────────────────────
    try:
        provenance = compute_input_provenance(dirs, geneset)
        if save_outputs:
            with open(os.path.join(out, f"input_manifest_{label}.json"), "w") as fh:
                json.dump(provenance, fh, indent=2)
    except Exception as e:
        provenance = {"error": str(e)}
        pipeline_warnings.append(f"Provenance manifest failed: {e}")

    # ─── Step 1: Load disease data ─────────────────────────────────
    _print("[1/7] Loading S-PrediXcan meta-Z scores …")
    meta_z = {}
    id_maps = {}
    tissue_z = {}
    for tag, folder in zip(labels, dirs):
        _print(f"  {tag}:")
        mz, im, tz = load_spredixcan_folder(folder)
        meta_z[tag] = mz
        id_maps[tag] = im
        tissue_z[tag] = tz

    # Z-score sanity validation
    val_warnings = validate_zscores(meta_z)
    pipeline_warnings.extend(val_warnings)
    for w in val_warnings:
        _print(f"  ⚠ {w}")

    all_genes_union = set()
    for mz in meta_z.values():
        all_genes_union.update(mz.keys())
    _print(f"\n  Total unique genes across all diseases: {len(all_genes_union):,}")

    # ─── Step 2: Load gene set ─────────────────────────────────────
    _print(f"\n[2/7] Loading gene set …")
    gene_list = load_geneset(geneset)
    gene_list = harmonize_geneset(gene_list, meta_z, id_maps, verbose=verbose)

    found_any = sum(1 for g in gene_list if g in all_genes_union)
    _print(f"  {found_any}/{len(gene_list)} genes found in at least one disease dataset")

    if found_any == 0:
        raise ValueError(
            "No genes from the set were found in the data. "
            "Check gene names (should match gene_name column, case-insensitive)."
        )
    if found_any < 5:
        msg = (f"Only {found_any} gene(s) found across diseases — results "
               f"are EXPLORATORY. Differential / proximity tests may be unreliable.")
        pipeline_warnings.append(msg)
        _print(f"  ⚠ {msg}")

    # ─── Initialise default results ────────────────────────────────
    enrichment = {}
    enrich_df = pd.DataFrame()
    diff = {
        "n_common_genes": 0, "common_genes": [],
        "kruskal_h": np.nan, "kruskal_p": np.nan,
        "pairwise_df": pd.DataFrame()
    }
    prox_df = pd.DataFrame()
    profile_matrix = pd.DataFrame()
    pairwise_stats_df = pd.DataFrame()
    gene_detail = pd.DataFrame()
    influence_df = pd.DataFrame()
    family_wise_fdr_df = pd.DataFrame()

    # ─── Step 3: Enrichment ────────────────────────────────────────
    _print(f"\n[3/7] Computing per-disease enrichment "
           f"(perm={n_perm}, boot={'on' if bootstrap else 'off'}, robust={robust}) …")
    try:
        for d in labels:
            # Spawn distinct seed per disease for full reproducibility
            disease_seed = int(master_rng.integers(0, 2**31 - 1))
            enrichment[d] = gene_set_enrichment(
                meta_z[d], gene_list,
                n_perm=n_perm,
                random_seed=disease_seed,
                robust=robust,
                bootstrap=bootstrap,
                n_boot=n_boot,
            )
            info = enrichment[d]

            ci_str = ""
            if bootstrap and not np.isnan(info.get("stouffer_ci_low", np.nan)):
                ci_str = (f" CI95=[{info['stouffer_ci_low']}, "
                          f"{info['stouffer_ci_high']}]")
            _print(f"  {d}: {info['n_genes_found']}/{info['n_genes_total']} "
                   f"(coverage={info['coverage_pct']}%) | "
                   f"Stouffer Z = {info['stouffer_z']}{ci_str} | "
                   f"|Z| = {info['mean_abs_z']} | Perm p = {info['permutation_p']}")
            if info["low_coverage_warning"]:
                pipeline_warnings.append(
                    f"{d}: gene-set coverage {info['coverage_pct']}% < 60% — interpret cautiously")
            if info["low_n_warning"]:
                pipeline_warnings.append(
                    f"{d}: only {info['n_genes_found']} genes found — exploratory")

        enrich_rows = []
        for d in labels:
            row = {"Disease": d}
            row.update({k: v for k, v in enrichment[d].items() if k != "z_scores"})
            enrich_rows.append(row)
        enrich_df = pd.DataFrame(enrich_rows)

        if save_outputs:
            enrich_df.to_csv(os.path.join(out, f"enrichment_{label}.csv"), index=False)
            _print(f"  → {out}/enrichment_{label}.csv")
    except Exception as e:
        step_errors["enrichment"] = str(e)
        _print(f"  ⚠ Step 3 (Enrichment) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 4: Differential tests ────────────────────────────────
    _print(f"\n[4/7] Differential tests across diseases …")
    try:
        diff = differential_tests(meta_z, gene_list)
        _print(f"  Common genes in all {len(labels)} diseases: {diff['n_common_genes']}")
        _print(f"  Kruskal-Wallis H = {diff['kruskal_h']}, p = {diff['kruskal_p']}")

        if save_outputs and not diff["pairwise_df"].empty:
            pw_path = os.path.join(out, f"differential_pairwise_{label}.csv")
            diff["pairwise_df"].to_csv(pw_path, index=False)
            _print(f"  → {pw_path}")
        if verbose and not diff["pairwise_df"].empty:
            _print(diff["pairwise_df"].to_string(index=False))
    except Exception as e:
        step_errors["differential"] = str(e)
        _print(f"  ⚠ Step 4 (Differential tests) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 5: Proximity / distance ──────────────────────────────
    _print(f"\n[5/7] Profile proximity & distance …")
    try:
        prox_df, profile_matrix = profile_proximity(meta_z, gene_list)

        if save_outputs and not prox_df.empty:
            prox_path = os.path.join(out, f"proximity_{label}.csv")
            prox_df.to_csv(prox_path, index=False)
            _print(f"  → {prox_path}")
        if verbose and not prox_df.empty:
            _print(prox_df.to_string(index=False))
    except Exception as e:
        step_errors["proximity"] = str(e)
        _print(f"  ⚠ Step 5 (Proximity) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 6: Pairwise disease statistical tests ────────────────
    _print(f"\n[6/7] Pairwise disease statistical tests "
           f"(gene-set-restricted, robust={robust}) …")
    try:
        pw_seed = int(master_rng.integers(0, 2**31 - 1))
        pairwise_stats_df = pairwise_disease_geneset_tests(
            meta_z, gene_list, n_perm=n_perm,
            random_seed=pw_seed, robust=robust,
        )
        if not pairwise_stats_df.empty:
            _print(f"  {len(pairwise_stats_df)} disease pair(s) tested")
            for _, row in pairwise_stats_df.iterrows():
                conc_rate_str = (f"{row['sign_concordance_rate']:.3f}"
                                 if not pd.isna(row.get('sign_concordance_rate', np.nan))
                                 else "NA")
                conc_p_str = (f"{row['sign_concordance_p']:.4f}"
                              if not pd.isna(row.get('sign_concordance_p', np.nan))
                              else "NA")
                ccc_str = (f"{row['Lin_CCC']:.3f}"
                           if not pd.isna(row.get('Lin_CCC', np.nan)) else "NA")
                tau_str = (f"{row['Kendall_tau']:.3f}"
                           if not pd.isna(row.get('Kendall_tau', np.nan)) else "NA")
                _print(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"n_paired={row['n_genes_paired']}, "
                    f"mean_diff={row.get('mean_diff', 'NA')}, "
                    f"KS p={row.get('KS_p', 'NA')}, "
                    f"Welch p={row.get('Welch_p', 'NA')}, "
                    f"Paired-t p={row.get('Paired_t_p', 'NA')}, "
                    f"Cohen d(paired)={row.get('Cohens_d_paired', 'NA')}, "
                    f"SignConc={conc_rate_str} (p={conc_p_str}), "
                    f"CCC={ccc_str}, τ={tau_str}"
                )
            if save_outputs:
                pw_stats_path = os.path.join(out, f"pairwise_disease_stats_{label}.csv")
                pairwise_stats_df.to_csv(pw_stats_path, index=False)
                _print(f"  → {pw_stats_path}")
        else:
            _print("  (fewer than 2 diseases – skipped)")
    except Exception as e:
        step_errors["pairwise_stats"] = str(e)
        _print(f"  ⚠ Step 6 (Pairwise disease stats) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Step 7: Gene-level LOO influence analysis ─────────────────
    _print(f"\n[7/7] Gene-level leave-one-out influence analysis …")
    try:
        influence_df = gene_influence_analysis(
            meta_z, gene_list, threshold_pct=influence_threshold_pct
        )
        if not influence_df.empty:
            n_inf = int(influence_df["Influential"].sum())
            _print(f"  {n_inf} gene(s) flagged as influential "
                   f"(>|{influence_threshold_pct}%| change or sign-flip)")
            if save_outputs:
                inf_path = os.path.join(out, f"gene_influence_{label}.csv")
                influence_df.to_csv(inf_path, index=False)
                _print(f"  → {inf_path}")
        else:
            _print("  (insufficient data for LOO analysis)")
    except Exception as e:
        step_errors["influence"] = str(e)
        _print(f"  ⚠ Step 7 (Influence) FAILED: {e}")
        _print(f"    Traceback: {traceback.format_exc().strip()}")

    # ─── Per-gene detail ───────────────────────────────────────────
    try:
        gene_detail = build_gene_detail_table(meta_z, gene_list)
        if save_outputs:
            detail_path = os.path.join(out, f"gene_detail_{label}.csv")
            gene_detail.to_csv(detail_path, index=False)
            _print(f"  → {detail_path}")
    except Exception as e:
        step_errors["gene_detail"] = str(e)
        _print(f"  ⚠ Per-gene detail table FAILED: {e}")

    # ─── Optional: family-wise FDR across all p-values ─────────────
    if family_wise_fdr:
        try:
            fw_rows = []
            # Enrichment p's
            if not enrich_df.empty:
                for _, r in enrich_df.iterrows():
                    for col in ("wilcoxon_p", "permutation_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "enrichment", "Test": col,
                                "Comparison": r["Disease"], "p_value": float(v)})
            # Differential pairwise
            if isinstance(diff.get("pairwise_df"), pd.DataFrame) and not diff["pairwise_df"].empty:
                for _, r in diff["pairwise_df"].iterrows():
                    for col in ("MWU_p", "Paired_Wilcoxon_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "differential", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
                # Kruskal-Wallis
                if pd.notna(diff.get("kruskal_p", np.nan)):
                    fw_rows.append({
                        "Source": "differential", "Test": "kruskal_p",
                        "Comparison": "all_diseases",
                        "p_value": float(diff["kruskal_p"])})
            # Proximity p's
            if isinstance(prox_df, pd.DataFrame) and not prox_df.empty:
                for _, r in prox_df.iterrows():
                    for col in ("Pearson_p", "Spearman_p"):
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "proximity", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
            # Pairwise stats p's
            if isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
                p_cols = [c for c in pairwise_stats_df.columns if c.endswith("_p")]
                for _, r in pairwise_stats_df.iterrows():
                    for col in p_cols:
                        v = r.get(col, np.nan)
                        if pd.notna(v):
                            fw_rows.append({
                                "Source": "pairwise_stats", "Test": col,
                                "Comparison": f"{r['Disease_1']}__vs__{r['Disease_2']}",
                                "p_value": float(v)})
            family_wise_fdr_df = pd.DataFrame(fw_rows)
            if not family_wise_fdr_df.empty:
                family_wise_fdr_df["family_wise_FDR"] = benjamini_hochberg(
                    family_wise_fdr_df["p_value"].values
                )
                family_wise_fdr_df = family_wise_fdr_df.sort_values(
                    "p_value").reset_index(drop=True)
                if save_outputs:
                    fw_path = os.path.join(out, f"family_wise_fdr_{label}.csv")
                    family_wise_fdr_df.to_csv(fw_path, index=False)
                    _print(f"  → {fw_path}  ({len(family_wise_fdr_df)} p-values)")
        except Exception as e:
            step_errors["family_wise_fdr"] = str(e)
            _print(f"  ⚠ Family-wise FDR FAILED: {e}")

    # ─── Plots ─────────────────────────────────────────────────────
    if plot:
        _print(f"\nGenerating plots …")
        try:
            if enrichment:
                plot_zscore_boxplot(enrichment, label, out)
        except Exception as e:
            _print(f"  ⚠ Boxplot failed: {e}")

        try:
            if enrichment:
                plot_enrichment_bar(enrichment, label, out)
        except Exception as e:
            _print(f"  ⚠ Enrichment bar plot failed: {e}")

        try:
            if not prox_df.empty:
                plot_proximity_heatmap(prox_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Proximity heatmap failed: {e}")

        try:
            if not prox_df.empty:
                plot_proximity_dendrogram(prox_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Proximity dendrogram failed: {e}")

        try:
            if isinstance(profile_matrix, pd.DataFrame) and not profile_matrix.empty:
                plot_scatter_matrix(profile_matrix, label, out)
        except Exception as e:
            _print(f"  ⚠ Scatter matrix plot failed: {e}")

        try:
            if isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
                plot_pairwise_stats_heatmap(pairwise_stats_df, labels, label, out)
        except Exception as e:
            _print(f"  ⚠ Pairwise stats heatmap failed: {e}")

        try:
            if gene_list:
                plot_zscore_qq(meta_z, gene_list, label, out)
        except Exception as e:
            _print(f"  ⚠ QQ-plot failed: {e}")

        try:
            if isinstance(influence_df, pd.DataFrame) and not influence_df.empty:
                plot_gene_influence(influence_df, label, out)
        except Exception as e:
            _print(f"  ⚠ Gene-influence plot failed: {e}")

    # ─── Summary ───────────────────────────────────────────────────
    summary_text = ""
    try:
        lines = [
            f"{'='*72}",
            f"  GENE-SET ANALYSIS SUMMARY – {label}",
            f"  {len(gene_list)} genes in set, {found_any} found in data",
            f"  Conditions: {', '.join(labels)}",
            f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"{'='*72}",
            "",
            "  RUN PARAMETERS:",
            f"    Random seed         : {random_seed}",
            f"    Robust (winsorized) : {robust}",
            f"    Bootstrap CIs       : {bootstrap}  (n_boot={n_boot})",
            f"    Permutations (n)    : {n_perm}",
            f"    Family-wise FDR     : {family_wise_fdr}",
            f"    Influence threshold : ±{influence_threshold_pct}%",
            "",
            "  INPUT PROVENANCE (SHA256, first 16 hex):"
        ]
        try:
            for dir_key, dir_info in provenance.get("input_dirs", {}).items():
                lines.append(f"    [{dir_key}]")
                for fname, finfo in dir_info.get("files", {}).items():
                    lines.append(f"      {fname}: {finfo.get('sha256', '')[:16]}")
            gs_info = provenance.get("geneset_input")
            if isinstance(gs_info, dict) and "sha256" in gs_info:
                lines.append(f"    [geneset]: {gs_info['sha256'][:16]} "
                             f"({Path(gs_info.get('path','')).name})")
            elif isinstance(gs_info, dict):
                lines.append(f"    [geneset]: type={gs_info.get('type','?')}")
        except Exception:
            lines.append("    (provenance unavailable)")

        lines += ["", "  ENRICHMENT (per disease):"]
        if enrichment:
            for d in labels:
                e = enrichment.get(d, {})
                ci_blurb = ""
                if bootstrap and not pd.isna(e.get("stouffer_ci_low", np.nan)):
                    ci_blurb = (f" [CI95: {e['stouffer_ci_low']}, "
                                f"{e['stouffer_ci_high']}]")
                lines.append(
                    f"    {d}: n={e.get('n_genes_found','?')}/{e.get('n_genes_total','?')} "
                    f"(cov={e.get('coverage_pct','?')}%) | "
                    f"Stouffer Z={e.get('stouffer_z','?')}{ci_blurb} | "
                    f"|Z|={e.get('mean_abs_z','?')} | "
                    f"trimZ={e.get('trimmed_mean_z','?')} | "
                    f"Wilcoxon p={e.get('wilcoxon_p','?')} | "
                    f"Perm p={e.get('permutation_p','?')}"
                )
                flags = []
                if e.get("low_n_warning"): flags.append("LOW_N")
                if e.get("low_coverage_warning"): flags.append("LOW_COVERAGE")
                if e.get("wilcoxon_unreliable"): flags.append("WILCOXON<6")
                if flags:
                    lines.append(f"        ⚠ Flags: {', '.join(flags)}")
        else:
            lines.append("    (enrichment step failed)")

        if "enrichment" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['enrichment']}")

        lines += [
            "",
            "  DIFFERENTIAL TESTS:",
            f"    Common genes (all {len(labels)}): {diff['n_common_genes']}",
            f"    Kruskal-Wallis H={diff['kruskal_h']}, p={diff['kruskal_p']}"
        ]
        if "differential" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['differential']}")
        elif not diff["pairwise_df"].empty:
            for _, row in diff["pairwise_df"].iterrows():
                fdr_val = row.get("MWU_FDR", np.nan)
                fdr_str = f"{fdr_val:.4f}" if not pd.isna(fdr_val) else "NA"
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"MWU p={row['MWU_p']}, FDR={fdr_str}, "
                    f"mean-Z diff={row['Mean_Z_diff']}"
                )

        lines += ["", "  PROFILE PROXIMITY:"]
        if "proximity" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['proximity']}")
        elif not prox_df.empty:
            for _, row in prox_df.iterrows():
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']}: "
                    f"r={row['Pearson_r']}, rho={row['Spearman_rho']}, "
                    f"cos={row['Cosine_sim']}, Eucl={row['Euclidean_dist']}"
                )
        else:
            lines.append("    (insufficient gene overlap)")

        lines += ["", "  PAIRWISE DISEASE STAT TESTS (gene-set-restricted):"]
        if "pairwise_stats" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['pairwise_stats']}")
        elif isinstance(pairwise_stats_df, pd.DataFrame) and not pairwise_stats_df.empty:
            for _, row in pairwise_stats_df.iterrows():
                cd_str = (f"{row['Cohens_d_paired']:.3f}"
                          if not pd.isna(row.get('Cohens_d_paired', np.nan)) else "NA")
                pt_str = (f"{row['Paired_t_p']:.4f}"
                          if not pd.isna(row.get('Paired_t_p', np.nan)) else "NA")
                ks_str = (f"{row['KS_p']:.4f}"
                          if not pd.isna(row.get('KS_p', np.nan)) else "NA")
                we_str = (f"{row['Welch_p']:.4f}"
                          if not pd.isna(row.get('Welch_p', np.nan)) else "NA")
                pm_str = (f"{row['Permutation_p']:.4f}"
                          if not pd.isna(row.get('Permutation_p', np.nan)) else "NA")
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']} "
                    f"(n_paired={row['n_genes_paired']}): "
                    f"mean_diff={row.get('mean_diff','NA')}, "
                    f"Cohen_d={cd_str}, "
                    f"KS_p={ks_str}, Welch_p={we_str}, "
                    f"Paired_t_p={pt_str}, Perm_p={pm_str}"
                )
            lines.append("")
            lines.append("  CONCORDANCE (gene-set-restricted, paired genes):")
            for _, row in pairwise_stats_df.iterrows():
                conc_rate = row.get("sign_concordance_rate", np.nan)
                conc_p    = row.get("sign_concordance_p", np.nan)
                n_conc    = row.get("n_concordant", np.nan)
                n_disc    = row.get("n_discordant", np.nan)
                ccc_val   = row.get("Lin_CCC", np.nan)
                tau_val   = row.get("Kendall_tau", np.nan)
                tau_p     = row.get("Kendall_p", np.nan)

                conc_rate_str = f"{conc_rate:.3f}" if not pd.isna(conc_rate) else "NA"
                conc_p_str    = f"{conc_p:.4f}"    if not pd.isna(conc_p)    else "NA"
                n_conc_str    = f"{int(n_conc)}"   if not pd.isna(n_conc)    else "NA"
                n_disc_str    = f"{int(n_disc)}"   if not pd.isna(n_disc)    else "NA"
                ccc_str       = f"{ccc_val:.3f}"   if not pd.isna(ccc_val)   else "NA"
                tau_str       = f"{tau_val:.3f}"   if not pd.isna(tau_val)   else "NA"
                tau_p_str     = f"{tau_p:.4f}"     if not pd.isna(tau_p)     else "NA"
                lines.append(
                    f"    {row['Disease_1']} vs {row['Disease_2']} "
                    f"(n_paired={row['n_genes_paired']}): "
                    f"concordant={n_conc_str}, discordant={n_disc_str}, "
                    f"sign_conc_rate={conc_rate_str} (p={conc_p_str}), "
                    f"Lin_CCC={ccc_str}, "
                    f"Kendall_τ={tau_str} (p={tau_p_str})"
                )
        else:
            lines.append("    (fewer than 2 diseases or insufficient genes)")

        # ── Influence summary ──────────────────────────────────────
        lines += ["", "  GENE INFLUENCE (LOO):"]
        if "influence" in step_errors:
            lines.append(f"    ⚠ STEP FAILED: {step_errors['influence']}")
        elif isinstance(influence_df, pd.DataFrame) and not influence_df.empty:
            n_total_inf = int(influence_df["Influential"].sum())
            lines.append(f"    Genes flagged as influential "
                         f"(|ΔStouffer|>{influence_threshold_pct}% or sign-flip): "
                         f"{n_total_inf}")
            for d in labels:
                sub = influence_df[influence_df["Disease"] == d]
                if sub.empty:
                    continue
                flagged = sub[sub["Influential"]]
                if not flagged.empty:
                    top = flagged.head(5)[["Gene", "Delta_Stouffer",
                                           "Pct_Change_Stouffer", "Sign_Change"]]
                    desc = ", ".join(
                        f"{r.Gene} (Δ={r.Delta_Stouffer:+.2f}, "
                        f"{r.Pct_Change_Stouffer:.1f}%"
                        f"{', SIGN' if r.Sign_Change else ''})"
                        for r in top.itertuples(index=False))
                    lines.append(f"    {d}: {desc}")
                else:
                    lines.append(f"    {d}: no genes exceed influence threshold")
        else:
            lines.append("    (insufficient data)")

        # ── Family-wise FDR section ────────────────────────────────
        if family_wise_fdr:
            lines += ["", "  FAMILY-WISE FDR (across all in-run p-values):"]
            if "family_wise_fdr" in step_errors:
                lines.append(f"    ⚠ STEP FAILED: {step_errors['family_wise_fdr']}")
            elif not family_wise_fdr_df.empty:
                n_total = len(family_wise_fdr_df)
                n_sig = int((family_wise_fdr_df["family_wise_FDR"] < 0.05).sum())
                lines.append(f"    Total p-values pooled: {n_total}")
                lines.append(f"    Significant after global FDR (<0.05): {n_sig}")
                top5 = family_wise_fdr_df.head(5)
                for _, r in top5.iterrows():
                    lines.append(
                        f"    {r['Source']}/{r['Test']} [{r['Comparison']}]: "
                        f"p={r['p_value']:.2e}, FDR={r['family_wise_FDR']:.4f}"
                    )
            else:
                lines.append("    (no p-values aggregated)")

        # ── Pipeline warnings ──────────────────────────────────────
        if pipeline_warnings:
            lines += ["", "  PIPELINE WARNINGS:"]
            for w in pipeline_warnings:
                lines.append(f"    ⚠ {w}")

        if step_errors:
            lines += ["", "  STEP ERRORS:"]
            for step_name, err_msg in step_errors.items():
                lines.append(f"    {step_name}: {err_msg}")

        # ── Limitations & interpretation block ─────────────────────
        lines += ["", _LIMITATIONS_BLOCK.rstrip(), "", f"{'='*72}"]

        summary_text = "\n".join(lines)
        _print(f"\n{summary_text}")

        if save_outputs:
            summary_path = os.path.join(out, f"summary_{label}.txt")
            with open(summary_path, "w") as fh:
                fh.write(summary_text)
    except Exception as e:
        summary_text = f"Summary generation failed: {e}"
        _print(f"  ⚠ Summary generation FAILED: {e}")

    _print(f"\n{'='*72}")
    if step_errors:
        _print(f"  DONE (with {len(step_errors)} step error(s)) – outputs in {out}/")
        for step_name, err_msg in step_errors.items():
            _print(f"    ⚠ {step_name}: {err_msg}")
    else:
        _print(f"  DONE – all outputs in {out}/")
    _print(f"{'='*72}\n")

    return {
        "meta_z": meta_z,
        "tissue_z": tissue_z,
        "gene_list": gene_list,
        "enrichment": enrichment,
        "enrichment_df": enrich_df,
        "differential": diff,
        "proximity_df": prox_df,
        "profile_matrix": profile_matrix,
        "pairwise_stats_df": pairwise_stats_df,
        "gene_detail_df": gene_detail,
        "influence_df": influence_df,
        "family_wise_fdr_df": family_wise_fdr_df,
        "summary_text": summary_text,
        "step_errors": step_errors,
        "warnings": pipeline_warnings,
        "provenance": provenance,
        "run_params": {
            "random_seed": random_seed,
            "robust": robust,
            "bootstrap": bootstrap,
            "n_boot": n_boot,
            "n_perm": n_perm,
            "family_wise_fdr": family_wise_fdr,
            "influence_threshold_pct": influence_threshold_pct,
        },
    }


def run_multi_geneset_pipeline(
    dirs,
    labels,
    gene_sets,
    out="results",
    n_perm=10000,
    tissue_n_perm=0,
    save_outputs=True,
    plot=True,
    verbose=True,
    # ── New robustness parameters (forwarded to run_pipeline) ─────
    random_seed=42,
    robust=False,
    bootstrap=False,
    n_boot=2000,
    family_wise_fdr=False,
    influence_threshold_pct=20.0,
):
    """
    Run the same pipeline over multiple gene sets.

    Same new parameters as run_pipeline; they are forwarded per-set.
    """
    _print = print if verbose else lambda *a, **k: None

    gene_sets = load_multiple_genesets(gene_sets)

    if save_outputs or plot:
        os.makedirs(out, exist_ok=True)

    all_results = {}
    summary_rows = []
    tissue_summary_rows = []
    gene_detail_frames = []
    pairwise_stats_frames = []
    influence_frames = []

    for gs_label, genes in gene_sets.items():
        safe_label = str(gs_label).replace(" ", "_")
        sub_out = os.path.join(out, safe_label)

        _print(f"\n{'#'*72}")
        _print(f"Running gene set: {gs_label}  (n={len(genes)})")
        _print(f"{'#'*72}")

        try:
            res = run_pipeline(
                dirs=dirs,
                labels=labels,
                geneset=genes,
                label=gs_label,
                out=sub_out,
                n_perm=n_perm,
                save_outputs=save_outputs,
                plot=plot,
                verbose=verbose,
                random_seed=random_seed,
                robust=robust,
                bootstrap=bootstrap,
                n_boot=n_boot,
                family_wise_fdr=family_wise_fdr,
                influence_threshold_pct=influence_threshold_pct,
            )
            all_results[gs_label] = res

            if isinstance(res.get("enrichment_df"), pd.DataFrame) and not res["enrichment_df"].empty:
                for _, row in res["enrichment_df"].iterrows():
                    summary_rows.append({
                        "GeneSet": gs_label,
                        "Disease": row["Disease"],
                        "Tissue": "meta_across_tissues",
                        "n_genes_found": row["n_genes_found"],
                        "n_genes_total": row["n_genes_total"],
                        "coverage_pct": row["coverage_pct"],
                        "stouffer_z": row["stouffer_z"],
                        "stouffer_ci_low":  row.get("stouffer_ci_low", np.nan),
                        "stouffer_ci_high": row.get("stouffer_ci_high", np.nan),
                        "mean_abs_z": row["mean_abs_z"],
                        "mean_abs_z_ci_low":  row.get("mean_abs_z_ci_low", np.nan),
                        "mean_abs_z_ci_high": row.get("mean_abs_z_ci_high", np.nan),
                        "mean_z": row["mean_z"],
                        "mean_z_ci_low":  row.get("mean_z_ci_low", np.nan),
                        "mean_z_ci_high": row.get("mean_z_ci_high", np.nan),
                        "trimmed_mean_z": row.get("trimmed_mean_z", np.nan),
                        "median_z": row["median_z"],
                        "wilcoxon_p": row["wilcoxon_p"],
                        "permutation_p": row["permutation_p"],
                        "low_n_warning":        row.get("low_n_warning", False),
                        "low_coverage_warning": row.get("low_coverage_warning", False),
                    })

            res_tissue_z = res.get("tissue_z", {})
            harmonized_genes = res.get("gene_list", [])
            for disease_label in labels:
                disease_tissues = res_tissue_z.get(disease_label, {})
                for tissue_name in sorted(disease_tissues.keys()):
                    tissue_dict = disease_tissues[tissue_name]
                    try:
                        te = gene_set_enrichment(
                            tissue_dict, harmonized_genes,
                            n_perm=tissue_n_perm,
                            random_seed=random_seed,
                            robust=robust,
                            bootstrap=False,  # keep speed for tissues
                        )
                        tissue_summary_rows.append({
                            "GeneSet": gs_label,
                            "Disease": disease_label,
                            "Tissue": tissue_name,
                            "n_genes_found": te["n_genes_found"],
                            "n_genes_total": te["n_genes_total"],
                            "coverage_pct": te["coverage_pct"],
                            "stouffer_z": te["stouffer_z"],
                            "mean_abs_z": te["mean_abs_z"],
                            "mean_z": te["mean_z"],
                            "trimmed_mean_z": te.get("trimmed_mean_z", np.nan),
                            "median_z": te["median_z"],
                            "wilcoxon_p": te["wilcoxon_p"],
                            "permutation_p": te["permutation_p"],
                        })
                    except Exception as te_err:
                        _print(f"  ⚠ Tissue enrichment failed for "
                               f"{disease_label}/{tissue_name}: {te_err}")

            pw_res = res.get("pairwise_stats_df", pd.DataFrame())
            if isinstance(pw_res, pd.DataFrame) and not pw_res.empty:
                pw_copy = pw_res.copy()
                pw_copy.insert(0, "GeneSet", gs_label)
                pairwise_stats_frames.append(pw_copy)

            inf_res = res.get("influence_df", pd.DataFrame())
            if isinstance(inf_res, pd.DataFrame) and not inf_res.empty:
                inf_copy = inf_res.copy()
                inf_copy.insert(0, "GeneSet", gs_label)
                influence_frames.append(inf_copy)

            if ("gene_detail_df" in res
                    and isinstance(res["gene_detail_df"], pd.DataFrame)
                    and not res["gene_detail_df"].empty):
                gdf = res["gene_detail_df"].copy()
                gdf.insert(0, "GeneSet", gs_label)

                for disease_label in labels:
                    disease_tissues = res_tissue_z.get(disease_label, {})
                    for tissue_name in sorted(disease_tissues.keys()):
                        col_name = f"Z_{disease_label}__{tissue_name}"
                        td = disease_tissues[tissue_name]
                        gdf[col_name] = gdf["Gene"].map(
                            lambda g, _td=td: round(_td[g], 4) if g in _td else np.nan
                        )

                gene_detail_frames.append(gdf)

        except Exception as e:
            _print(f"  ⚠ Failed for gene set '{gs_label}': {e}")
            all_results[gs_label] = {"error": str(e)}

    # ── Save enrichment summary (meta + tissue rows) ─────────────────
    if save_outputs and (summary_rows or tissue_summary_rows):
        all_enrich = summary_rows + tissue_summary_rows
        enrich_summary_df = pd.DataFrame(all_enrich)
        front = ["GeneSet", "Disease", "Tissue"]
        rest = [c for c in enrich_summary_df.columns if c not in front]
        enrich_summary_df = enrich_summary_df[front + rest]
        enrich_summary_df.sort_values(
            ["GeneSet", "Disease", "Tissue"], ignore_index=True, inplace=True
        )
        enrich_path = os.path.join(out, "multi_geneset_enrichment_summary.csv")
        enrich_summary_df.to_csv(enrich_path, index=False)
        _print(f"\n  → Enrichment summary (with by-tissue + bootstrap CI cols): {enrich_path}")
        n_meta = len(summary_rows)
        n_tiss = len(tissue_summary_rows)
        _print(f"    ({n_meta} meta-level rows + {n_tiss} tissue-level rows)")

    # ── Save combined pairwise disease stats (with global FDR) ───────
    combined_pairwise_stats_df = pd.DataFrame()
    if pairwise_stats_frames:
        combined_pairwise_stats_df = pd.concat(
            pairwise_stats_frames, ignore_index=True
        )
        p_cols = [c for c in combined_pairwise_stats_df.columns if c.endswith("_p")]
        for pc in p_cols:
            global_fdr_col = pc.replace("_p", "_globalFDR")
            vals = combined_pairwise_stats_df[pc].values.astype(float)
            mask = ~np.isnan(vals)
            if mask.sum() > 1:
                fdr_vals = np.full(len(vals), np.nan)
                fdr_vals[mask] = benjamini_hochberg(vals[mask])
                combined_pairwise_stats_df[global_fdr_col] = fdr_vals
            elif mask.sum() == 1:
                combined_pairwise_stats_df[global_fdr_col] = vals

        if save_outputs:
            pw_path = os.path.join(out, "multi_geneset_pairwise_disease_stats.csv")
            combined_pairwise_stats_df.to_csv(pw_path, index=False)
            _print(f"\n  → Pairwise disease stats (all gene sets): {pw_path}")
            _print(f"    ({len(combined_pairwise_stats_df)} rows across "
                   f"{combined_pairwise_stats_df['GeneSet'].nunique()} gene set(s))")

    all_results["_combined_pairwise_stats_df"] = combined_pairwise_stats_df

    # ── Combined influence table ─────────────────────────────────────
    combined_influence_df = pd.DataFrame()
    if influence_frames:
        combined_influence_df = pd.concat(influence_frames, ignore_index=True)
        if save_outputs:
            inf_path = os.path.join(out, "multi_geneset_gene_influence.csv")
            combined_influence_df.to_csv(inf_path, index=False)
            _print(f"\n  → Combined gene-influence (LOO) table: {inf_path}")
    all_results["_combined_influence_df"] = combined_influence_df

    # ── Combined gene-detail CSV ─────────────────────────────────────
    combined_gene_df = pd.DataFrame()
    if gene_detail_frames:
        combined_gene_df = pd.concat(gene_detail_frames, ignore_index=True)
        front_cols = ["GeneSet", "Gene"]
        meta_z_cols = [c for c in combined_gene_df.columns
                       if c.startswith("Z_") and "__" not in c
                       and c not in front_cols]
        tissue_z_cols = [c for c in combined_gene_df.columns
                         if c.startswith("Z_") and "__" in c]
        other_cols = [c for c in combined_gene_df.columns
                      if c not in front_cols + meta_z_cols + tissue_z_cols]
        combined_gene_df = combined_gene_df[
            front_cols + meta_z_cols + other_cols + tissue_z_cols
        ]
        if save_outputs:
            combined_path = os.path.join(out, "all_genes_all_genesets.csv")
            combined_gene_df.to_csv(combined_path, index=False)
            _print(f"\n  → Combined gene-level CSV (with by-tissue Z): {combined_path}")
            _print(f"    ({len(combined_gene_df)} rows across "
                   f"{combined_gene_df['GeneSet'].nunique()} gene set(s), "
                   f"{len(tissue_z_cols)} tissue Z columns)")

    all_results["_combined_gene_df"] = combined_gene_df

    # ── Top-level run-parameter manifest ─────────────────────────────
    if save_outputs:
        try:
            run_manifest = {
                "generated_at": datetime.now().isoformat(timespec="seconds"),
                "n_gene_sets": len(gene_sets),
                "labels": labels,
                "dirs": [str(d) for d in dirs],
                "params": {
                    "random_seed": random_seed,
                    "robust": robust,
                    "bootstrap": bootstrap,
                    "n_boot": n_boot,
                    "n_perm": n_perm,
                    "tissue_n_perm": tissue_n_perm,
                    "family_wise_fdr": family_wise_fdr,
                    "influence_threshold_pct": influence_threshold_pct,
                },
            }
            with open(os.path.join(out, "multi_geneset_run_manifest.json"), "w") as fh:
                json.dump(run_manifest, fh, indent=2)
        except Exception as e:
            _print(f"  ⚠ Failed to write run manifest: {e}")

    _print(f"\n{'='*72}")
    _print(f"  MULTI-GENESET PIPELINE COMPLETE – all outputs in {out}/")
    _print(f"{'='*72}\n")

    return all_results


# ===================================================================
#  9.  CLI ENTRY POINT
# ===================================================================

def main():
    """Thin CLI wrapper around run_pipeline()."""
    parser = argparse.ArgumentParser(
        description="Psychiatric Gene-Set Differential & Proximity Pipeline",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=textwrap.dedent("""\
        Examples:
          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --geneset my_genes.txt --label "Dopamine_Receptors"

          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --geneset DRD1,DRD2,DRD3,DRD4,DRD5,COMT --label "DA_Genes" \
              --bootstrap --robust --random_seed 1234

          python geneset_pipeline.py --dirs data/mdd/ data/bip/ data/ocd/ \
              --labels MDD BIP OCD \
              --genesets_file genesets.py --out results/ \
              --family_wise_fdr
        """)
    )
    parser.add_argument("--dirs", required=True, nargs="+",
                        help="Folders with S-PrediXcan results (one per condition)")
    parser.add_argument("--labels", required=True, nargs="+",
                        help="Labels for each folder (same order as --dirs)")
    parser.add_argument("--geneset", default=None,
                        help="Path to txt/tsv file OR comma-separated gene list")
    parser.add_argument("--genesets_file", default=None,
                        help="Path to file containing multiple gene sets")
    parser.add_argument("--label", default="CustomGeneSet",
                        help="Label for gene set (used in titles and filenames)")
    parser.add_argument("--n_perm", type=int, default=10000,
                        help="Permutations for enrichment p-value (default: 10000; "
                             "set to 0 to skip permutations entirely)")
    parser.add_argument("--tissue_n_perm", type=int, default=0,
                        help="Permutations for per-tissue enrichment (default: 0)")
    parser.add_argument("--out", default="results", help="Output directory")
    parser.add_argument("--no_plot", action="store_true", help="Skip plot generation")
    # ── New flags ──────────────────────────────────────────────────
    parser.add_argument("--random_seed", type=int, default=42,
                        help="Master RNG seed for reproducibility (default: 42)")
    parser.add_argument("--robust", action="store_true",
                        help="Winsorize Z-scores (1st/99th pct) before parametric stats")
    parser.add_argument("--bootstrap", action="store_true",
                        help="Compute 95% bootstrap CIs for Stouffer Z, mean Z, mean |Z|")
    parser.add_argument("--n_boot", type=int, default=2000,
                        help="Number of bootstrap resamples (default: 2000)")
    parser.add_argument("--family_wise_fdr", action="store_true",
                        help="Compute global FDR across ALL p-values in the run")
    parser.add_argument("--influence_threshold_pct", type=float, default=20.0,
                        help="LOO percent-change threshold for flagging influential genes")
    args = parser.parse_args()

    if len(args.dirs) != len(args.labels):
        parser.error(
            f"--dirs ({len(args.dirs)} items) and --labels ({len(args.labels)} items) "
            f"must have the same number of arguments."
        )

    if not args.geneset and not args.genesets_file:
        parser.error("Provide either --geneset or --genesets_file")
    if args.geneset and args.genesets_file:
        parser.error("Use only one of --geneset or --genesets_file")

    common_kwargs = dict(
        random_seed=args.random_seed,
        robust=args.robust,
        bootstrap=args.bootstrap,
        n_boot=args.n_boot,
        family_wise_fdr=args.family_wise_fdr,
        influence_threshold_pct=args.influence_threshold_pct,
    )

    try:
        if args.genesets_file:
            run_multi_geneset_pipeline(
                dirs=args.dirs,
                labels=args.labels,
                gene_sets=args.genesets_file,
                out=args.out,
                n_perm=args.n_perm,
                tissue_n_perm=args.tissue_n_perm,
                save_outputs=True,
                plot=not args.no_plot,
                verbose=True,
                **common_kwargs,
            )
        else:
            run_pipeline(
                dirs=args.dirs,
                labels=args.labels,
                geneset=args.geneset,
                label=args.label,
                out=args.out,
                n_perm=args.n_perm,
                save_outputs=True,
                plot=not args.no_plot,
                verbose=True,
                **common_kwargs,
            )
    except Exception as e:
        # Clean error report + traceback to file
        os.makedirs(args.out, exist_ok=True)
        err_path = os.path.join(args.out, "PIPELINE_ERROR.txt")
        with open(err_path, "w") as fh:
            fh.write(f"Pipeline failed at {datetime.now().isoformat(timespec='seconds')}\n")
            fh.write(f"Error: {e}\n\nTraceback:\n{traceback.format_exc()}")
        print(f"\n✗ Pipeline FAILED: {e}")
        print(f"  Full traceback written to: {err_path}")
        sys.exit(1)


# ===================================================================
#  10.  REPORT GENERATOR
# ===================================================================

def save_geneset_report(results, gene_set_name=None, output_dir=None):
    """
    Generate and save a full report from gene-set enrichment results.

    Multi-result support: if `results` is a dict of {label: single_run_result},
    reports are generated into subfolders.

    Outputs (additions in this update marked NEW):
    <output_dir>/
    ├── summary.txt
    ├── tables/
    │   ├── enrichment_summary.csv         (now includes bootstrap CI cols)
    │   ├── pairwise_differential.csv
    │   ├── pairwise_disease_stats.csv
    │   ├── profile_proximity.csv
    │   ├── profile_matrix.csv
    │   ├── gene_detail.csv
    │   ├── meta_z_geneset.csv
    │   ├── gene_influence.csv             (NEW)
    │   ├── family_wise_fdr.csv            (NEW; if computed)
    │   └── input_manifest.json            (NEW)
    └── plots/
        ├── 01_enrichment_overview.png/.pdf
        ├── 02_profile_heatmap.png/.pdf
        ├── 03_gene_barplot.png/.pdf
        ├── 04_pairwise_scatter.png/.pdf
        ├── 05_gene_z_range.png/.pdf
        ├── 06_z_distribution_boxplot.png/.pdf
        ├── 07_enrichment_metrics.png/.pdf
        ├── 08_pairwise_stats_heatmap.png/.pdf
        ├── 09_concordance_heatmap.png/.pdf
        ├── 10_qq_plot.png/.pdf            (NEW)
        ├── 11_dendrogram.png/.pdf         (NEW)
        └── 12_gene_influence.png/.pdf     (NEW)
    """
    # Multi-result support
    if "enrichment" not in results and all(isinstance(v, dict) for v in results.values()):
        out_dirs = {}
        base_dir = output_dir
        if base_dir is None:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            base_dir = f"./geneset_reports_{ts}"
        os.makedirs(base_dir, exist_ok=True)

        for label, res in results.items():
            if not isinstance(res, dict) or "enrichment" not in res:
                continue
            subdir = os.path.join(base_dir, str(label).replace(" ", "_"))
            out_dirs[label] = save_geneset_report(
                res, gene_set_name=label, output_dir=subdir
            )
        return out_dirs

    if gene_set_name is None:
        try:
            line = results["summary_text"].split("\n")[1]
            gene_set_name = line.split("–")[-1].strip() if "–" in line else "gene_set"
        except Exception:
            gene_set_name = "gene_set"

    safe_name = gene_set_name.replace(" ", "_")

    if output_dir is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_dir = f"./geneset_report_{safe_name}_{ts}"

    tables_dir = os.path.join(output_dir, "tables")
    plots_dir  = os.path.join(output_dir, "plots")
    os.makedirs(tables_dir, exist_ok=True)
    os.makedirs(plots_dir,  exist_ok=True)

    diseases = list(results["enrichment"].keys()) if results.get("enrichment") else []
    edf      = results.get("enrichment_df", pd.DataFrame()).copy()

    _pal = sns.color_palette("Set2", max(len(diseases), 3))
    dcol = {d: _pal[i] for i, d in enumerate(diseases)}

    # ── 1. Summary text ───────────────────────────────────────────
    try:
        summary_path = os.path.join(output_dir, "summary.txt")
        with open(summary_path, "w") as fh:
            fh.write(results.get("summary_text", "(no summary available)"))
            fh.write(f"\n\nReport generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            fh.write(f"Output directory : {os.path.abspath(output_dir)}\n")
        print(f"[✓] summary.txt")
    except Exception as e:
        print(f"[⚠] summary.txt failed: {e}")

    # ── 2. Tables ─────────────────────────────────────────────────
    try:
        if not edf.empty:
            edf.to_csv(os.path.join(tables_dir, "enrichment_summary.csv"), index=False)
    except Exception as e:
        print(f"[⚠] enrichment_summary.csv failed: {e}")

    try:
        if "differential" in results and "pairwise_df" in results["differential"]:
            pw = results["differential"]["pairwise_df"]
            if isinstance(pw, pd.DataFrame) and not pw.empty:
                pw.to_csv(os.path.join(tables_dir, "pairwise_differential.csv"), index=False)
    except Exception as e:
        print(f"[⚠] pairwise_differential.csv failed: {e}")

    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty:
            pw_stats.to_csv(os.path.join(tables_dir, "pairwise_disease_stats.csv"), index=False)
    except Exception as e:
        print(f"[⚠] pairwise_disease_stats.csv failed: {e}")

    try:
        prox = results.get("proximity_df", pd.DataFrame())
        if isinstance(prox, pd.DataFrame) and not prox.empty:
            prox.to_csv(os.path.join(tables_dir, "profile_proximity.csv"), index=False)
    except Exception as e:
        print(f"[⚠] profile_proximity.csv failed: {e}")

    try:
        pm = results.get("profile_matrix", pd.DataFrame())
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            pm.to_csv(os.path.join(tables_dir, "profile_matrix.csv"))
    except Exception as e:
        print(f"[⚠] profile_matrix.csv failed: {e}")

    try:
        gd = results.get("gene_detail_df", pd.DataFrame())
        if isinstance(gd, pd.DataFrame) and not gd.empty:
            gd.to_csv(os.path.join(tables_dir, "gene_detail.csv"), index=False)
    except Exception as e:
        print(f"[⚠] gene_detail.csv failed: {e}")

    # NEW: gene influence table
    try:
        inf = results.get("influence_df", pd.DataFrame())
        if isinstance(inf, pd.DataFrame) and not inf.empty:
            inf.to_csv(os.path.join(tables_dir, "gene_influence.csv"), index=False)
    except Exception as e:
        print(f"[⚠] gene_influence.csv failed: {e}")

    # NEW: family-wise FDR
    try:
        fw = results.get("family_wise_fdr_df", pd.DataFrame())
        if isinstance(fw, pd.DataFrame) and not fw.empty:
            fw.to_csv(os.path.join(tables_dir, "family_wise_fdr.csv"), index=False)
    except Exception as e:
        print(f"[⚠] family_wise_fdr.csv failed: {e}")

    # NEW: input manifest
    try:
        prov = results.get("provenance", {})
        if prov:
            with open(os.path.join(tables_dir, "input_manifest.json"), "w") as fh:
                json.dump(prov, fh, indent=2)
    except Exception as e:
        print(f"[⚠] input_manifest.json failed: {e}")

    try:
        meta_rows = []
        for gene in results.get("gene_list", []):
            row = {"Gene": gene}
            for d in diseases:
                row[f"meta_z_{d}"] = results.get("meta_z", {}).get(d, {}).get(gene, np.nan)
            meta_rows.append(row)
        if meta_rows:
            pd.DataFrame(meta_rows).to_csv(
                os.path.join(tables_dir, "meta_z_geneset.csv"), index=False
            )
    except Exception as e:
        print(f"[⚠] meta_z_geneset.csv failed: {e}")

    n_table_files = len([f for f in os.listdir(tables_dir)]) if os.path.isdir(tables_dir) else 0
    print(f"[✓] tables/  ({n_table_files} files)")

    def _save(fig, stem):
        for ext in ("png", "pdf"):
            fig.savefig(
                os.path.join(plots_dir, f"{stem}.{ext}"),
                dpi=200, bbox_inches="tight",
            )
        plt.close(fig)

    pm = results.get("profile_matrix", pd.DataFrame())
    if isinstance(pm, pd.DataFrame) and not pm.empty:
        pm = pm.dropna()

    # ---- 01  Enrichment overview (Stouffer Z + −log10 perm-p) ----
    try:
        if not edf.empty and diseases:
            fig, axes = plt.subplots(1, 2, figsize=(max(12, len(diseases) * 3), 5))

            bars = axes[0].bar(
                edf["Disease"], edf["stouffer_z"],
                color=[dcol[d] for d in edf["Disease"]],
                edgecolor="black", linewidth=0.5,
            )
            axes[0].axhline(0, color="grey", lw=0.8, ls="--")
            for thr, c in [(1.96, "red"), (-1.96, "red")]:
                axes[0].axhline(thr, color=c, lw=0.6, ls=":", alpha=0.6)
            axes[0].set_ylabel("Stouffer Z")
            # Add CI error bars when available
            if "stouffer_ci_low" in edf.columns and edf["stouffer_ci_low"].notna().any():
                for b, lo, hi in zip(bars, edf["stouffer_ci_low"], edf["stouffer_ci_high"]):
                    if not (pd.isna(lo) or pd.isna(hi)):
                        axes[0].plot([b.get_x() + b.get_width()/2]*2, [lo, hi],
                                     color="black", lw=1.2)
                axes[0].set_title("Stouffer Z per Disease (95% bootstrap CI)")
            else:
                axes[0].set_title("Stouffer Z per Disease")
            for b, pv in zip(bars, edf["permutation_p"]):
                if pd.isna(pv):
                    continue
                sig = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else "ns"
                va  = "bottom" if b.get_height() >= 0 else "top"
                axes[0].text(
                    b.get_x() + b.get_width() / 2, b.get_height(),
                    f"p={pv:.4f}\n{sig}", ha="center", va=va, fontsize=9,
                )

            neg_log = -np.log10(edf["permutation_p"].replace(0, 1e-10).fillna(1.0))
            axes[1].bar(
                edf["Disease"], neg_log,
                color=[dcol[d] for d in edf["Disease"]],
                edgecolor="black", linewidth=0.5,
            )
            axes[1].axhline(-np.log10(0.05),  color="red",     lw=0.8, ls="--", label="p = 0.05")
            axes[1].axhline(-np.log10(0.001), color="darkred",  lw=0.8, ls=":",  label="p = 0.001")
            axes[1].set_ylabel("−log₁₀(permutation p)")
            axes[1].set_title("Enrichment Significance")
            axes[1].legend(fontsize=8)

            fig.suptitle(f"{gene_set_name} — Enrichment Overview", fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "01_enrichment_overview")
    except Exception as e:
        print(f"[⚠] Plot 01 (enrichment overview) failed: {e}")
        plt.close("all")

    # ---- 02  Heatmap of profile matrix --------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            vabs = max(abs(pm.values.min()), abs(pm.values.max()))
            fig, ax = plt.subplots(figsize=(max(6, len(diseases) * 2.5), max(6, len(pm) * 0.38)))
            sns.heatmap(
                pm, cmap="RdBu_r", center=0, vmin=-vabs, vmax=vabs,
                annot=True, fmt=".2f", linewidths=0.5, linecolor="white",
                cbar_kws={"label": "meta-Z"}, ax=ax,
            )
            ax.set_title(f"{gene_set_name} — Gene × Disease Z-score Heatmap",
                         fontsize=12, fontweight="bold")
            ax.set_ylabel("Gene"); ax.set_xlabel("Disease")
            plt.tight_layout()
            _save(fig, "02_profile_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 02 (profile heatmap) failed: {e}")
        plt.close("all")

    # ---- 03  Grouped bar chart ----------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            order = pm.loc[pm.abs().mean(axis=1).sort_values(ascending=True).index]
            fig, ax = plt.subplots(figsize=(max(9, len(order) * 0.55), 6))
            x = np.arange(len(order))
            w = 0.8 / len(diseases)
            for i, d in enumerate(diseases):
                off = (i - len(diseases) / 2 + 0.5) * w
                ax.bar(x + off, order[d], w, label=d,
                       color=dcol[d], edgecolor="black", linewidth=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels(order.index, rotation=45, ha="right", fontsize=8)
            ax.axhline(0, color="grey", lw=0.8)
            ax.set_ylabel("meta-Z score")
            ax.set_title(f"{gene_set_name} — Gene-level Z-scores by Disease",
                         fontsize=12, fontweight="bold")
            ax.legend(title="Disease")
            plt.tight_layout()
            _save(fig, "03_gene_barplot")
    except Exception as e:
        print(f"[⚠] Plot 03 (gene barplot) failed: {e}")
        plt.close("all")

    # ---- 04  Pairwise scatter ------------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty and len(diseases) >= 2:
            pairs  = list(combinations(diseases, 2))
            fig, axes_sc = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5.5))
            if len(pairs) == 1:
                axes_sc = [axes_sc]
            prox = results.get("proximity_df", None)

            for idx, (d1, d2) in enumerate(pairs):
                ax = axes_sc[idx]
                xv, yv = pm[d1].values, pm[d2].values
                ax.scatter(xv, yv, c="steelblue", edgecolor="black", lw=0.3, s=50, alpha=0.8)
                for g, xi, yi in zip(pm.index, xv, yv):
                    ax.annotate(g, (xi, yi), fontsize=6, alpha=0.7,
                                xytext=(3, 3), textcoords="offset points")
                m = ~(np.isnan(xv) | np.isnan(yv))
                if m.sum() > 2:
                    coef = np.polyfit(xv[m], yv[m], 1)
                    xl   = np.linspace(xv[m].min(), xv[m].max(), 100)
                    ax.plot(xl, np.polyval(coef, xl), "r--", alpha=0.6, lw=1)
                if prox is not None and isinstance(prox, pd.DataFrame) and not prox.empty:
                    row = prox[((prox["Disease_1"] == d1) & (prox["Disease_2"] == d2)) |
                               ((prox["Disease_1"] == d2) & (prox["Disease_2"] == d1))]
                    if not row.empty:
                        r = row["Pearson_r"].values[0]; p = row["Pearson_p"].values[0]
                        ax.set_title(f"{d1} vs {d2}\nr = {r:.3f}, p = {p:.4f}", fontsize=11)
                    else:
                        ax.set_title(f"{d1} vs {d2}")
                else:
                    ax.set_title(f"{d1} vs {d2}")
                ax.set_xlabel(f"{d1} meta-Z"); ax.set_ylabel(f"{d2} meta-Z")
                ax.axhline(0, color="grey", lw=0.5, ls="--")
                ax.axvline(0, color="grey", lw=0.5, ls="--")

            fig.suptitle(f"{gene_set_name} — Pairwise Z-score Correlations",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "04_pairwise_scatter")
    except Exception as e:
        print(f"[⚠] Plot 04 (pairwise scatter) failed: {e}")
        plt.close("all")

    # ---- 05  Z-range lollipop -----------------------------------
    try:
        gdf = results.get("gene_detail_df", pd.DataFrame())
        if isinstance(gdf, pd.DataFrame) and not gdf.empty and "Z_range" in gdf.columns:
            gdf = gdf.dropna(subset=["Z_range"]).sort_values("Z_range", ascending=False)
            if not gdf.empty:
                fig, ax = plt.subplots(figsize=(8, max(5, len(gdf) * 0.38)))
                y = np.arange(len(gdf))
                ax.barh(y, gdf["Z_range"], color="salmon", edgecolor="black", lw=0.3, height=0.6)
                ax.set_yticks(y)
                ax.set_yticklabels(gdf["Gene"], fontsize=9)
                ax.set_xlabel("Z-score range (max − min across diseases)")
                ax.set_title(f"{gene_set_name} — Cross-Disease Variability per Gene",
                             fontsize=12, fontweight="bold")
                ax.invert_yaxis()
                plt.tight_layout()
                _save(fig, "05_gene_z_range")
    except Exception as e:
        print(f"[⚠] Plot 05 (gene z-range) failed: {e}")
        plt.close("all")

    # ---- 06  Box + strip plot -----------------------------------
    try:
        if isinstance(pm, pd.DataFrame) and not pm.empty:
            fig, ax = plt.subplots(figsize=(max(6, len(diseases) * 2.2), 5.5))
            bp = ax.boxplot(
                [pm[d].values for d in diseases],
                labels=diseases, patch_artist=True, showmeans=True,
                meanprops=dict(marker="D", markerfacecolor="red", markersize=6),
            )
            for patch, d in zip(bp["boxes"], diseases):
                patch.set_facecolor(dcol[d]); patch.set_alpha(0.55)
            for i, d in enumerate(diseases):
                jitter = np.random.default_rng(42).normal(0, 0.04, len(pm))
                ax.scatter(
                    np.full(len(pm), i + 1) + jitter, pm[d].values,
                    c="black", s=18, alpha=0.5, zorder=3,
                )
            ax.axhline(0, color="grey", lw=0.8, ls="--")
            ax.set_ylabel("meta-Z score")
            ax.set_title(f"{gene_set_name} — Z-score Distributions",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            _save(fig, "06_z_distribution_boxplot")
    except Exception as e:
        print(f"[⚠] Plot 06 (z-distribution boxplot) failed: {e}")
        plt.close("all")

    # ---- 07  Enrichment metrics comparison -----------------------
    try:
        if len(diseases) >= 2 and not edf.empty:
            metrics = ["stouffer_z", "mean_z", "median_z", "mean_abs_z"]
            metric_labels = ["Stouffer Z", "Mean Z", "Median Z", "Mean |Z|"]
            fig, ax = plt.subplots(figsize=(max(10, len(diseases) * 2.5), 5))
            x = np.arange(len(metrics))
            w = 0.8 / len(diseases)
            for i, d in enumerate(diseases):
                row  = edf[edf["Disease"] == d].iloc[0]
                vals = [float(row[m]) for m in metrics]
                off  = (i - len(diseases) / 2 + 0.5) * w
                ax.bar(x + off, vals, w, label=d,
                       color=dcol[d], edgecolor="black", linewidth=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels(metric_labels, fontsize=10)
            ax.axhline(0, color="grey", lw=0.8, ls="--")
            ax.set_ylabel("Value")
            ax.set_title(f"{gene_set_name} — Enrichment Metrics Comparison",
                         fontsize=12, fontweight="bold")
            ax.legend(title="Disease")
            plt.tight_layout()
            _save(fig, "07_enrichment_metrics")
    except Exception as e:
        print(f"[⚠] Plot 07 (enrichment metrics) failed: {e}")
        plt.close("all")

    # ---- 08  Pairwise stats heatmap -----------------------------
    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty and len(diseases) >= 2:
            p_candidates = [
                ("KS_p",               "KS 2-sample"),
                ("Welch_p",            "Welch t"),
                ("Paired_Wilcoxon_p",  "Paired Wilcoxon"),
                ("Paired_t_p",         "Paired t"),
                ("Permutation_p",      "Permutation"),
            ]
            p_metrics = [(col, lbl) for col, lbl in p_candidates
                         if col in pw_stats.columns and pw_stats[col].notna().any()]
            d_candidates = [
                ("Cohens_d_unpaired", "Cohen's d (unpaired)"),
                ("Cohens_d_paired",   "Cohen's d (paired)"),
            ]
            d_metrics = [(col, lbl) for col, lbl in d_candidates
                         if col in pw_stats.columns and pw_stats[col].notna().any()]
            n_panels = len(p_metrics) + len(d_metrics)
            if n_panels > 0:
                fig, axes_pw = plt.subplots(
                    1, n_panels,
                    figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
                )
                if n_panels == 1:
                    axes_pw = [axes_pw]
                panel_idx = 0
                for col, lbl in p_metrics:
                    ax = axes_pw[panel_idx]; panel_idx += 1
                    mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            neg_log = -np.log10(max(val, 1e-300))
                            mat.loc[d1, d2] = neg_log
                            mat.loc[d2, d1] = neg_log
                    for d in diseases:
                        mat.loc[d, d] = 0.0
                    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                                square=True, ax=ax, linewidths=0.5, linecolor="white", vmin=0)
                    ax.set_title(f"−log₁₀(p)\n{lbl}", fontsize=10)
                for col, lbl in d_metrics:
                    ax = axes_pw[panel_idx]; panel_idx += 1
                    mat = pd.DataFrame(0.0, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            mat.loc[d1, d2] =  val
                            mat.loc[d2, d1] = -val
                    vabs = max(abs(mat.values.min()), abs(mat.values.max()), 0.01)
                    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
                                center=0, vmin=-vabs, vmax=vabs, square=True, ax=ax,
                                linewidths=0.5, linecolor="white")
                    ax.set_title(lbl, fontsize=10)
                fig.suptitle(f"{gene_set_name} — Pairwise Disease Stat Tests",
                             fontsize=13, fontweight="bold")
                plt.tight_layout()
                _save(fig, "08_pairwise_stats_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 08 (pairwise stats heatmap) failed: {e}")
        plt.close("all")

    # ---- 09  Concordance heatmap --------------------------------
    try:
        pw_stats = results.get("pairwise_stats_df", pd.DataFrame())
        if isinstance(pw_stats, pd.DataFrame) and not pw_stats.empty and len(diseases) >= 2:
            conc_candidates = [
                ("sign_concordance_rate", "Sign Concordance Rate", "YlGn",    0.0, 1.0, 0.5),
                ("Lin_CCC",               "Lin's CCC",            "RdBu_r", -1.0, 1.0, 0.0),
                ("Kendall_tau",            "Kendall τ",            "RdBu_r", -1.0, 1.0, 0.0),
            ]
            conc_metrics_avail = [
                (col, lbl, cmap, vmin, vmax, center)
                for col, lbl, cmap, vmin, vmax, center in conc_candidates
                if col in pw_stats.columns and pw_stats[col].notna().any()
            ]
            conc_p_avail = []
            if "sign_concordance_p" in pw_stats.columns and pw_stats["sign_concordance_p"].notna().any():
                conc_p_avail.append(("sign_concordance_p", "Sign Conc. −log₁₀(p)"))
            if "Kendall_p" in pw_stats.columns and pw_stats["Kendall_p"].notna().any():
                conc_p_avail.append(("Kendall_p", "Kendall τ −log₁₀(p)"))
            n_panels = len(conc_metrics_avail) + len(conc_p_avail)
            if n_panels > 0:
                fig, axes_c = plt.subplots(
                    1, n_panels,
                    figsize=(max(5 * n_panels, 10), max(4, len(diseases) * 1.5))
                )
                if n_panels == 1:
                    axes_c = [axes_c]
                pi = 0
                for col, lbl, cmap, vmin_val, vmax_val, center_val in conc_metrics_avail:
                    ax = axes_c[pi]; pi += 1
                    mat = pd.DataFrame(1.0, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            mat.loc[d1, d2] = val
                            mat.loc[d2, d1] = val
                    sns.heatmap(
                        mat.astype(float), annot=True, fmt=".3f", cmap=cmap,
                        center=center_val, vmin=vmin_val, vmax=vmax_val,
                        square=True, ax=ax, linewidths=0.5, linecolor="white",
                    )
                    ax.set_title(lbl, fontsize=10)
                for col, lbl in conc_p_avail:
                    ax = axes_c[pi]; pi += 1
                    mat = pd.DataFrame(np.nan, index=diseases, columns=diseases)
                    for _, rw in pw_stats.iterrows():
                        d1, d2 = rw["Disease_1"], rw["Disease_2"]
                        val = rw[col]
                        if d1 in diseases and d2 in diseases and not np.isnan(val):
                            neg_log = -np.log10(max(val, 1e-300))
                            mat.loc[d1, d2] = neg_log
                            mat.loc[d2, d1] = neg_log
                    for d in diseases:
                        mat.loc[d, d] = 0.0
                    sns.heatmap(
                        mat.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
                        square=True, ax=ax, linewidths=0.5, linecolor="white", vmin=0,
                    )
                    ax.set_title(lbl, fontsize=10)
                fig.suptitle(f"{gene_set_name} — Concordance Metrics",
                             fontsize=13, fontweight="bold")
                plt.tight_layout()
                _save(fig, "09_concordance_heatmap")
    except Exception as e:
        print(f"[⚠] Plot 09 (concordance heatmap) failed: {e}")
        plt.close("all")

    # ---- 10  QQ-plot (NEW) --------------------------------------
    try:
        meta_z = results.get("meta_z", {})
        gene_list = results.get("gene_list", [])
        if meta_z and gene_list:
            diseases_local = list(meta_z.keys())
            n = len(diseases_local)
            fig, axes = plt.subplots(1, n, figsize=(max(5 * n, 6), 5))
            if n == 1:
                axes = [axes]
            for ax, d in zip(axes, diseases_local):
                gs_z = np.array([meta_z[d].get(g, np.nan) for g in gene_list])
                gs_z = gs_z[~np.isnan(gs_z)]
                if len(gs_z) < 3:
                    ax.text(0.5, 0.5, f"{d}: n<3", ha="center", va="center",
                            transform=ax.transAxes); continue
                observed = np.sort(gs_z)
                theoretical = stats.norm.ppf((np.arange(1, len(observed) + 1) - 0.5) / len(observed))
                ax.scatter(theoretical, observed, alpha=0.7, s=30,
                           c=dcol.get(d, "steelblue"), edgecolor="black", lw=0.3)
                lims = [min(theoretical.min(), observed.min()),
                        max(theoretical.max(), observed.max())]
                ax.plot(lims, lims, "r--", lw=1, label="N(0,1) reference")
                ax.axhline(0, color="gray", lw=0.5, ls=":")
                ax.set_xlabel("Theoretical N(0,1) quantile")
                ax.set_ylabel("Observed gene-set Z")
                ax.set_title(f"{d} (n={len(observed)})", fontsize=11)
                ax.legend(fontsize=8)
            fig.suptitle(f"{gene_set_name} — QQ-plot vs N(0,1)",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "10_qq_plot")
    except Exception as e:
        print(f"[⚠] Plot 10 (QQ-plot) failed: {e}")
        plt.close("all")

    # ---- 11  Dendrogram (NEW) ------------------------------------
    try:
        prox = results.get("proximity_df", pd.DataFrame())
        if isinstance(prox, pd.DataFrame) and not prox.empty and len(diseases) >= 3:
            from scipy.cluster.hierarchy import linkage, dendrogram
            from scipy.spatial.distance import squareform
            n = len(diseases)
            dmat = np.zeros((n, n))
            idx = {d: i for i, d in enumerate(diseases)}
            for _, row in prox.iterrows():
                d1, d2 = row["Disease_1"], row["Disease_2"]
                if d1 in idx and d2 in idx:
                    dist = max(0.0, 1 - float(row["Pearson_r"]))
                    dmat[idx[d1], idx[d2]] = dist
                    dmat[idx[d2], idx[d1]] = dist
            condensed = squareform(dmat, checks=False)
            Z = linkage(condensed, method="average")
            fig, ax = plt.subplots(figsize=(max(6, n * 1.3), 5))
            dendrogram(Z, labels=diseases, ax=ax, color_threshold=0)
            ax.set_ylabel("Distance (1 − Pearson r)")
            ax.set_title(f"{gene_set_name} — Disease Hierarchical Clustering",
                         fontsize=12, fontweight="bold")
            plt.tight_layout()
            _save(fig, "11_dendrogram")
    except Exception as e:
        print(f"[⚠] Plot 11 (dendrogram) failed: {e}")
        plt.close("all")

    # ---- 12  Gene influence (NEW) --------------------------------
    try:
        inf = results.get("influence_df", pd.DataFrame())
        if isinstance(inf, pd.DataFrame) and not inf.empty:
            diseases_local = list(inf["Disease"].unique())
            n = len(diseases_local)
            fig, axes = plt.subplots(1, n, figsize=(max(6 * n, 8), max(5, 15 * 0.35)))
            if n == 1:
                axes = [axes]
            for ax, d in zip(axes, diseases_local):
                sub = inf[inf["Disease"] == d].copy()
                sub = sub.reindex(sub["Delta_Stouffer"].abs().sort_values(ascending=False).index)
                sub = sub.head(15).iloc[::-1]
                if sub.empty:
                    continue
                colors = ["#e74c3c" if x else "#3498db" for x in sub["Influential"]]
                ax.barh(sub["Gene"], sub["Delta_Stouffer"], color=colors,
                        edgecolor="black", lw=0.3)
                ax.axvline(0, color="gray", ls="--", lw=0.8)
                ax.set_xlabel("Δ Stouffer Z (full − LOO)")
                ax.set_title(f"{d}: top influential genes", fontsize=11)
            fig.suptitle(f"{gene_set_name} — Gene-level LOO Influence",
                         fontsize=13, fontweight="bold")
            plt.tight_layout()
            _save(fig, "12_gene_influence")
    except Exception as e:
        print(f"[⚠] Plot 12 (gene influence) failed: {e}")
        plt.close("all")

    n_plots = len([f for f in os.listdir(plots_dir) if f.endswith(".png")]) if os.path.isdir(plots_dir) else 0
    print(f"[✓] plots/   ({n_plots} PNGs + {n_plots} PDFs)")
    print(f"\n{'='*60}")
    print(f"  Report saved → {os.path.abspath(output_dir)}/")
    print(f"{'='*60}")

    return output_dir



### Loading

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/mdd" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/bip" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/scz" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/ocd" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/ptsd" /content/ -r

In [3]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/bip_afr" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/bip_eas" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/bip_eur" /content/ -r

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/scz2026_eur" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/scz_cloz" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/scz_agran" /content/ -r

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/alz" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/anx_cc" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/hoarding" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/an" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/tics" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity/ppd" /content/ -r

In [4]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GMT/ageing_polycomb" /content/ -r
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GMT/GenAge_Human" /content/ -r

cp: cannot open '/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GMT/ageing_polycomb/Standardized KEGG Pathways Associated with Each Module.gdoc' for reading: Operation not supported
cp: cannot open '/content/drive/MyDrive/Dr Uccello/00_Studies/Z_GMT/GenAge_Human/GenAge Build 21: Release Notes.gdoc' for reading: Operation not supported


In [5]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database" /content/ -r

# AP-related genesets

In [ ]:
"""
Pipeline to parse all drugs under ATC group N05A (Antipsychotics)
from the WHO ATC/DDD Index (https://atcddd.fhi.no/atc_ddd_index/).

Strategy:
    1. Fetch the N05A index page.
    2. Extract the child subgroup codes. For N05A these typically include:
         N05AA  Phenothiazines with aliphatic side-chain
         N05AB  Phenothiazines with piperazine structure
         N05AC  Phenothiazines with piperidine structure
         N05AD  Butyrophenone derivatives
         N05AE  Indole derivatives
         N05AF  Thioxanthene derivatives
         N05AG  Diphenylbutylpiperidine derivatives
         N05AH  Diazepines, oxazepines, thiazepines and oxepines
         N05AL  Benzamides
         N05AN  Lithium
         N05AX  Other antipsychotics
       (Codes are discovered dynamically, so any additions/removals in the
        2026 index are handled automatically.)
    3. Visit each subgroup page and parse the drug-level rows
       (ATC code + name, and where available DDD / unit / route / note).
    4. Return a flat list of drug dicts.
"""

import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://atcddd.fhi.no/atc_ddd_index/"
HEADERS = {"User-Agent": "Mozilla/5.0 (ATC-DDD-parser; research use)"}


# ----------------------------------------------------------------------
# Low-level fetch
# ----------------------------------------------------------------------
def fetch(code: str, session: requests.Session) -> BeautifulSoup:
    """Fetch a single ATC code page and return parsed soup."""
    resp = session.get(BASE_URL, params={"code": code}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ----------------------------------------------------------------------
# Discover subgroup codes under a parent code
# ----------------------------------------------------------------------
def get_child_codes(soup: BeautifulSoup, parent_code: str) -> list[str]:
    """
    Extract immediate child ATC codes from a page.
    Children appear as links of the form '?code=N05AA'.

    Note: N05A subgroups use letter suffixes (N05AA, N05AB, ... N05AX),
    so the 'one level deeper' rule (len == len(parent)+1) still applies.
    """
    codes = []
    for a in soup.select("a[href*='code=']"):
        href = a["href"]
        code = href.split("code=")[-1].split("&")[0].strip()
        # keep codes that extend the parent (e.g. N05AA under N05A)
        if (
            code.startswith(parent_code)
            and code != parent_code
            and len(code) == len(parent_code) + 1   # one level deeper
            and code not in codes
        ):
            codes.append(code)
    return codes


# ----------------------------------------------------------------------
# Parse drug-level rows from a subgroup page
# ----------------------------------------------------------------------
def parse_drugs(soup: BeautifulSoup, subgroup_code: str) -> list[dict]:
    """
    On a subgroup (chemical subgroup) page the substances are listed in a
    table whose rows look like:
        ATC code | Name | DDD | Unit | Adm.R | Note
    The 7-character codes (e.g. N05AH04 = quetiapine) identify chemical
    substances.
    """
    drugs = []
    for row in soup.select("table tr"):
        cells = [c.get_text(strip=True) for c in row.find_all("td")]
        if not cells:
            continue

        atc_code = cells[0]
        # A substance-level code is the subgroup code + 2 digits, e.g. N05AH04
        if not (atc_code.startswith(subgroup_code) and len(atc_code) == len(subgroup_code) + 2):
            continue

        drug = {
            "atc_code": atc_code,
            "name":     cells[1] if len(cells) > 1 else None,
            "ddd":      cells[2] if len(cells) > 2 and cells[2] else None,
            "unit":     cells[3] if len(cells) > 3 and cells[3] else None,
            "route":    cells[4] if len(cells) > 4 and cells[4] else None,
            "note":     cells[5] if len(cells) > 5 and cells[5] else None,
            "subgroup": subgroup_code,
        }
        # Some substances span multiple DDD/route rows (very common for
        # antipsychotics, e.g. oral vs parenteral vs depot forms); the name
        # cell is empty on continuation rows, so carry the previous name forward.
        if not drug["name"] and drugs:
            drug["name"] = drugs[-1]["name"]
        drugs.append(drug)
    return drugs


# ----------------------------------------------------------------------
# Orchestration
# ----------------------------------------------------------------------
def scrape_atc(parent_code: str = "N05A", polite_delay: float = 0.5) -> list[dict]:
    with requests.Session() as session:
        root = fetch(parent_code, session)
        subgroups = get_child_codes(root, parent_code)

        all_drugs = []
        for sub in subgroups:
            sub_soup = fetch(sub, session)
            all_drugs.extend(parse_drugs(sub_soup, sub))
            time.sleep(polite_delay)   # be courteous to the server
        return all_drugs


# ----------------------------------------------------------------------
if __name__ == "__main__":
    drugs = scrape_atc("N05A")
    print(f"Found {len(drugs)} drug entries under N05A (Antipsychotics)\n")
    for d in drugs:
        ddd = f"{d['ddd']} {d['unit']} ({d['route']})" if d["ddd"] else "no DDD"
        print(f"{d['atc_code']:<9} {d['name']:<30} {ddd}")

    # If you only want the plain list of drug names:
    names = sorted({d["name"] for d in drugs if d["name"]})
    print("\nUnique drug names:")
    print(names)

Found 70 drug entries under N05A (Antipsychotics)

N05AA01   chlorpromazine                 0.3 g (O)
N05AA02   levomepromazine                0.3 g (O)
N05AA03   promazine                      0.3 g (O)
N05AA04   acepromazine                   0.1 g (O)
N05AA05   triflupromazine                0.1 g (O)
N05AA06   cyamemazine                    no DDD
N05AA07   chlorproethazine               no DDD
N05AB01   dixyrazine                     50 mg (O)
N05AB02   fluphenazine                   10 mg (O)
N05AB03   perphenazine                   30 mg (O)
N05AB04   prochlorperazine               0.1 g (O)
N05AB05   thiopropazate                  60 mg (O)
N05AB06   trifluoperazine                20 mg (O)
N05AB07   acetophenazine                 50 mg (O)
N05AB08   thioproperazine                75 mg (O)
N05AB09   butaperazine                   10 mg (O)
N05AB10   perazine                       0.1 g (O)
N05AC01   periciazine                    50 mg (O)
N05AC02   thioridazine               

In [ ]:
print(", ".join(names))

acepromazine, acetophenazine, amisulpride, aripiprazole, asenapine, benperidol, brexpiprazole, bromperidol, butaperazine, cariprazine, chlorproethazine, chlorpromazine, chlorprothixene, clopenthixol, clotiapine, clozapine, cyamemazine, dixyrazine, droperidol, fluanisone, flupentixol, fluphenazine, fluspirilene, haloperidol, iloperidone, levomepromazine, levosulpiride, lithium, loxapine, lumateperone, lurasidone, melperone, mesoridazine, molindone, moperone, mosapramine, olanzapine, olanzapine and samidorphan, oxypertine, paliperidone, penfluridol, perazine, periciazine, perphenazine, pimavanserin, pimozide, pipamperone, pipotiazine, prochlorperazine, promazine, prothipendyl, quetiapine, remoxipride, risperidone, sertindole, sulpiride, sultopride, thiopropazate, thioproperazine, thioridazine, tiapride, tiotixene, trifluoperazine, trifluperidol, triflupromazine, veralipride, xanomeline and trospium, ziprasidone, zotepine, zuclopenthixol


### Search and export in https://dgidb.org/

### Converting into gmt

In [ ]:
# ======================================================================
# Pipeline: drug_interaction_results.tsv  ->  drug gene sets (.gmt)
# Designed to run in Google Colab.
#
# GMT format (tab-separated, one gene set per line):
#   <set_name>\t<description>\t<gene1>\t<gene2>\t...\t<geneN>
# ======================================================================

import pandas as pd
from collections import OrderedDict


# ----------------------------------------------------------------------
# 1. Load the data
#    In Colab, either upload the file with the snippet below, or set
#    INPUT_PATH to a path already in your Colab session / Drive.
# ----------------------------------------------------------------------
USE_UPLOAD = False          # set False if file is already on disk
INPUT_PATH = "/content/Z_database/drug_interaction_antipsychotics.tsv"
OUTPUT_PATH = "AP_related.gmt"

if USE_UPLOAD:
    try:
        from google.colab import files
        uploaded = files.upload()          # prompts a file picker
        INPUT_PATH = next(iter(uploaded))  # use the uploaded filename
    except ImportError:
        print("Not in Colab — falling back to INPUT_PATH on disk.")

df = pd.read_csv(INPUT_PATH, sep="\t")

# Normalise column names (strip whitespace, lower-case for matching)
df.columns = [c.strip() for c in df.columns]
expected = {"drug", "gene", "regulatory approval", "interaction score"}
missing = expected - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}")


# ----------------------------------------------------------------------
# 2. Clean / filter (all configurable)
# ----------------------------------------------------------------------
SCORE_THRESHOLD = 0.0          # keep interactions with score >= this value
APPROVED_ONLY   = False        # if True, keep only 'Approved' interactions
DEDUPE_GENES    = True         # drop duplicate genes within a drug

df["drug"] = df["drug"].astype(str).str.strip()
df["gene"] = df["gene"].astype(str).str.strip()
df["interaction score"] = pd.to_numeric(df["interaction score"], errors="coerce")

mask = df["interaction score"] >= SCORE_THRESHOLD
if APPROVED_ONLY:
    mask &= df["regulatory approval"].str.strip().str.lower() == "approved"
df = df[mask].dropna(subset=["gene", "interaction score"])


# ----------------------------------------------------------------------
# 3. Build gene sets: one set per drug
#    Genes are ordered by descending interaction score (strongest first).
# ----------------------------------------------------------------------
gene_sets = OrderedDict()
descriptions = {}

for drug, sub in df.groupby("drug", sort=True):
    sub = sub.sort_values("interaction score", ascending=False)
    genes = sub["gene"].tolist()
    if DEDUPE_GENES:                       # keep first (highest-scoring) occurrence
        genes = list(OrderedDict.fromkeys(genes))
    gene_sets[drug] = genes

    # Description column: record approval status + gene count.
    approvals = sub["regulatory approval"].str.strip().unique()
    approval_str = "/".join(sorted(approvals))
    descriptions[drug] = f"approval={approval_str};n_genes={len(genes)}"


# ----------------------------------------------------------------------
# 4. Write the GMT file
# ----------------------------------------------------------------------
def write_gmt(path, gene_sets, descriptions):
    with open(path, "w", encoding="utf-8") as fh:
        for name, genes in gene_sets.items():
            if not genes:
                continue
            row = [name, descriptions.get(name, "na")] + genes
            fh.write("\t".join(row) + "\n")

write_gmt(OUTPUT_PATH, gene_sets, descriptions)

print(f"Wrote {len(gene_sets)} gene sets to {OUTPUT_PATH}")
print(f"Total interactions kept: {len(df)}")


# ----------------------------------------------------------------------
# 5. Quick preview + download
# ----------------------------------------------------------------------
print("\n--- Preview (first 3 lines) ---")
with open(OUTPUT_PATH) as fh:
    for i, line in zip(range(3), fh):
        parts = line.rstrip("\n").split("\t")
        print(f"{parts[0]} | {parts[1]} | {len(parts) - 2} genes: {parts[2:7]} ...")

# Trigger a browser download in Colab
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except ImportError:
    pass

Wrote 53 gene sets to AP_related.gmt
Total interactions kept: 987

--- Preview (first 3 lines) ---
ACEPROMAZINE | approval=Approved;n_genes=2 | 2 genes: ['CYP1A2', 'CYP2D6'] ...
ACETOPHENAZINE | approval=Approved;n_genes=3 | 3 genes: ['DRD2', 'DRD1', 'AR'] ...
AMISULPRIDE | approval=Approved;n_genes=26 | 26 genes: ['ANKS1B', 'SNAP25', 'HTR5A', 'HTR1E', 'SH2B1'] ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Flat-unique

In [6]:
# ======================================================================
# Pipeline: parse a .gmt file  ->  flatten to one gene list  ->  dict
# Runs anywhere (plain Python / Colab).
# ======================================================================

from collections import OrderedDict

INPUT_PATH = "/content/Z_database/AP_related.gmt"


# ----------------------------------------------------------------------
# 1. Parse the GMT file
#    Each line: <set_name>\t<description>\t<gene1>\t<gene2>\t...
# ----------------------------------------------------------------------
def parse_gmt(path):
    """Return a dict: {set_name: {'description': str, 'genes': [..]}}."""
    parsed = OrderedDict()
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) < 3:           # need name + description + >=1 gene
                continue
            name, description, *genes = parts
            genes = [g.strip() for g in genes if g.strip()]
            parsed[name.strip()] = {"description": description.strip(),
                                    "genes": genes}
    return parsed


gmt = parse_gmt(INPUT_PATH)
print(f"Parsed {len(gmt)} gene sets.")


# ----------------------------------------------------------------------
# 2. Flatten all genes, then keep unique (first-appearance order)
# ----------------------------------------------------------------------
flat_list_all = [gene for entry in gmt.values() for gene in entry["genes"]]
flat_unique   = list(OrderedDict.fromkeys(flat_list_all))

print(f"Total gene occurrences (with duplicates): {len(flat_list_all)}")
print(f"Unique genes: {len(flat_unique)}")


# ----------------------------------------------------------------------
# 3. Convert flat_unique into a dict  (gene -> index)
# ----------------------------------------------------------------------
flat_unique_dict = {gene: i for i, gene in enumerate(flat_unique)}


# ----------------------------------------------------------------------
# 4. Preview
# ----------------------------------------------------------------------
print("\n--- flat_unique_dict (first 10 items) ---")
for gene, idx in list(flat_unique_dict.items())[:10]:
    print(f"{idx:3d}  {gene}")

Parsed 53 gene sets.
Total gene occurrences (with duplicates): 987
Unique genes: 322

--- flat_unique_dict (first 10 items) ---
  0  CYP1A2
  1  CYP2D6
  2  DRD2
  3  DRD1
  4  AR
  5  ANKS1B
  6  SNAP25
  7  HTR5A
  8  HTR1E
  9  SH2B1


# Combining gene sets

In [7]:
import os
import glob
import re

# Folder containing your GMT and/or TXT files
folder_path = "/content/ageing_polycomb"  # <-- change this to your folder

# Dictionary to hold all gene sets {SET_NAME: [genes...]}
gene_sets = {}

def sanitize(name: str) -> str:
    """Make a string safe to use as a Python variable name (UPPER_CASE)."""
    name = name.lstrip("\ufeff")                 # strip BOM if present
    name = re.sub(r'[^0-9a-zA-Z_]', '_', name)   # replace illegal chars
    name = re.sub(r'_+', '_', name).strip('_')   # collapse underscores
    if name and name[0].isdigit():
        name = "_" + name
    return name.upper()

def split_fields(line: str):
    """
    Split a GMT-style line into fields.
    Prefers tabs; falls back to runs of 2+ whitespace characters
    (handles files where tabs were converted to spaces).
    """
    parts = line.split("\t")
    if len(parts) < 3:
        # Fall back: split on runs of 2+ whitespace
        parts = re.split(r'\s{2,}', line)
    return [p.strip() for p in parts if p.strip()]

def parse_gmt_line(line: str):
    """
    Parse a single GMT-format line:
        SET_NAME <sep> DESCRIPTION <sep> GENE1 <sep> GENE2 ...
    Returns (set_name, [genes]) or None if not a valid GMT line.
    """
    parts = split_fields(line)
    if len(parts) < 3:
        return None
    set_name = sanitize(parts[0])
    genes = parts[2:]
    if genes:
        return set_name, genes
    return None

def parse_gmt_file(file_path: str):
    """Parse a .gmt file (one gene set per line)."""
    parsed_sets = {}
    with open(file_path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.rstrip("\n").rstrip("\r")
            if not line.strip():
                continue
            result = parse_gmt_line(line)
            if result:
                set_name, genes = result
                parsed_sets[set_name] = genes
    return parsed_sets

def looks_like_gmt(file_path: str) -> bool:
    """
    Heuristic: a TXT file is GMT-style if its non-empty lines
    typically contain a tab or a run of 2+ spaces separating
    at least 3 fields.
    """
    with open(file_path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.rstrip("\n").rstrip("\r")
            if not line.strip():
                continue
            parts = split_fields(line)
            return len(parts) >= 3
    return False

def parse_txt_file(file_path: str):
    """
    Parse a .txt file.

    - If the content looks like GMT (tab/multi-space separated, >=3 fields
      per line), parse it as GMT (multiple sets, one per line).
    - Otherwise, treat it as a simple one-gene-per-line list, using the
      filename as the set name.
    """
    if looks_like_gmt(file_path):
        return parse_gmt_file(file_path)

    # Simple one-gene-per-line list
    set_name = sanitize(os.path.splitext(os.path.basename(file_path))[0])
    genes = []
    with open(file_path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.lower() in {"gene", "genes", "symbol", "gene_symbol"}:
                continue
            genes.append(line)
    return {set_name: genes} if genes else {}

# Find all .gmt and .txt files in the folder
gmt_files = glob.glob(os.path.join(folder_path, "*.gmt"))
txt_files = glob.glob(os.path.join(folder_path, "*.txt"))

all_files = gmt_files + txt_files

print(f"Found {len(gmt_files)} GMT files")
print(f"Found {len(txt_files)} TXT files")
print(f"Found {len(all_files)} total files\n")

for file_path in all_files:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".gmt":
        parsed = parse_gmt_file(file_path)
    elif ext == ".txt":
        parsed = parse_txt_file(file_path)
    else:
        continue
    gene_sets.update(parsed)

# ---- Inject each gene set as a global Python variable ----
for set_name, genes in gene_sets.items():
    globals()[set_name] = genes

print(f"Loaded {len(gene_sets)} gene sets as Python variables.\n")

for set_name, genes in list(gene_sets.items())[:5]:
    print(f"{set_name}: {len(genes)} genes — first 5: {genes[:5]}")

# ---- Optional: write them out as a .py file in the desired format ----
output_py = "/content/gene_sets.py"

def format_list(genes, per_line=8, indent="    "):
    lines = []
    for i in range(0, len(genes), per_line):
        chunk = ", ".join(f'"{g}"' for g in genes[i:i+per_line])
        lines.append(indent + chunk)
    return ",\n".join(lines)

with open(output_py, "w") as out:
    out.write("# Auto-generated gene sets from GMT and TXT files\n\n")
    for set_name, genes in gene_sets.items():
        out.write(f"{set_name} = [\n{format_list(genes)}\n]\n\n")

print(f"\nAll gene sets written to: {output_py}")

Found 3 GMT files
Found 0 TXT files
Found 3 total files

Loaded 16 gene sets as Python variables.

HSA04142_LYSOSOME: 250 genes — first 5: ['ABCA2', 'ABCA5', 'ABCB9', 'ACP2', 'ACP5']
HSA04216_FERROPTOSIS: 42 genes — first 5: ['ACSL1', 'ACSL3', 'ACSL4', 'ACSL5', 'ACSL6']
HSA04810_REGULATION_OF_ACTIN_CYTOSKELETON: 232 genes — first 5: ['ABI2', 'ACTB', 'ACTG1', 'ACTN1', 'ACTN4']
HSA03020_RNA_POLYMERASE: 34 genes — first 5: ['CRCP', 'POLR1A', 'POLR1B', 'POLR1C', 'POLR1D']
HSA04614_RENIN_ANGIOTENSIN_SYSTEM: 23 genes — first 5: ['ACE', 'ACE2', 'AGT', 'AGTR1', 'AGTR2']

All gene sets written to: /content/gene_sets.py


In [8]:
print(gene_sets.keys())

dict_keys(['HSA04142_LYSOSOME', 'HSA04216_FERROPTOSIS', 'HSA04810_REGULATION_OF_ACTIN_CYTOSKELETON', 'HSA03020_RNA_POLYMERASE', 'HSA04614_RENIN_ANGIOTENSIN_SYSTEM', 'HSA03040_SPLICEOSOME', 'HSA04514_CELL_ADHESION_MOLECULES_CAMS', 'HSA04360_AXON_GUIDANCE', 'HSA04520_ADHERENS_JUNCTION', 'HSA04721_SYNAPTIC_VESICLE_CYCLE', 'HSA04724_GLUTAMATERGIC_SYNAPSE', 'HSA04720_LONG_TERM_POTENTIATION', 'HSA04080_NEUROACTIVE_LIGAND_RECEPTOR_INTERACTION', 'HSA04725_CHOLINERGIC_SYNAPSE', 'HSA04726_SEROTONERGIC_SYNAPSE', 'HSA04727_GABAERGIC_SYNAPSE'])


# For GeneAge

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# 1. Load GenAge_Human into a gene-set dict
# ---------------------------------------------------------------------------
genage_path = "/content/GenAge_Human/genage_human.csv"

# Read (handles comma or tab automatically)
ga = pd.read_csv(genage_path, sep=None, engine="python", encoding="utf-8-sig")
print("GenAge columns:", list(ga.columns))

# Find the gene-symbol column (GenAge usually calls it 'symbol')
sym_col = None
for cand in ["symbol", "gene_symbol", "gene name", "name", "Gene", "gene"]:
    for c in ga.columns:
        if c.strip().lower() == cand.lower():
            sym_col = c
            break
    if sym_col:
        break

if sym_col is None:
    raise ValueError(f"Could not find a symbol column. Columns are: {list(ga.columns)}")

genage_genes = (
    ga[sym_col]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)
genage_genes = [g for g in dict.fromkeys(genage_genes) if g and g not in ("NAN", "NA", "NONE")]

GenAge_dict = {"GENAGE_HUMAN": genage_genes}
print(f"\nGenAge_Human: {len(genage_genes)} unique genes — first 10: {genage_genes[:10]}")

GenAge columns: ['GenAge ID', 'symbol', 'name', 'entrez gene id', 'uniprot', 'why']

GenAge_Human: 307 unique genes — first 10: ['GHR', 'GHRH', 'SHC1', 'POU1F1', 'PROP1', 'TP53', 'TERC', 'TERT', 'ATM', 'PLAU']


# Splitting the Genesets

In [ ]:
# ---------------------------------------------------------------------------
# 2. Split gene_sets into 4 collections, then FLATTEN each into one gene list
# ---------------------------------------------------------------------------
keys = list(gene_sets.keys())          # preserves insertion order
n = len(keys)
print(f"Total gene sets: {n}")

# three near-equal contiguous blocks
splits = np.array_split(keys, 3)       # -> 3 arrays of keys
keys_A, keys_B, keys_C = [list(s) for s in splits]


def _flatten(keys_subset):
    """Union all member genes from the given sub-set keys into one deduped list."""
    seen, flat = set(), []
    for k in keys_subset:
        for g in gene_sets[k]:
            g = str(g).upper()
            if g not in seen:
                seen.add(g)
                flat.append(g)
    return flat


# Each collection is now a SINGLE flat list of genes (no sub-sets)
gene_sets_all = _flatten(keys)         # i)   everything
gene_sets_A   = _flatten(keys_A)       # ii)
gene_sets_B   = _flatten(keys_B)       # iii)
gene_sets_C   = _flatten(keys_C)       # iv)

for name, flat in [("ALL", gene_sets_all), ("A", gene_sets_A),
                   ("B", gene_sets_B), ("C", gene_sets_C)]:
    print(f"[{name}] {len(flat)} genes (flattened)")

Total gene sets: 16
[ALL] 1649 genes (flattened)
[A] 760 genes (flattened)
[B] 579 genes (flattened)
[C] 572 genes (flattened)


# Head-to-head

In [9]:
# ============================================================================
#  Generic head-to-head TWAS enrichment:
#  group1 {dict of flat gene sets}  vs  group2 {dict of flat gene sets}
#  -> every (set1 x set2) pair compared exhaustively, per trait.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import hypergeom
import statsmodels.api as sm
from datetime import datetime


# ---------------------------------------------------------------------------
# Small helpers  (UNCHANGED)
# ---------------------------------------------------------------------------
def _bh_fdr(pvals):
    p = np.asarray(pvals, float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    adj = np.empty(n)
    cmin = 1.0
    for i in range(n - 1, -1, -1):
        cmin = min(cmin, p[order[i]] * n / (i + 1))
        adj[order[i]] = min(cmin, 1.0)
    return adj


def _es_from_mask(abs_corr_sorted, mask, N, k):
    """Weighted GSEA running-sum enrichment score for a boolean membership mask."""
    if k == 0 or k >= N:
        return 0.0, None
    hit = abs_corr_sorted * mask
    norm = hit.sum()
    if norm == 0:
        return 0.0, None
    P_hit = np.cumsum(hit) / norm
    P_miss = np.cumsum(~mask) / (N - k)
    dev = P_hit - P_miss
    i = int(np.argmax(np.abs(dev)))
    return float(dev[i]), i


def _null_es(abs_corr_sorted, k, n_perm, rng):
    """Competitive (gene-set permutation) null: random sets of identical size k."""
    N = len(abs_corr_sorted)
    null = np.empty(n_perm)
    for j in range(n_perm):
        pos = rng.choice(N, size=k, replace=False)
        mask = np.zeros(N, dtype=bool)
        mask[pos] = True
        hit = abs_corr_sorted * mask
        norm = hit.sum()
        if norm == 0:
            null[j] = 0.0
            continue
        P_hit = np.cumsum(hit) / norm
        P_miss = np.cumsum(~mask) / (N - k)
        dev = P_hit - P_miss
        null[j] = dev[int(np.argmax(np.abs(dev)))]
    return null


def _nes_and_p(obs, null):
    pos = null[null > 0]
    neg = null[null < 0]
    if obs >= 0:
        m = pos.mean() if len(pos) else np.nan
        nes = obs / m if (m and m > 0) else np.nan
        p = (np.sum(null >= obs) + 1) / (len(null) + 1)
    else:
        m = abs(neg.mean()) if len(neg) else np.nan
        nes = obs / m if (m and m > 0) else np.nan
        p = (np.sum(null <= obs) + 1) / (len(null) + 1)
    return float(nes) if nes == nes else np.nan, float(p)


# ---------------------------------------------------------------------------
# 1. Build a ranked TWAS list from an S-PrediXcan folder  (UNCHANGED)
# ---------------------------------------------------------------------------
def build_ranked_list(folder, ranking="abs"):
    meta_z, _, _ = load_spredixcan_folder(folder)
    meta_z = {str(g).upper(): float(z) for g, z in meta_z.items()
              if np.isfinite(z)}
    genes = list(meta_z.keys())
    z = np.array([meta_z[g] for g in genes])
    stat = np.abs(z) if ranking == "abs" else z
    order = np.argsort(-stat)                      # descending
    genes_sorted = [genes[i] for i in order]
    corr_sorted = stat[order].astype(float)
    abs_corr = np.abs(corr_sorted)
    pos_of = {g: i for i, g in enumerate(genes_sorted)}
    return dict(genes_sorted=genes_sorted, abs_corr=abs_corr,
                pos_of=pos_of, meta_z=meta_z, N=len(genes_sorted))


# ---------------------------------------------------------------------------
# 2. Single-set GSEA against a prepared ranked list  (UNCHANGED)
# ---------------------------------------------------------------------------
def gsea_one(ranklist, gene_set, n_perm, rng, null_cache, weight=1.0):
    N = ranklist["N"]
    abs_corr = ranklist["abs_corr"] ** weight
    pos_of = ranklist["pos_of"]

    present = [g for g in {str(x).upper() for x in gene_set} if g in pos_of]
    k = len(present)
    out = dict(size_input=len(set(map(str.upper, gene_set))),
               size_found=k, ES=np.nan, NES=np.nan, pval=np.nan,
               leading_edge=[])
    if k < 3:
        return out

    mask = np.zeros(N, dtype=bool)
    positions = np.array([pos_of[g] for g in present])
    mask[positions] = True
    es, peak = _es_from_mask(abs_corr, mask, N, k)

    # leading-edge genes (members up to the peak, for positive ES)
    if peak is not None and es >= 0:
        le_pos = positions[positions <= peak]
        le = [ranklist["genes_sorted"][p] for p in sorted(le_pos)]
    else:
        le = []

    if k not in null_cache:
        null_cache[k] = _null_es(abs_corr, k, n_perm, rng)
    nes, p = _nes_and_p(es, null_cache[k])

    out.update(ES=round(es, 4), NES=round(nes, 4) if nes == nes else np.nan,
               pval=round(p, 6), leading_edge=le)
    return out


# ---------------------------------------------------------------------------
# 3. ORA (hypergeometric) for one set  (UNCHANGED)
# ---------------------------------------------------------------------------
def ora_one(ranklist, gene_set, sig_genes_set):
    pos_of = ranklist["pos_of"]
    M = ranklist["N"]                              # background size
    n_sig = len(sig_genes_set)
    present = {g for g in (str(x).upper() for x in gene_set) if g in pos_of}
    K = len(present)
    overlap = present & sig_genes_set
    x = len(overlap)
    if K == 0 or n_sig == 0:
        return dict(K=K, overlap=x, ORA_p=np.nan, overlap_genes=[])
    p = hypergeom.sf(x - 1, M, n_sig, K)
    return dict(K=K, overlap=x, ORA_p=round(float(p), 6),
                overlap_genes=sorted(overlap))


# ---------------------------------------------------------------------------
# 4. Master EXHAUSTIVE head-to-head pipeline  (REWRITTEN to take TWO groups)
#    group1, group2 : dict {set_name: [flat gene list]}
#    Every set in group1 is compared against every set in group2.
# ---------------------------------------------------------------------------
def genage_vs_genesets_pipeline(
    trait_dirs,
    trait_labels,
    group1,                        # dict {set_name: [genes]}  ("test" side)
    group2,                        # dict {set_name: [genes]}  ("comparator" side)
    group1_label="G1",
    group2_label="G2",
    out="/content/094E_genage_compare",
    ranking="abs",                 # "abs" (|Z|) or "signed" (Z)
    n_perm=2000,
    weight=1.0,
    random_seed=42,
    sig_z=3.0,                     # |Z| cutoff for ORA "significant" genes
    make_plots=True,
):
    os.makedirs(out, exist_ok=True)

    def _safe(s):                  # filesystem-safe set name
        return "".join(c if (c.isalnum() or c in "-_") else "_" for c in str(s))

    # normalise input dicts -> {name: [UPPER genes]}
    group1 = {str(k): [str(g).upper() for g in v] for k, v in group1.items()}
    group2 = {str(k): [str(g).upper() for g in v] for k, v in group2.items()}

    master_rows = []

    for trait, folder in zip(trait_labels, trait_dirs):
        print(f"\n{'='*72}\n  TRAIT: {trait}   (ranking={ranking})\n{'='*72}")
        rl = build_ranked_list(folder, ranking=ranking)
        print(f"  ranked genes: {rl['N']:,}")

        # significant genes for ORA
        sig = {g for g, z in rl["meta_z"].items() if abs(z) > sig_z}
        print(f"  |Z|>{sig_z} significant genes: {len(sig)}")

        rng = np.random.default_rng(random_seed)
        null_cache = {}

        trait_dir = os.path.join(out, trait)
        os.makedirs(trait_dir, exist_ok=True)

        # ---- evaluate every set ONCE per group (GSEA + ORA), reuse null_cache ----
        def _eval_group(grp, lbl):
            res = {}
            for name, genes in grp.items():
                g = gsea_one(rl, genes, n_perm, rng, null_cache, weight)
                o = ora_one(rl, genes, sig)
                res[name] = dict(genes=genes, gsea=g, ora=o)
                print(f"  [{lbl}] {name}: found {g['size_found']}/{g['size_input']}  "
                      f"NES={g['NES']}  p={g['pval']}")
            return res

        print(f"\n  --- evaluating group1 ({group1_label}) ---")
        res1 = _eval_group(group1, group1_label)
        print(f"\n  --- evaluating group2 ({group2_label}) ---")
        res2 = _eval_group(group2, group2_label)

        # ---- per-set BH-FDR (computed on the unique sets, then mapped to pairs) ----
        def _set_fdr(res):
            names = list(res.keys())
            p = np.array([res[n]["gsea"]["pval"] for n in names], float)
            p = np.where(np.isfinite(p), p, 1.0)
            return dict(zip(names, _bh_fdr(p)))
        fdr1 = _set_fdr(res1)
        fdr2 = _set_fdr(res2)

        # ---- EXHAUSTIVE pairwise comparison (set1 x set2) ----
        rows = []
        for n1, r1 in res1.items():
            g1, o1, genes1 = r1["gsea"], r1["ora"], r1["genes"]
            nes1 = g1["NES"] if g1["NES"] == g1["NES"] else -np.inf
            for n2, r2 in res2.items():
                g2, o2, genes2 = r2["gsea"], r2["ora"], r2["genes"]
                nes2 = g2["NES"] if g2["NES"] == g2["NES"] else -np.inf

                # P(random set of size k1 reaches set2's ES)  -- same idea as before
                k1 = g1["size_found"]
                p_reach = np.nan
                if k1 in null_cache and g2["ES"] == g2["ES"]:
                    p_reach = round(float(np.mean(null_cache[k1] >= g2["ES"])), 4)

                verdict = ("set1_beats_set2"
                           if (nes1 > nes2 and g1["pval"] < 0.05)
                           else "not_better")

                rows.append(dict(
                    Trait=trait,
                    Group1=group1_label, Set1=n1,
                    Group2=group2_label, Set2=n2,
                    Size1_found=g1["size_found"], Size1_input=g1["size_input"],
                    NES1=g1["NES"], GSEA_p1=g1["pval"], GSEA_FDR1=round(fdr1[n1], 6),
                    ORA_overlap1=o1["overlap"], ORA_p1=o1["ORA_p"],
                    Size2_found=g2["size_found"], Size2_input=g2["size_input"],
                    NES2=g2["NES"], GSEA_p2=g2["pval"], GSEA_FDR2=round(fdr2[n2], 6),
                    ORA_overlap2=o2["overlap"], ORA_p2=o2["ORA_p"],
                    dNES_1_vs_2=(round(nes1 - nes2, 4)
                                 if np.isfinite(nes1) and np.isfinite(nes2) else np.nan),
                    p_set2ES_reached_by_randomK1=p_reach,
                    Verdict=verdict,
                    LeadingEdge1=";".join(g1["leading_edge"][:25]),
                    LeadingEdge2=";".join(g2["leading_edge"][:25]),
                ))
                print(f"      {n1} vs {n2}: NES1={g1['NES']} NES2={g2['NES']} "
                      f"dNES={rows[-1]['dNES_1_vs_2']}  -> {verdict}")

                # ---- linear model: |Z| ~ in_set1 + in_set2 ----
                try:
                    genes_all = rl["genes_sorted"]
                    yabs = np.array([abs(rl["meta_z"][gg]) for gg in genes_all])
                    s1 = set(genes1)
                    s2 = set(genes2)
                    in1 = np.array([gg in s1 for gg in genes_all], float)
                    in2 = np.array([gg in s2 for gg in genes_all], float)
                    X = sm.add_constant(np.column_stack([in1, in2]))
                    lm = sm.OLS(yabs, X).fit()
                    fn = f"linearmodel_{trait}_{_safe(n1)}_vs_{_safe(n2)}.txt"
                    with open(os.path.join(trait_dir, fn), "w") as fh:
                        fh.write(f"Outcome: |meta-Z|   "
                                 f"Predictors: in_{n1} ({group1_label}), "
                                 f"in_{n2} ({group2_label})\n\n")
                        fh.write(str(lm.summary()))
                except Exception as e:
                    print(f"      [warn] linear model failed ({n1} vs {n2}): {e}")

        trait_df = pd.DataFrame(rows)
        # BH-FDR across all pairwise dNES tests (pairwise-level FDR on set1 p-values)
        pp = np.where(np.isfinite(trait_df["GSEA_p1"].values),
                      trait_df["GSEA_p1"].values, 1.0)
        trait_df["pair_GSEA_FDR"] = _bh_fdr(pp)
        trait_df = trait_df.sort_values("dNES_1_vs_2", ascending=False)
        trait_df.to_csv(os.path.join(trait_dir, f"headtohead_{trait}.csv"),
                        index=False)
        master_rows.append(trait_df)

        # ---- plot: dNES heatmap (set1 rows x set2 cols) ----
        if make_plots:
            try:
                piv = trait_df.pivot(index="Set1", columns="Set2",
                                     values="dNES_1_vs_2")
                piv = piv.reindex(index=list(group1.keys()),
                                  columns=list(group2.keys()))
                fig, ax = plt.subplots(
                    figsize=(max(4, 1.1 * piv.shape[1] + 2),
                             max(3, 0.6 * piv.shape[0] + 1)))
                vmax = np.nanmax(np.abs(piv.values)) if np.isfinite(
                    np.nanmax(np.abs(piv.values))) else 1.0
                im = ax.imshow(piv.values, cmap="RdBu_r",
                               vmin=-vmax, vmax=vmax, aspect="auto")
                ax.set_xticks(range(piv.shape[1]))
                ax.set_xticklabels(piv.columns, rotation=45, ha="right", fontsize=8)
                ax.set_yticks(range(piv.shape[0]))
                ax.set_yticklabels(piv.index, fontsize=8)
                for i in range(piv.shape[0]):
                    for j in range(piv.shape[1]):
                        v = piv.values[i, j]
                        if np.isfinite(v):
                            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                                    fontsize=7,
                                    color="white" if abs(v) > vmax * 0.6 else "black")
                ax.set_xlabel(f"{group2_label} (set2)")
                ax.set_ylabel(f"{group1_label} (set1)")
                ax.set_title(f"{trait} — dNES (NES1 - NES2)")
                fig.colorbar(im, ax=ax, shrink=0.8, label="dNES")
                plt.tight_layout()
                fig.savefig(os.path.join(trait_dir, f"dNES_heatmap_{trait}.png"),
                            dpi=150)
                plt.close(fig)
            except Exception as e:
                print(f"    [warn] plot failed: {e}")

    master = pd.concat(master_rows, ignore_index=True)
    master.to_csv(os.path.join(out, "MASTER_headtohead.csv"), index=False)

    # ---- compact summary txt ----
    lines = ["="*78,
             f"  {group1_label}  vs  {group2_label}  — EXHAUSTIVE PAIRWISE SUMMARY",
             f"  Generated: {datetime.now():%Y-%m-%d %H:%M:%S}",
             f"  Ranking={ranking} | n_perm={n_perm} | sig_z={sig_z} | weight={weight}",
             f"  group1 sets: {list(group1.keys())}",
             f"  group2 sets: {list(group2.keys())}",
             "="*78, ""]
    for trait in trait_labels:
        lines.append(f"[{trait}]")
        sub = master[master.Trait == trait]
        for _, r in sub.iterrows():
            lines.append(
                f"  {r['Set1']} vs {r['Set2']}: "
                f"NES1={r['NES1']} (p={r['GSEA_p1']}) | "
                f"NES2={r['NES2']} (p={r['GSEA_p2']}) | "
                f"dNES={r['dNES_1_vs_2']} -> {r['Verdict']}")
        lines.append("")
    summ = "\n".join(lines)
    with open(os.path.join(out, "SUMMARY.txt"), "w") as fh:
        fh.write(summ)
    print("\n" + summ)
    print(f"\nDONE → {os.path.abspath(out)}")
    return master

### Running

In [ ]:
!ls /content/

093FvB_genage_compare  drive				       ppd
AD_related.gmt	       drug_interaction_results-6_15_2026.tsv  ptsd
ageing_polycomb        GenAge_Human			       sample_data
alz		       gene_sets.py			       scz
an		       hoarding				       tics
anx_cc		       mdd
bip		       ocd


In [10]:
# group1 = the four flat parts; group2 = the GenAge comparator, a 1-set dict
group1 = gene_sets
group2 = {
    "AntiPsy_Genes": flat_unique_dict,
}

master = genage_vs_genesets_pipeline(
    trait_dirs=[
        "/content/bip_afr",
        "/content/bip_eas",
        "/content/bip_eur",
    ],
    trait_labels=[
        "bip_afr",
        "bip_eas",
        "bip_eur",
    ],
    group1=group1,
    group2=group2,
    group1_label="ageing_polycomb",
    group2_label="AntiPsy",
    out="/content/096AvAp3_compare",
    ranking="abs",      # |Z| ranking
    n_perm=2000,        # raise to 10000 for final run
    sig_z=3.0,
)


  TRAIT: bip_afr   (ranking=abs)
    6 file(s) in bip_afr/
    ⚠ Brain_Amygdala_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Anterior_cingulate_cortex_BA24_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Caudate_basal_ganglia_spredixcan.csv: 2 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Frontal_Cortex_BA9_spredixcan.csv: 2 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Hippocampus_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Nucleus_accumbens_basal_ganglia_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    → meta-Z computed for 10,703 genes
    → Ensembl aliases mapped: 21,412
    → 6 tissue file(s) indexed
  ranked genes: 10,703
  |Z|>3.0 significant genes: 673

  --- evaluating group1 (ageing_polycomb) ---
  [ageing_polycomb] HSA04142_LYSOSOME: found 151/250  NES=1.1447  p=0.075962
  [ageing_polycomb] HSA04216_FERROPTOSIS: 

# Post-hoc

In [11]:
# ============================================================================
#  GENERIC POST-HOC PIPELINE  —  group1 {dict} vs group2 {dict}, exhaustive
#  Requires (from earlier cells): build_ranked_list, gsea_one, ora_one,
#                                  _bh_fdr, _nes_and_p
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr, pearsonr, binomtest
from datetime import datetime


def _U(genes):
    """Uppercase + dedupe preserving order."""
    seen, out = set(), []
    for g in genes:
        g = str(g).upper()
        if g not in seen:
            seen.add(g); out.append(g)
    return out


# ---------------------------------------------------------------------------
# 0. Recompute leading edge per set on a ranked list (so it's exact + signed)
#    (UNCHANGED)
# ---------------------------------------------------------------------------
def _leading_edge(rl, gene_set, weight=1.0):
    """Return leading-edge gene list (members up to running-sum peak) for a set."""
    N = rl["N"]
    abs_corr = rl["abs_corr"] ** weight
    pos_of = rl["pos_of"]
    present = [g for g in _U(gene_set) if g in pos_of]
    k = len(present)
    if k < 3:
        return []
    positions = np.array([pos_of[g] for g in present])
    mask = np.zeros(N, dtype=bool); mask[positions] = True
    hit = abs_corr * mask
    norm = hit.sum()
    if norm == 0:
        return []
    P_hit = np.cumsum(hit) / norm
    P_miss = np.cumsum(~mask) / (N - k)
    dev = P_hit - P_miss
    peak = int(np.argmax(np.abs(dev)))
    if dev[peak] >= 0:
        le_pos = positions[positions <= peak]
    else:                                   # negative ES: tail side
        le_pos = positions[positions >= peak]
    return [rl["genes_sorted"][p] for p in sorted(le_pos)]


# ---------------------------------------------------------------------------
# 1. MASTER post-hoc driver  (REWRITTEN to take TWO groups, exhaustive)
#    group1, group2 : dict {set_name: [flat gene list]}
# ---------------------------------------------------------------------------
def posthoc_pipeline(
    trait_dirs,
    trait_labels,
    group1,                         # dict {set_name: [genes]}  ("test" side)
    group2,                         # dict {set_name: [genes]}  ("comparator" side)
    group1_label="G1",
    group2_label="G2",
    out="/content/094F_posthoc",
    ranking="abs",
    weight=1.0,
    n_perm=2000,
    random_seed=42,
    direction_z="signed",           # use signed meta-Z for directionality
    alpha=0.05,                     # significance threshold for flagging in summary
    top_n=20,                       # how many top genes to list per detailed block
):
    os.makedirs(out, exist_ok=True)

    # normalise input dicts -> {name: [UPPER deduped genes]}
    group1 = {str(k): _U(v) for k, v in group1.items()}
    group2 = {str(k): _U(v) for k, v in group2.items()}
    names1 = list(group1.keys())
    names2 = list(group2.keys())

    overlap_rows, dir_rows, contrib_rows = [], [], []
    clean_rows, loo_rows, dist_rows = [], [], []

    for trait, folder in zip(trait_labels, trait_dirs):
        print(f"\n{'='*72}\n  POST-HOC | TRAIT: {trait}\n{'='*72}")
        rl = build_ranked_list(folder, ranking=ranking)
        meta_z = rl["meta_z"]                     # signed Z always available
        pos_of = rl["pos_of"]
        rng = np.random.default_rng(random_seed)
        null_cache = {}
        tdir = os.path.join(out, trait); os.makedirs(tdir, exist_ok=True)

        # ---- precompute leading edges (recomputed, exact) for ALL sets ----
        le1 = {n: _leading_edge(rl, group1[n], weight) for n in names1}
        le2 = {n: _leading_edge(rl, group2[n], weight) for n in names2}

        # =====================================================
        # TEST 1 — Leading-edge overlap (EVERY group1 set vs EVERY group2 set)
        # =====================================================
        for a in names1:
            A = set(le1[a])
            for b in names2:
                B = set(le2[b])
                inter = A & B
                jacc = len(inter) / len(A | B) if (A | B) else np.nan
                overlap_rows.append(dict(
                    Trait=trait,
                    Group1=group1_label, Set1=a,
                    Group2=group2_label, Set2=b,
                    n_1=len(A), n_2=len(B), n_overlap=len(inter),
                    jaccard=round(jacc, 4) if jacc == jacc else np.nan,
                    overlap_genes=";".join(sorted(inter)[:30]),
                ))
        print("  [1] leading-edge overlap (exhaustive cross) done")

        # =====================================================
        # TEST 2 — Directionality of leading-edge genes (every set, both groups)
        # =====================================================
        for grp_label, le_dict in [(group1_label, le1), (group2_label, le2)]:
            for setname, members in le_dict.items():
                zs = np.array([meta_z[g] for g in members if g in meta_z])
                if len(zs) == 0:
                    continue
                n_pos = int((zs > 0).sum()); n_neg = int((zs < 0).sum())
                n = n_pos + n_neg
                if n > 0:
                    k = max(n_pos, n_neg)
                    sign_p = binomtest(k, n, 0.5, alternative="two-sided").pvalue
                else:
                    sign_p = np.nan
                dir_rows.append(dict(
                    Trait=trait, Group=grp_label, Set=setname, n_LE=len(zs),
                    n_pos=n_pos, n_neg=n_neg,
                    pct_neg=round(100 * n_neg / max(n, 1), 1),
                    mean_Z=round(float(zs.mean()), 4),
                    mean_signedZ=round(float(zs.mean()), 4),
                    sign_test_p=round(float(sign_p), 5) if sign_p == sign_p else np.nan,
                ))
        print("  [2] directionality done")

        # =====================================================
        # TEST 3 — Top gene contributions (every set, both groups)
        # =====================================================
        for grp_label, le_dict in [(group1_label, le1), (group2_label, le2)]:
            for setname, members in le_dict.items():
                for g in members:
                    if g in meta_z:
                        contrib_rows.append(dict(
                            Trait=trait, Group=grp_label, Set=setname, gene=g,
                            Z=round(meta_z[g], 4), absZ=round(abs(meta_z[g]), 4),
                            rank=pos_of[g] + 1,
                        ))
        print("  [3] gene-contribution ranking done")

        # =====================================================
        # TEST 4 — Remove genes overlapping a group2 set, re-test each group1 set
        #          (EXHAUSTIVE: every group1 set x every group2 set)
        # =====================================================
        # comparator baselines (one GSEA per group2 set)
        comp_nes = {}
        for b in names2:
            gb = gsea_one(rl, group2[b], n_perm, rng, null_cache, weight)
            comp_nes[b] = gb["NES"]
        for a in names1:
            full = group1[a]
            g_full = gsea_one(rl, full, n_perm, rng, null_cache, weight)
            for b in names2:
                rm_set = set(group2[b])
                clean = [g for g in full if g not in rm_set]
                g_clean = gsea_one(rl, clean, n_perm, rng, null_cache, weight)
                clean_rows.append(dict(
                    Trait=trait,
                    Group1=group1_label, Set1=a,
                    Group2=group2_label, Set2=b,
                    n_full=len(full), n_removed=len(full) - len(clean),
                    NES_full=g_full["NES"], p_full=g_full["pval"],
                    NES_clean=g_clean["NES"], p_clean=g_clean["pval"],
                    dNES_after_removal=(round(g_clean["NES"] - g_full["NES"], 4)
                                        if (g_clean["NES"] == g_clean["NES"]
                                            and g_full["NES"] == g_full["NES"]) else np.nan),
                    NES_comparator=comp_nes[b],
                ))
        print("  [4] remove-overlap sensitivity (exhaustive) done")

        # =====================================================
        # TEST 5 — Leave-one-out on each group2 set (which genes drive its NES?)
        # =====================================================
        for b in names2:
            base = gsea_one(rl, group2[b], n_perm, rng, null_cache, weight)
            base_nes = base["NES"] if base["NES"] == base["NES"] else np.nan
            for g in le2[b]:
                reduced = [x for x in group2[b] if x != g]
                gr = gsea_one(rl, reduced, n_perm, rng, null_cache, weight)
                dn = (gr["NES"] - base_nes) if (gr["NES"] == gr["NES"]
                                                and base_nes == base_nes) else np.nan
                loo_rows.append(dict(
                    Trait=trait, Group2=group2_label, Set2=b, removed_gene=g,
                    Z=round(meta_z.get(g, np.nan), 4),
                    NES_base=round(base_nes, 4) if base_nes == base_nes else np.nan,
                    NES_without=gr["NES"],
                    delta_NES=round(dn, 4) if dn == dn else np.nan,
                ))
        print("  [5] leave-one-out on group2 sets done")

        # =====================================================
        # TEST 6 — |Z|-distribution comparison (every group1 set vs every group2 set)
        # =====================================================
        comp_abs = {b: np.array([abs(meta_z[g]) for g in group2[b] if g in meta_z])
                    for b in names2}
        for a in names1:
            part_abs = np.array([abs(meta_z[g]) for g in group1[a] if g in meta_z])
            for b in names2:
                cab = comp_abs[b]
                if len(part_abs) and len(cab):
                    U, p_mw = mannwhitneyu(part_abs, cab, alternative="two-sided")
                else:
                    p_mw = np.nan
                dist_rows.append(dict(
                    Trait=trait,
                    Group1=group1_label, Set1=a,
                    Group2=group2_label, Set2=b,
                    median_absZ_set1=round(float(np.median(part_abs)), 4) if len(part_abs) else np.nan,
                    median_absZ_set2=round(float(np.median(cab)), 4) if len(cab) else np.nan,
                    MWU_p=round(float(p_mw), 5) if p_mw == p_mw else np.nan,
                ))
        print("  [6] Z-distribution comparison done")

        # ---- per-trait violin of |Z| (all sets, both groups) ----
        try:
            data, names = [], []
            for b in names2:
                data.append([abs(meta_z[g]) for g in group2[b] if g in meta_z])
                names.append(f"{group2_label}:{b}")
            for a in names1:
                data.append([abs(meta_z[g]) for g in group1[a] if g in meta_z])
                names.append(f"{group1_label}:{a}")
            fig, ax = plt.subplots(figsize=(max(7, 0.9 * len(names) + 2), 4))
            ax.violinplot(data, showmedians=True)
            ax.set_xticks(range(1, len(names) + 1))
            ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
            ax.set_ylabel("|meta-Z|"); ax.set_title(f"{trait} — |Z| distribution by set")
            plt.tight_layout(); fig.savefig(os.path.join(tdir, f"absZ_violin_{trait}.png"), dpi=150)
            plt.close(fig)
        except Exception as e:
            print(f"    [warn] violin failed: {e}")

    # ---------------------------------------------------------------------
    # Save all tables
    # ---------------------------------------------------------------------
    tables = {
        "T1_leading_edge_overlap": pd.DataFrame(overlap_rows),
        "T2_directionality":        pd.DataFrame(dir_rows),
        "T3_gene_contributions":    pd.DataFrame(contrib_rows)
                                       .sort_values(["Trait", "Group", "Set", "absZ"],
                                                    ascending=[True, True, True, False]),
        "T4_remove_overlap_retest": pd.DataFrame(clean_rows),
        "T5_LOO_group2":            pd.DataFrame(loo_rows)
                                       .sort_values(["Trait", "Set2", "delta_NES"]),
        "T6_Zdist_comparison":      pd.DataFrame(dist_rows),
    }
    for name, df in tables.items():
        df.to_csv(os.path.join(out, f"{name}.csv"), index=False)

    # =====================================================================
    # DETAILED POST-HOC SUMMARY  —  all important / significant results
    # =====================================================================
    def _sig(p):
        if p != p or p is None:
            return ""
        if p < 0.001:  return " ***"
        if p < 0.01:   return " **"
        if p < alpha:  return " *"
        return ""

    T1 = tables["T1_leading_edge_overlap"]
    T2 = tables["T2_directionality"]
    T3 = tables["T3_gene_contributions"]
    T4 = tables["T4_remove_overlap_retest"]
    T5 = tables["T5_LOO_group2"]
    T6 = tables["T6_Zdist_comparison"]

    L = []
    add = L.append
    add("#" * 80)
    add("#  POST-HOC DETAILED SUMMARY  (exhaustive group1 x group2)")
    add(f"#  Generated : {datetime.now():%Y-%m-%d %H:%M:%S}")
    add(f"#  Params    : ranking={ranking} | weight={weight} | n_perm={n_perm} | "
        f"alpha={alpha}")
    add(f"#  group1 ({group1_label}) : {names1}")
    add(f"#  group2 ({group2_label}) : {names2}")
    add(f"#  Traits    : {', '.join(trait_labels)}")
    add(f"#  Sig codes : *** p<0.001   ** p<0.01   * p<{alpha}")
    add("#" * 80)
    add("")

    for trait in trait_labels:
        add("=" * 80)
        add(f"  TRAIT: {trait}")
        add("=" * 80)

        # ---------------- TEST 2 first: headline directionality ----------
        add("")
        add("-" * 80)
        add("  [TEST 2]  DIRECTIONALITY OF LEADING-EDGE GENES  (signed meta-Z)")
        add("  Hypothesis: repressed (negative-Z) genes enriched => supports Polycomb model")
        add("-" * 80)
        d = T2[T2.Trait == trait]
        add(f"  {'Group':<8}{'Set':<12}{'n_LE':>6}{'n_pos':>7}{'n_neg':>7}{'%neg':>7}"
            f"{'meanZ':>9}{'sign_p':>11}")
        for _, r in d.iterrows():
            add(f"  {r['Group']:<8}{str(r['Set']):<12}{int(r['n_LE']):>6}{int(r['n_pos']):>7}"
                f"{int(r['n_neg']):>7}{r['pct_neg']:>7}{r['mean_Z']:>9}"
                f"{r['sign_test_p']:>11}{_sig(r['sign_test_p'])}")
        sig_dir = d[d.sign_test_p < alpha]
        if not sig_dir.empty:
            add("")
            add("  >> SIGNIFICANT directional skew:")
            for _, r in sig_dir.iterrows():
                skew = "NEGATIVE/repressed" if r['mean_Z'] < 0 else "POSITIVE"
                add(f"     {r['Group']}:{r['Set']}: {r['pct_neg']}% negative-Z, "
                    f"mean Z={r['mean_Z']} ({skew}), "
                    f"sign-test p={r['sign_test_p']}{_sig(r['sign_test_p'])}")
        else:
            add("")
            add("  >> No set shows a significant +/- sign imbalance in its leading edge.")

        # ---------------- TEST 1: leading-edge overlap (exhaustive) ------
        add("")
        add("-" * 80)
        add(f"  [TEST 1]  LEADING-EDGE OVERLAP  ({group1_label} set vs {group2_label} set)")
        add("  (is the enrichment signal shared or distinct?)")
        add("-" * 80)
        ov = T1[T1.Trait == trait]
        add(f"  {'Set1':<12}{'Set2':<12}{'n1_LE':>7}{'n2_LE':>7}"
            f"{'overlap':>9}{'jaccard':>9}")
        for _, r in ov.iterrows():
            add(f"  {str(r['Set1']):<12}{str(r['Set2']):<12}{int(r['n_1']):>7}"
                f"{int(r['n_2']):>7}{int(r['n_overlap']):>9}{r['jaccard']:>9}")
        add("")
        add("  Shared leading-edge genes per pair (Set1 ∩ Set2):")
        for _, r in ov.iterrows():
            genes = r["overlap_genes"]
            add(f"     {r['Set1']} ∩ {r['Set2']}: {genes if genes else '(none)'}")

        # ---------------- TEST 4: remove-overlap robustness --------------
        add("")
        add("-" * 80)
        add(f"  [TEST 4]  ROBUSTNESS: re-test each {group1_label} set AFTER removing "
            f"a {group2_label} set's genes")
        add("  Q: does the set survive independently of the comparator's genes?")
        add("-" * 80)
        c = T4[T4.Trait == trait]
        add(f"  {'Set1':<10}{'Set2':<10}{'n_full':>7}{'n_rm':>6}{'NES_full':>10}"
            f"{'p_full':>10}{'NES_clean':>11}{'p_clean':>10}{'dNES':>9}{'NES_cmp':>9}")
        for _, r in c.iterrows():
            add(f"  {str(r['Set1']):<10}{str(r['Set2']):<10}{int(r['n_full']):>7}"
                f"{int(r['n_removed']):>6}{r['NES_full']:>10}{r['p_full']:>10}"
                f"{r['NES_clean']:>11}{r['p_clean']:>10}{r['dNES_after_removal']:>9}"
                f"{r['NES_comparator']:>9}{_sig(r['p_clean'])}")
        add("")
        for _, r in c.iterrows():
            verdict = []
            if r['p_clean'] == r['p_clean'] and r['p_clean'] < alpha:
                verdict.append("still SIGNIFICANT after removal")
            if (r['NES_clean'] == r['NES_clean'] and r['NES_comparator'] == r['NES_comparator']
                    and r['NES_clean'] > r['NES_comparator']):
                verdict.append("clean NES > comparator")
            dn = r['dNES_after_removal']
            if dn == dn and dn > 0:
                verdict.append("strengthened by removal")
            elif dn == dn and dn < 0:
                verdict.append("weakened by removal")
            if verdict:
                add(f"     {r['Set1']} (rm {r['Set2']}): " + "; ".join(verdict) + ".")

        # ---------------- TEST 5: LOO drivers of each comparator ---------
        add("")
        add("-" * 80)
        add(f"  [TEST 5]  LEAVE-ONE-OUT on {group2_label} sets — top {top_n} NES drivers")
        add("  (most negative delta_NES = removing the gene hurts the set most)")
        add("-" * 80)
        for b in names2:
            l5 = T5[(T5.Trait == trait) & (T5.Set2 == b)].sort_values("delta_NES").head(top_n)
            if l5.empty:
                continue
            add(f"  -- {group2_label}:{b} --")
            add(f"     {'removed_gene':<16}{'Z':>9}{'NES_base':>10}{'NES_without':>13}{'dNES':>9}")
            for _, r in l5.iterrows():
                add(f"     {str(r['removed_gene']):<16}{r['Z']:>9}{r['NES_base']:>10}"
                    f"{r['NES_without']:>13}{r['delta_NES']:>9}")

        # ---------------- TEST 3: top gene contributions ----------------
        add("")
        add("-" * 80)
        add(f"  [TEST 3]  TOP {top_n} LEADING-EDGE GENES by |meta-Z|  (every set)")
        add("-" * 80)
        for grp in [group2_label, group1_label]:
            sets = names2 if grp == group2_label else names1
            for setname in sets:
                sub = T3[(T3.Trait == trait) & (T3.Group == grp)
                         & (T3.Set == setname)].head(top_n)
                if sub.empty:
                    continue
                add(f"  -- {grp}:{setname} --")
                add(f"     {'gene':<14}{'Z':>9}{'|Z|':>8}{'rank':>8}")
                for _, r in sub.iterrows():
                    add(f"     {str(r['gene']):<14}{r['Z']:>9}{r['absZ']:>8}{int(r['rank']):>8}")

        # ---------------- TEST 6: Z-distribution ------------------------
        add("")
        add("-" * 80)
        add(f"  [TEST 6]  |Z| DISTRIBUTION: each {group1_label} set vs each "
            f"{group2_label} set (Mann-Whitney U)")
        add("-" * 80)
        z6 = T6[T6.Trait == trait]
        add(f"  {'Set1':<10}{'Set2':<10}{'med|Z|_1':>11}{'med|Z|_2':>11}{'MWU_p':>11}")
        for _, r in z6.iterrows():
            add(f"  {str(r['Set1']):<10}{str(r['Set2']):<10}{r['median_absZ_set1']:>11}"
                f"{r['median_absZ_set2']:>11}{r['MWU_p']:>11}{_sig(r['MWU_p'])}")
        sig_z6 = z6[z6.MWU_p < alpha]
        if not sig_z6.empty:
            add("")
            add("  >> Pairs whose |Z| distributions differ significantly:")
            for _, r in sig_z6.iterrows():
                hi = "higher" if r['median_absZ_set1'] > r['median_absZ_set2'] else "lower"
                add(f"     {r['Set1']} vs {r['Set2']}: median |Z| {hi} for {r['Set1']} "
                    f"(p={r['MWU_p']}{_sig(r['MWU_p'])})")

        # ---------------- per-trait verdict -----------------------------
        add("")
        add("-" * 80)
        add("  OVERALL VERDICT (this trait)")
        add("-" * 80)
        any_dir = T2[(T2.Trait == trait) & (T2.sign_test_p < alpha)]
        any_clean = T4[(T4.Trait == trait) & (T4.p_clean < alpha)]
        any_beat = T4[(T4.Trait == trait) & (T4.NES_clean > T4.NES_comparator)]
        dir_list = [f"{r['Group']}:{r['Set']}" for _, r in any_dir.iterrows()]
        clean_list = [f"{r['Set1']}(rm {r['Set2']})" for _, r in any_clean.iterrows()]
        beat_list = [f"{r['Set1']}(rm {r['Set2']})" for _, r in any_beat.iterrows()]
        add(f"  - Sets with significant directional skew      : "
            f"{', '.join(dir_list) if dir_list else 'none'}")
        add(f"  - {group1_label} sets significant after removal : "
            f"{', '.join(clean_list) if clean_list else 'none'}")
        add(f"  - {group1_label} sets whose clean NES > cmp     : "
            f"{', '.join(beat_list) if beat_list else 'none'}")
        add("")

    summ = "\n".join(L)
    with open(os.path.join(out, "POSTHOC_SUMMARY.txt"), "w") as fh:
        fh.write(summ)
    print("\n" + summ)
    print(f"\nDONE → {os.path.abspath(out)}")
    return tables

### Running

### post-hoc for traits with ≥1 winning pathway

In [12]:
# ============================================================
# Post-hoc pipeline loop — BIP ancestry-specific traits only
# ============================================================

winning_traits = [
    "bip_afr",
    "bip_eas",
    "bip_eur",
]

# Map trait label → actual folder path
trait_dir_map = {
    "bip_afr": "/content/bip_afr",
    "bip_eas": "/content/bip_eas",
    "bip_eur": "/content/bip_eur",
}

# Base output directory for all post-hoc runs
base_out = "/content/096AvAp3_posthoc"

for trait in winning_traits:
    trait_dir = trait_dir_map[trait]
    out_dir   = f"{base_out}_{trait}"

    print(f"\n{'='*70}")
    print(f"Running posthoc_pipeline on: {trait}")
    print(f"Input dir : {trait_dir}")
    print(f"Output dir: {out_dir}")
    print('='*70)

    tables = posthoc_pipeline(
        trait_dirs   = [trait_dir],
        trait_labels = [trait],
        group1       = group1,
        group2       = group2,
        group1_label = "polycomb",
        group2_label = "AntiPsy",
        out          = out_dir,
        ranking      = "abs",        # keep consistent with head-to-head
        n_perm       = 2000,
    )

    print(f"✓ Finished {trait} → results saved to {out_dir}\n")

print("\n✅ All post-hoc runs completed for BIP ancestry-specific traits.")


Running posthoc_pipeline on: bip_afr
Input dir : /content/bip_afr
Output dir: /content/096AvAp3_posthoc_bip_afr

  POST-HOC | TRAIT: bip_afr
    6 file(s) in bip_afr/
    ⚠ Brain_Amygdala_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Anterior_cingulate_cortex_BA24_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Caudate_basal_ganglia_spredixcan.csv: 2 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Frontal_Cortex_BA9_spredixcan.csv: 2 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Hippocampus_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Nucleus_accumbens_basal_ganglia_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    → meta-Z computed for 10,703 genes
    → Ensembl aliases mapped: 21,412
    → 6 tissue file(s) indexed
  [1] leading-edge overlap (exhaustive cross) done
  [2] directionality done
  [3] gene-contribution ranking done


# Follow-up

In [13]:
# ============================================================================
#  093FvA — FOLLOW-UP PIPELINE
#  Runs every follow-up test on the head-to-head / post-hoc outputs and
#  returns ALL results in a single dict.
#
#  Tests included
#    [0] Harvest & union leading-edge genes (from T1/T3/MASTER) + key hubs
#    [2] Functional enrichment of the union (Enrichr via gseapy; g:Profiler URL)
#    [3] Cell-type / tissue specificity for key genes (per-tissue meta-Z)
#    [4] Colocalization / fine-mapping priority list (cross-trait Z range)
#    [5] Drug-repurposing screen (DGIdb GraphQL) on the leading-edge union
#    [6] Signed vs absolute re-analysis (repression-model test)
#
#  Requires from earlier cells:
#    load_spredixcan_folder, build_ranked_list, gsea_one, ora_one,
#    genage_vs_genesets_pipeline,  and the objects group1/group2 (or rebuild).
# ============================================================================

# --- one-time installs (safe to re-run) -----------------------------------
try:
    import gseapy as gp           # noqa
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gseapy"], check=False)

import os, re, json, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import requests
from datetime import datetime
warnings.filterwarnings("ignore")


# ===========================================================================
# [0]  Harvest leading-edge genes from saved post-hoc / head-to-head tables
# ===========================================================================
def harvest_leading_edge(
    posthoc_dir,
    headtohead_csv=None,
    extra_hub_genes=None,
    top_n_per_set=30,
):
    """
    Parse leading-edge genes from:
      - <posthoc_dir>/T3_gene_contributions.csv   (Trait, Group, Set, gene, Z, absZ)
      - <posthoc_dir>/T1_leading_edge_overlap.csv (overlap_genes column)
      - MASTER_headtohead.csv                     (LeadingEdge1/LeadingEdge2)
    Returns dict with per-(group,set) leading edges + a global union.
    """
    per_set = {}          # {(Group, Set): set(genes)}
    union = set()

    # ---- T3: most reliable (one gene per row) ----
    t3p = os.path.join(posthoc_dir, "T3_gene_contributions.csv")
    if os.path.isfile(t3p):
        t3 = pd.read_csv(t3p)
        for (grp, st), sub in t3.groupby(["Group", "Set"]):
            sub = sub.sort_values("absZ", ascending=False).head(top_n_per_set)
            genes = set(sub["gene"].astype(str).str.upper())
            per_set[(grp, st)] = per_set.get((grp, st), set()) | genes
            union |= genes

    # ---- T1: overlap_genes (semicolon list) ----
    t1p = os.path.join(posthoc_dir, "T1_leading_edge_overlap.csv")
    if os.path.isfile(t1p):
        t1 = pd.read_csv(t1p)
        for col in ("overlap_genes",):
            if col in t1.columns:
                for v in t1[col].dropna():
                    union |= {g.strip().upper() for g in str(v).split(";") if g.strip()}

    # ---- MASTER head-to-head LeadingEdge1 / LeadingEdge2 ----
    if headtohead_csv and os.path.isfile(headtohead_csv):
        m = pd.read_csv(headtohead_csv)
        for col in ("LeadingEdge1", "LeadingEdge2"):
            if col in m.columns:
                for v in m[col].dropna():
                    union |= {g.strip().upper() for g in str(v).split(";") if g.strip()}

    # ---- manual hubs ----
    if extra_hub_genes:
        union |= {g.strip().upper() for g in extra_hub_genes}

    union = sorted(g for g in union if g and g not in ("NAN", "NA", "NONE"))
    print(f"[0] Harvested {len(union)} unique leading-edge genes "
          f"across {len(per_set)} (group,set) blocks")
    return {"per_set": per_set, "union": union}


# ===========================================================================
# [2]  Functional enrichment (Enrichr via gseapy) + g:Profiler URL
# ===========================================================================
def functional_enrichment(gene_list, out_dir,
                          libraries=("GO_Biological_Process_2021",
                                     "KEGG_2021_Human",
                                     "Reactome_2022",
                                     "GWAS_Catalog_2023")):
    os.makedirs(out_dir, exist_ok=True)
    # save the input list
    with open(os.path.join(out_dir, "leading_edge_union.txt"), "w") as f:
        f.write("\n".join(gene_list))

    results = {}
    try:
        import gseapy as gp
        enr = gp.enrichr(gene_list=gene_list,
                         gene_sets=list(libraries),
                         organism="human", outdir=None)
        res = enr.results.copy()
        res = res.sort_values("Adjusted P-value")
        res.to_csv(os.path.join(out_dir, "enrichr_results.csv"), index=False)
        results["enrichr"] = res
        print(f"[2] Enrichr: {len(res)} terms; "
              f"{(res['Adjusted P-value'] < 0.05).sum()} sig (FDR<0.05)")
        # quick bar of top 15
        try:
            top = res.head(15).iloc[::-1]
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.barh(top["Term"].str.slice(0, 55),
                    -np.log10(top["Adjusted P-value"].clip(lower=1e-300)),
                    color="steelblue", edgecolor="black", lw=0.3)
            ax.set_xlabel("-log10(adj p)")
            ax.set_title("Top enriched terms — leading-edge union")
            plt.tight_layout()
            fig.savefig(os.path.join(out_dir, "enrichr_top15.png"), dpi=150)
            plt.close(fig)
        except Exception as e:
            print(f"    [warn] enrichr plot failed: {e}")
    except Exception as e:
        print(f"[2] Enrichr failed ({e}); falling back to URL export only")
        results["enrichr"] = pd.DataFrame()

    gp_url = "https://biit.cs.ut.ee/gprofiler/gost"
    enrichr_url = "https://maayanlab.cloud/Enrichr/"
    print(f"    Manual options → g:Profiler: {gp_url} | Enrichr: {enrichr_url}")
    results["urls"] = {"gprofiler": gp_url, "enrichr": enrichr_url}
    return results


# ===========================================================================
# [3]  Cell-type / tissue specificity for key genes (per-tissue meta-Z)
# ===========================================================================
def tissue_specificity(trait_dirs, trait_labels, key_genes, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    key_genes = [g.upper() for g in key_genes]
    rows = []
    for trait, folder in zip(trait_labels, trait_dirs):
        _, _, tissue_gene_z = load_spredixcan_folder(folder)
        for tissue, zdict in tissue_gene_z.items():
            zdict_u = {str(k).upper(): v for k, v in zdict.items()}
            for g in key_genes:
                if g in zdict_u:
                    rows.append({"Trait": trait, "Tissue": tissue,
                                 "Gene": g, "Z": round(float(zdict_u[g]), 4)})
    df = pd.DataFrame(rows)
    if df.empty:
        print("[3] No tissue-level matches for key genes")
        return df
    df.to_csv(os.path.join(out_dir, "tissue_specificity_long.csv"), index=False)

    # one heatmap (Tissue x Gene) per trait
    for trait in trait_labels:
        sub = df[df.Trait == trait]
        if sub.empty:
            continue
        piv = sub.pivot_table(index="Tissue", columns="Gene", values="Z")
        piv.to_csv(os.path.join(out_dir, f"tissue_matrix_{trait}.csv"))
        try:
            vmax = np.nanmax(np.abs(piv.values)) or 1.0
            fig, ax = plt.subplots(
                figsize=(max(6, 0.6 * piv.shape[1] + 2),
                         max(4, 0.32 * piv.shape[0] + 1)))
            im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                           aspect="auto")
            ax.set_xticks(range(piv.shape[1]))
            ax.set_xticklabels(piv.columns, rotation=90, fontsize=7)
            ax.set_yticks(range(piv.shape[0]))
            ax.set_yticklabels(piv.index, fontsize=6)
            ax.set_title(f"{trait} — per-tissue meta-Z (key genes)")
            fig.colorbar(im, ax=ax, shrink=0.7, label="Z")
            plt.tight_layout()
            fig.savefig(os.path.join(out_dir, f"tissue_heatmap_{trait}.png"), dpi=150)
            plt.close(fig)
        except Exception as e:
            print(f"    [warn] tissue heatmap failed ({trait}): {e}")
    print(f"[3] Tissue specificity: {len(df)} rows over {df.Gene.nunique()} genes")
    return df


# ===========================================================================
# [4]  Colocalization / fine-mapping priority list (cross-trait Z variability)
# ===========================================================================
def coloc_priority(trait_dirs, trait_labels, candidate_genes, out_dir, top_n=30):
    os.makedirs(out_dir, exist_ok=True)
    candidate_genes = [g.upper() for g in candidate_genes]
    meta = {}
    for trait, folder in zip(trait_labels, trait_dirs):
        mz, _, _ = load_spredixcan_folder(folder)
        meta[trait] = {str(k).upper(): float(v) for k, v in mz.items()
                       if np.isfinite(v)}
    rows = []
    for g in candidate_genes:
        row = {"Gene": g}
        zs = []
        for trait in trait_labels:
            z = meta[trait].get(g, np.nan)
            row[f"Z_{trait}"] = round(z, 4) if z == z else np.nan
            if z == z:
                zs.append(z)
        row["N_traits"] = len(zs)
        row["Z_range"] = round(max(zs) - min(zs), 4) if len(zs) >= 2 else np.nan
        row["Max_absZ"] = round(max(abs(np.array(zs)), default=np.nan), 4) if zs else np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    df = df.sort_values("Max_absZ", ascending=False)
    df.to_csv(os.path.join(out_dir, "coloc_priority_full.csv"), index=False)
    top = df.nlargest(top_n, "Max_absZ")
    top["Gene"].to_csv(os.path.join(out_dir, "top_for_coloc.txt"),
                       index=False, header=False)
    print(f"[4] Coloc priority: top {len(top)} genes by |Z| exported")
    return df


# ===========================================================================
# [5]  Drug-repurposing screen — DGIdb GraphQL on the leading-edge union
# ===========================================================================
def drug_repurposing(gene_list, out_dir, batch=80, known_drugs=None):
    os.makedirs(out_dir, exist_ok=True)
    gene_list = [g.upper() for g in gene_list]
    url = "https://dgidb.org/api/graphql"
    query = """
    query($names:[String!]!){
      genes(names:$names){
        nodes{
          name
          interactions{
            drug{ name conceptId approved }
            interactionScore
            interactionTypes{ type }
          }
        }
      }
    }"""
    rows = []
    for i in range(0, len(gene_list), batch):
        chunk = gene_list[i:i + batch]
        try:
            r = requests.post(url, json={"query": query,
                                         "variables": {"names": chunk}},
                              timeout=60)
            r.raise_for_status()
            data = r.json().get("data", {}).get("genes", {}).get("nodes", [])
            for node in data:
                gname = node.get("name")
                for it in node.get("interactions", []):
                    drug = it.get("drug", {}) or {}
                    itypes = ";".join(t.get("type", "")
                                      for t in it.get("interactionTypes", []))
                    rows.append({
                        "gene": gname,
                        "drug": drug.get("name"),
                        "drug_concept": drug.get("conceptId"),
                        "approved": drug.get("approved"),
                        "interaction_score": it.get("interactionScore"),
                        "interaction_types": itypes,
                    })
        except Exception as e:
            print(f"    [warn] DGIdb batch {i//batch} failed: {e}")
        time.sleep(0.4)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["gene", "interaction_score"],
                            ascending=[True, False])
        df.to_csv(os.path.join(out_dir, "dgidb_interactions.csv"), index=False)
        # rank drugs by number of distinct targets hit
        drug_hits = (df.dropna(subset=["drug"])
                       .groupby("drug")
                       .agg(n_targets=("gene", "nunique"),
                            mean_score=("interaction_score", "mean"),
                            genes=("gene", lambda s: ";".join(sorted(set(s)))))
                       .sort_values(["n_targets", "mean_score"],
                                    ascending=[False, False])
                       .reset_index())
        drug_hits.to_csv(os.path.join(out_dir, "dgidb_drug_ranking.csv"),
                         index=False)
        print(f"[5] DGIdb: {len(df)} interactions, "
              f"{drug_hits.shape[0]} distinct drugs")
    else:
        drug_hits = pd.DataFrame()
        print("[5] DGIdb returned no interactions")

    overlap = pd.DataFrame()
    if known_drugs and not df.empty:
        kd = {d.lower() for d in known_drugs}
        overlap = df[df["drug"].astype(str).str.lower().isin(kd)]
        overlap.to_csv(os.path.join(out_dir, "dgidb_known_drug_overlap.csv"),
                       index=False)
        print(f"    Overlap with known drugs: {overlap['drug'].nunique()} drugs")
    return {"interactions": df, "drug_ranking": drug_hits,
            "known_overlap": overlap}


# ===========================================================================
# [6]  Signed vs absolute re-analysis (repression-model test)
# ===========================================================================
def signed_vs_abs(trait_dirs, trait_labels, group1, group2,
                  group1_label, group2_label, out_dir, n_perm=2000):
    os.makedirs(out_dir, exist_ok=True)
    print("[6] Re-running head-to-head with ranking='signed' …")
    master_signed = genage_vs_genesets_pipeline(
        trait_dirs=trait_dirs, trait_labels=trait_labels,
        group1=group1, group2=group2,
        group1_label=group1_label, group2_label=group2_label,
        out=os.path.join(out_dir, "signed"),
        ranking="signed", n_perm=n_perm, sig_z=3.0, make_plots=True,
    )
    # try to load the absolute master for comparison
    abs_master_path = None
    for cand in [os.path.join(out_dir, "..", "093FvA_genage_compare",
                              "MASTER_headtohead.csv"),
                 "/content/093FvA_genage_compare/MASTER_headtohead.csv",
                 "/content/093FvA_bip_compare/MASTER_headtohead.csv"]:
        if os.path.isfile(cand):
            abs_master_path = cand
            break
    comp = pd.DataFrame()
    if abs_master_path:
        try:
            abs_m = pd.read_csv(abs_master_path)
            key = ["Trait", "Set1", "Set2"]
            a = abs_m[key + ["NES1", "NES2", "dNES_1_vs_2"]].rename(
                columns={"NES1": "NES1_abs", "NES2": "NES2_abs",
                         "dNES_1_vs_2": "dNES_abs"})
            s = master_signed[key + ["NES1", "NES2", "dNES_1_vs_2"]].rename(
                columns={"NES1": "NES1_signed", "NES2": "NES2_signed",
                         "dNES_1_vs_2": "dNES_signed"})
            comp = a.merge(s, on=key, how="outer")
            comp.to_csv(os.path.join(out_dir, "signed_vs_abs_comparison.csv"),
                        index=False)
            print(f"    Signed-vs-abs comparison written ({len(comp)} rows)")
        except Exception as e:
            print(f"    [warn] could not merge abs master: {e}")
    return {"master_signed": master_signed, "comparison": comp}


# ===========================================================================
#  [7]  DETAILED SUMMARY TXT  — collates all important / significant findings
# ===========================================================================
def build_followup_summary(results, out, trait_labels,
                           key_hub_genes, sig_fdr=0.05, top_n=25):
    """
    Build a human-readable summary.txt collating the most important and
    significant findings from every follow-up step.
    """
    L = []
    add = L.append
    sep = "=" * 80

    add("#" * 80)
    add("#  093FvA — FOLLOW-UP DETAILED SUMMARY")
    add(f"#  Generated : {datetime.now():%Y-%m-%d %H:%M:%S}")
    add(f"#  Traits    : {', '.join(trait_labels)}")
    add(f"#  Sig codes : *** FDR<0.001  ** FDR<0.01  * FDR<{sig_fdr}")
    add("#" * 80)
    add("")

    def _stars(p):
        if p != p or p is None:
            return ""
        if p < 0.001: return " ***"
        if p < 0.01:  return " **"
        if p < sig_fdr: return " *"
        return ""

    # ----- [0] Leading-edge union --------------------------------------
    le = results.get("leading_edge", {})
    union = le.get("union", [])
    per_set = le.get("per_set", {})
    add(sep); add("  [0]  LEADING-EDGE GENE UNION"); add(sep)
    add(f"  Total unique leading-edge genes: {len(union)}")
    if per_set:
        add("")
        add("  Per (group, set) leading-edge sizes:")
        for (grp, st), genes in sorted(per_set.items()):
            add(f"    {grp:<10} {str(st):<12} : {len(genes):>4} genes")
    add("")
    add("  Key hub genes tracked: " + ", ".join(key_hub_genes))
    hubs_in_union = [g for g in key_hub_genes if g in set(union)]
    add(f"  Hubs present in union ({len(hubs_in_union)}): "
        + (", ".join(hubs_in_union) if hubs_in_union else "none"))
    add("")
    add("  Full union (alphabetical):")
    for i in range(0, len(union), 10):
        add("    " + ", ".join(union[i:i + 10]))
    add("")

    # ----- [2] Functional enrichment -----------------------------------
    enr = results.get("enrichment", {}).get("enrichr", pd.DataFrame())
    add(sep); add("  [2]  FUNCTIONAL ENRICHMENT (Enrichr)"); add(sep)
    if isinstance(enr, pd.DataFrame) and not enr.empty:
        sig = enr[enr["Adjusted P-value"] < sig_fdr]
        add(f"  Total terms tested : {len(enr)}")
        add(f"  Significant (FDR<{sig_fdr}) : {len(sig)}")
        add("")
        show = sig if not sig.empty else enr.head(top_n)
        header = "significant" if not sig.empty else f"top {top_n} (none significant)"
        add(f"  Showing {header} terms:")
        add(f"  {'Library':<28}{'Term':<48}{'adjP':>11}{'Overlap':>9}")
        for _, r in show.head(top_n).iterrows():
            lib = str(r.get("Gene_set", ""))[:26]
            term = str(r.get("Term", ""))[:46]
            ap = r.get("Adjusted P-value", np.nan)
            ov = r.get("Overlap", "")
            add(f"  {lib:<28}{term:<48}{ap:>11.2e}{str(ov):>9}{_stars(ap)}")
        # leading-edge genes behind the top terms
        add("")
        add("  Genes driving top significant terms:")
        for _, r in show.head(10).iterrows():
            genes = str(r.get("Genes", "")).replace(";", ", ")
            add(f"    [{str(r.get('Term',''))[:50]}] {genes}")
    else:
        add("  (Enrichr unavailable or returned no results — see g:Profiler URL)")
    add("")

    # ----- [3] Tissue specificity --------------------------------------
    tis = results.get("tissue", pd.DataFrame())
    add(sep); add("  [3]  TISSUE / CELL-TYPE SPECIFICITY (per-tissue meta-Z)"); add(sep)
    if isinstance(tis, pd.DataFrame) and not tis.empty:
        add(f"  Rows: {len(tis)} | Genes: {tis.Gene.nunique()} | "
            f"Tissues: {tis.Tissue.nunique()} | Traits: {tis.Trait.nunique()}")
        add("")
        # strongest tissue-gene associations overall
        tis2 = tis.assign(absZ=tis["Z"].abs())
        add(f"  Top {top_n} strongest tissue-gene associations (|Z|):")
        add(f"  {'Trait':<8}{'Gene':<12}{'Tissue':<34}{'Z':>9}")
        for _, r in tis2.sort_values("absZ", ascending=False).head(top_n).iterrows():
            add(f"  {r['Trait']:<8}{r['Gene']:<12}{str(r['Tissue'])[:32]:<34}{r['Z']:>9}")
        add("")
        # focus on hub genes
        add("  Hub-gene tissue profile (max |Z| tissue per gene):")
        for g in key_hub_genes:
            sub = tis2[tis2.Gene == g]
            if sub.empty:
                continue
            best = sub.loc[sub["absZ"].idxmax()]
            add(f"    {g:<10}: peak Z={best['Z']:+.3f} in "
                f"{best['Tissue']} ({best['Trait']}); "
                f"n_tissues={sub.Tissue.nunique()}")
    else:
        add("  (no tissue-level matches)")
    add("")

    # ----- [4] Coloc priority ------------------------------------------
    col = results.get("coloc", pd.DataFrame())
    add(sep); add("  [4]  COLOCALIZATION / FINE-MAPPING PRIORITY"); add(sep)
    if isinstance(col, pd.DataFrame) and not col.empty:
        zc = [c for c in col.columns if c.startswith("Z_")]
        add(f"  Candidate genes scored: {len(col)} | trait Z-cols: {', '.join(zc)}")
        add("")
        add(f"  Top {top_n} priority genes by Max_absZ:")
        head_cols = "  " + f"{'Gene':<12}{'Max_absZ':>10}{'Z_range':>10}{'N_traits':>10}"
        add(head_cols + "".join(f"{c:>10}" for c in zc))
        for _, r in col.head(top_n).iterrows():
            line = (f"  {r['Gene']:<12}{r.get('Max_absZ', np.nan):>10}"
                    f"{r.get('Z_range', np.nan):>10}{int(r.get('N_traits',0)):>10}")
            line += "".join(f"{r.get(c, np.nan):>10}" for c in zc)
            add(line)
        # cross-trait divergent genes (only if >1 trait)
        if len(trait_labels) > 1 and "Z_range" in col.columns:
            add("")
            add(f"  Most cross-trait-divergent genes (top {top_n} by Z_range):")
            for _, r in col.sort_values("Z_range", ascending=False).head(top_n).iterrows():
                add(f"    {r['Gene']:<12} Z_range={r['Z_range']}")
    else:
        add("  (no coloc data)")
    add("")

    # ----- [5] Drug repurposing ----------------------------------------
    drugs = results.get("drugs", {})
    dr = drugs.get("drug_ranking", pd.DataFrame())
    inter = drugs.get("interactions", pd.DataFrame())
    overlap = drugs.get("known_overlap", pd.DataFrame())
    add(sep); add("  [5]  DRUG REPURPOSING (DGIdb)"); add(sep)
    if isinstance(inter, pd.DataFrame) and not inter.empty:
        add(f"  Total interactions: {len(inter)} | "
            f"distinct drugs: {dr.shape[0] if not dr.empty else 0} | "
            f"distinct genes hit: {inter['gene'].nunique()}")
        add("")
        if not dr.empty:
            add(f"  Top {top_n} drugs by #distinct targets in leading-edge union:")
            add(f"  {'Drug':<32}{'n_targets':>10}{'mean_score':>12}  genes")
            for _, r in dr.head(top_n).iterrows():
                add(f"  {str(r['drug'])[:30]:<32}{int(r['n_targets']):>10}"
                    f"{r['mean_score']:>12.3f}  {str(r['genes'])[:60]}")
        add("")
        if isinstance(overlap, pd.DataFrame) and not overlap.empty:
            add("  >> Overlap with KNOWN BD/psych drugs:")
            for drug, sub in overlap.groupby("drug"):
                genes = ", ".join(sorted(sub["gene"].unique()))
                add(f"     {drug}: targets {genes}")
        else:
            add("  No overlap with the supplied known-drug list.")
    else:
        add("  (DGIdb returned no interactions)")
    add("")

    # ----- [6] Signed vs absolute --------------------------------------
    add(sep); add("  [6]  SIGNED vs ABSOLUTE RE-ANALYSIS (repression-model test)"); add(sep)
    signed = results.get("signed", {})
    comp = signed.get("comparison", pd.DataFrame()) if isinstance(signed, dict) else pd.DataFrame()
    ms = signed.get("master_signed", pd.DataFrame()) if isinstance(signed, dict) else pd.DataFrame()
    if isinstance(comp, pd.DataFrame) and not comp.empty:
        add("  Signed vs absolute NES comparison (per Trait/Set1/Set2):")
        add(f"  {'Trait':<7}{'Set1':<9}{'Set2':<14}"
            f"{'NES1_abs':>10}{'NES1_sgn':>10}{'dNES_abs':>10}{'dNES_sgn':>10}")
        for _, r in comp.iterrows():
            add(f"  {str(r['Trait']):<7}{str(r['Set1']):<9}{str(r['Set2'])[:12]:<14}"
                f"{r.get('NES1_abs', np.nan):>10}{r.get('NES1_signed', np.nan):>10}"
                f"{r.get('dNES_abs', np.nan):>10}{r.get('dNES_signed', np.nan):>10}")
        add("")
        add("  Interpretation: a NES that stays positive under SIGNED ranking with")
        add("  negative leading-edge Z supports the repression (Polycomb) model;")
        add("  sign-flips between abs and signed indicate direction-driven signal.")
    elif isinstance(ms, pd.DataFrame) and not ms.empty:
        add("  Signed head-to-head NES (absolute master not found for merge):")
        cols = [c for c in ["Trait", "Set1", "Set2", "NES1", "GSEA_p1",
                            "NES2", "GSEA_p2", "dNES_1_vs_2", "Verdict"]
                if c in ms.columns]
        add("  " + ms[cols].to_string(index=False).replace("\n", "\n  "))
    else:
        add("  (signed re-analysis not run or produced no output)")
    add("")

    # ----- headline takeaways ------------------------------------------
    add(sep); add("  HEADLINE TAKEAWAYS"); add(sep)
    if isinstance(enr, pd.DataFrame) and not enr.empty:
        nsig = int((enr["Adjusted P-value"] < sig_fdr).sum())
        add(f"  - {nsig} pathway/ontology terms significant at FDR<{sig_fdr}.")
    if isinstance(col, pd.DataFrame) and not col.empty:
        topg = col.head(5)["Gene"].tolist()
        add(f"  - Highest-priority fine-mapping genes: {', '.join(topg)}.")
    if isinstance(dr, pd.DataFrame) and not dr.empty:
        topd = dr.head(5)["drug"].tolist()
        add(f"  - Top multi-target repurposing candidates: {', '.join(map(str, topd))}.")
    if hubs_in_union:
        add(f"  - Tracked hub genes recovered in union: {', '.join(hubs_in_union)}.")
    add("")
    add(sep)

    summ = "\n".join(L)
    path = os.path.join(out, "FOLLOWUP_SUMMARY.txt")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(summ)
    print(f"\n[7] Detailed summary written → {path}")
    return summ


# ===========================================================================
#  MASTER DRIVER — runs everything, builds summary, returns all  (UPDATED)
# ===========================================================================
def run_all_followups(
    trait_dirs,
    trait_labels,
    group1,
    group2,
    group1_label="Ageing",
    group2_label="AntiDep",
    posthoc_dir="/content/093FvA_posthoc",
    headtohead_csv="/content/093FvA_genage_compare/MASTER_headtohead.csv",
    out="/content/093FvA_followups",
    key_hub_genes=("GABRA6", "RPTOR", "MAPK3", "GABRB2", "GNB3", "DRD2",
                   "GSK3B", "ANK3", "CYP2D6", "CHRM5", "HTR6", "CACNA1B",
                   "MAPK1", "SF3B1", "FYN", "LIMK1", "CTNNB1", "GLS2"),
    known_bd_drugs=("lithium", "valproate", "lamotrigine", "quetiapine",
                    "olanzapine", "carbamazepine", "fluoxetine"),
    n_perm=2000,
    run_signed=True,
):
    os.makedirs(out, exist_ok=True)
    results = {}

    # [0] harvest
    le = harvest_leading_edge(posthoc_dir, headtohead_csv,
                              extra_hub_genes=key_hub_genes)
    results["leading_edge"] = le
    union = le["union"]

    # [2] functional enrichment
    results["enrichment"] = functional_enrichment(
        union, os.path.join(out, "02_enrichment"))

    # [3] tissue specificity (hubs + a capped sample of the union)
    tissue_genes = sorted(set(key_hub_genes) | set(union[:60]))
    results["tissue"] = tissue_specificity(
        trait_dirs, trait_labels, tissue_genes,
        os.path.join(out, "03_tissue"))

    # [4] coloc priority list (union, scored across traits)
    results["coloc"] = coloc_priority(
        trait_dirs, trait_labels, union,
        os.path.join(out, "04_coloc"))

    # [5] drug repurposing
    results["drugs"] = drug_repurposing(
        union, os.path.join(out, "05_drugs"),
        known_drugs=known_bd_drugs)

    # [6] signed vs absolute
    if run_signed:
        results["signed"] = signed_vs_abs(
            trait_dirs, trait_labels, group1, group2,
            group1_label, group2_label,
            os.path.join(out, "06_signed"), n_perm=n_perm)

    # [7] detailed summary txt  (NEW)
    results["summary_text"] = build_followup_summary(
        results, out, trait_labels, list(key_hub_genes))

    # manifest
    with open(os.path.join(out, "followup_manifest.json"), "w") as fh:
        json.dump({
            "generated_at": datetime.now().isoformat(timespec="seconds"),
            "n_union_genes": len(union),
            "trait_labels": trait_labels,
            "key_hub_genes": list(key_hub_genes),
            "n_perm": n_perm,
        }, fh, indent=2)

    print(f"\n{'='*72}\n  ALL FOLLOW-UPS COMPLETE → {os.path.abspath(out)}\n{'='*72}")
    return results

### Running

In [14]:
# ============================================================================
# Run Stage-1 followups for BIP ancestry-specific traits only
# ============================================================================

positive_traits = [
    "bip_afr",
    "bip_eas",
    "bip_eur",
]

# Map trait → actual folder name
trait_dir_map = {
    "bip_afr": "bip_afr",
    "bip_eas": "bip_eas",
    "bip_eur": "bip_eur",
}

followup_results_all = {}

for trait in positive_traits:
    folder_name = trait_dir_map[trait]

    print("\n" + "=" * 80)
    print(f"Running followups for: {trait}")
    print("=" * 80)

    followup_results_all[trait] = run_all_followups(
        trait_dirs      = [f"/content/{folder_name}"],
        trait_labels    = [trait],
        group1          = group1,
        group2          = group2,
        group1_label    = "polycomb",
        group2_label    = "AntiPsy",
        posthoc_dir     = f"/content/096AvAp3_posthoc_{trait}",
        headtohead_csv  = "/content/096AvAp3_compare/MASTER_headtohead.csv",
        out             = f"/content/096AvAp3_followups_{trait}",
        n_perm          = 2000,
        run_signed      = True,
    )

print("\n" + "=" * 80)
print("✅ All follow-up runs completed for BIP ancestry-specific traits.")
print("=" * 80)


Running followups for: bip_afr
[0] Harvested 604 unique leading-edge genes across 17 (group,set) blocks
[2] Enrichr: 6935 terms; 1578 sig (FDR<0.05)
    Manual options → g:Profiler: https://biit.cs.ut.ee/gprofiler/gost | Enrichr: https://maayanlab.cloud/Enrichr/
    6 file(s) in bip_afr/
    ⚠ Brain_Amygdala_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Anterior_cingulate_cortex_BA24_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Caudate_basal_ganglia_spredixcan.csv: 2 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Frontal_Cortex_BA9_spredixcan.csv: 2 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Hippocampus_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    ⚠ Brain_Nucleus_accumbens_basal_ganglia_spredixcan.csv: 1 duplicate 'gene_name' rows (kept first occurrence)
    → meta-Z computed for 10,703 genes
    → Ensembl aliases mapped: 21,412
    → 6 tissue file(s) 

# Stage 4

In [18]:
# ============================================================================
#  096AvAp3 — STAGE 4  TRANSANCESTRY INTEGRATIVE PIPELINE
#  Consumes ONLY the outputs of the head-to-head / post-hoc / follow-up runs.
#  No raw TWAS re-processing.  Focus: bip_afr vs bip_eas vs bip_eur
#  Analyses:
#    [1] Transancestry meta-NES & consistency score
#    [2] Cross-ancestry verdict concordance
#    [3] Shared vs ancestry-specific leading-edge genes
#    [4] Transancestry hub-gene prioritization
#    [5] Ancestry x set interaction on |Z| distributions
#    [6] Transancestry drug-repurposing overlap
#    [7] Signed-vs-abs repression-model consistency (from T2 + followup)
#    [8] BONUS: transancestry pathway-enrichment consistency (Enrichr)
#  Ends with one very detailed STAGE4_DETAILED_SUMMARY.txt
# ============================================================================
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, combine_pvalues
from datetime import datetime
from itertools import combinations
warnings.filterwarnings("ignore")


# ===========================================================================
#  CONFIG  — edit paths / labels here if yours differ
# ===========================================================================
CFG = dict(
    traits        = ["bip_afr", "bip_eas", "bip_eur"],
    compare_dir   = "/content/096AvAp3_compare",
    posthoc_tmpl  = "/content/096AvAp3_posthoc_{trait}",
    followup_tmpl = "/content/096AvAp3_followups_{trait}",
    out           = "/content/096AvAp3_stage4",
    polycomb_group_names = ["polycomb", "ageing_polycomb"],   # handles label diff
    antipsy_group_names  = ["AntiPsy", "AntiPsy_Genes"],
    hub_genes = ["GABRA6","RPTOR","MAPK3","GABRB2","GNB3","DRD2","GSK3B","ANK3",
                 "CYP2D6","CHRM5","HTR6","CACNA1B","MAPK1","SF3B1","FYN","LIMK1",
                 "CTNNB1","GLS2"],
    known_drugs = ["lithium","valproate","valproic acid","lamotrigine","quetiapine",
                   "olanzapine","carbamazepine","fluoxetine","risperidone",
                   "aripiprazole","clozapine","haloperidol","lithium carbonate"],
    alpha = 0.05,
    top_n = 30,
)


# ===========================================================================
#  small utilities
# ===========================================================================
def _read(path):
    try:
        if os.path.isfile(path):
            return pd.read_csv(path)
    except Exception as e:
        print(f"    [warn] read failed {path}: {e}")
    return pd.DataFrame()

def bh_fdr(p):
    p = np.asarray(p, float); n = len(p)
    if n == 0: return p
    order = np.argsort(p); adj = np.empty(n); cmin = 1.0
    for i in range(n - 1, -1, -1):
        cmin = min(cmin, p[order[i]] * n / (i + 1))
        adj[order[i]] = min(cmin, 1.0)
    return adj

def combine_p(pvals, method="stouffer"):
    pv = [p for p in pvals if p == p and p is not None]
    if len(pv) == 0: return np.nan
    pv = np.clip(pv, 1e-300, 1.0)
    if len(pv) == 1: return float(pv[0])
    try:    return float(combine_pvalues(pv, method=method)[1])
    except Exception:
        try:    return float(combine_pvalues(pv, method="fisher")[1])
        except Exception: return float(np.min(pv))

def f(x, nd=3):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)): return "NA"
        if isinstance(x, (int, np.integer)): return str(int(x))
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def stars(p, alpha=0.05):
    if p != p or p is None: return ""
    if p < 0.001: return " ***"
    if p < 0.01:  return " **"
    if p < alpha: return " *"
    return ""


# ===========================================================================
#  LOADERS  — pull every relevant table across the three ancestries
# ===========================================================================
def load_all(cfg):
    traits = cfg["traits"]
    D = dict(master=pd.DataFrame(),
             T1={}, T2={}, T3={}, T4={}, T5={}, T6={},
             coloc={}, drugs={}, drug_overlap={}, enrich={}, signed_cmp={},
             influence={}, le_union={})

    # --- MASTER head-to-head (all traits in one file) ---
    D["master"] = _read(os.path.join(cfg["compare_dir"], "MASTER_headtohead.csv"))

    for t in traits:
        pd_dir = cfg["posthoc_tmpl"].format(trait=t)
        fu_dir = cfg["followup_tmpl"].format(trait=t)

        D["T1"][t] = _read(os.path.join(pd_dir, "T1_leading_edge_overlap.csv"))
        D["T2"][t] = _read(os.path.join(pd_dir, "T2_directionality.csv"))
        D["T3"][t] = _read(os.path.join(pd_dir, "T3_gene_contributions.csv"))
        D["T4"][t] = _read(os.path.join(pd_dir, "T4_remove_overlap_retest.csv"))
        D["T5"][t] = _read(os.path.join(pd_dir, "T5_LOO_group2.csv"))
        D["T6"][t] = _read(os.path.join(pd_dir, "T6_Zdist_comparison.csv"))

        # optional gene-influence (only if a run_pipeline report exists)
        inf = pd.DataFrame()
        for cand in glob.glob(os.path.join(pd_dir, "*influence*.csv")) + \
                    glob.glob(os.path.join(pd_dir, "**", "gene_influence*.csv"),
                              recursive=True):
            inf = _read(cand)
            if not inf.empty: break
        D["influence"][t] = inf

        # follow-up artefacts
        D["coloc"][t]        = _read(os.path.join(fu_dir, "04_coloc",
                                                  "coloc_priority_full.csv"))
        D["drugs"][t]        = _read(os.path.join(fu_dir, "05_drugs",
                                                  "dgidb_drug_ranking.csv"))
        D["drug_overlap"][t] = _read(os.path.join(fu_dir, "05_drugs",
                                                  "dgidb_known_drug_overlap.csv"))
        D["enrich"][t]       = _read(os.path.join(fu_dir, "02_enrichment",
                                                  "enrichr_results.csv"))
        D["signed_cmp"][t]   = _read(os.path.join(fu_dir, "06_signed",
                                                  "signed_vs_abs_comparison.csv"))
        # leading-edge union list (txt)
        lp = os.path.join(fu_dir, "02_enrichment", "leading_edge_union.txt")
        if os.path.isfile(lp):
            with open(lp) as fh:
                D["le_union"][t] = [x.strip().upper() for x in fh if x.strip()]
        else:
            D["le_union"][t] = []
    return D


# ===========================================================================
#  ANALYSIS 1 — transancestry meta-NES & consistency
# ===========================================================================
def a1_meta_nes(D, cfg):
    m = D["master"].copy()
    if m.empty: return pd.DataFrame()
    # keep polycomb-vs-antipsy rows (Group1 = polycomb-ish)
    grp_ok = m["Group1"].astype(str).isin(cfg["polycomb_group_names"]) \
             if "Group1" in m.columns else pd.Series(True, index=m.index)
    m = m[grp_ok].copy()
    if "Verdict" in m.columns:
        m["beats"] = (m["Verdict"].astype(str) == "set1_beats_set2").astype(int)
    else:
        m["beats"] = np.nan

    rows = []
    for s1, sub in m.groupby("Set1"):
        nes = pd.to_numeric(sub.get("NES1"), errors="coerce")
        dne = pd.to_numeric(sub.get("dNES_1_vs_2"), errors="coerce")
        p1  = pd.to_numeric(sub.get("GSEA_p1"), errors="coerce")
        cp  = combine_p(p1.tolist())
        rows.append(dict(
            Set1=s1,
            n_ancestries=sub["Trait"].nunique(),
            NES1_mean=round(nes.mean(), 4),
            NES1_std=round(nes.std(ddof=0), 4) if nes.notna().sum() > 1 else np.nan,
            NES1_min=round(nes.min(), 4),
            dNES_mean=round(dne.mean(), 4),
            dNES_min=round(dne.min(), 4),
            frac_beats=round(sub["beats"].mean(), 4) if sub["beats"].notna().any() else np.nan,
            min_GSEA_p=round(p1.min(), 6) if p1.notna().any() else np.nan,
            combined_p_stouffer=round(cp, 6) if cp == cp else np.nan,
        ))
    out = pd.DataFrame(rows)
    if out.empty: return out
    out["combined_FDR"] = bh_fdr(np.where(np.isfinite(out["combined_p_stouffer"]),
                                          out["combined_p_stouffer"], 1.0))
    # consistency: reward winning everywhere with low variance
    penalty = out["NES1_std"].fillna(out["NES1_std"].max() if out["NES1_std"].notna().any() else 1.0)
    denom = (penalty.max() + 1e-9)
    out["consistency_score"] = round(
        out["frac_beats"].fillna(0) * (1 - penalty / denom), 4)
    out = out.sort_values(["consistency_score", "NES1_mean"],
                          ascending=[False, False]).reset_index(drop=True)
    out.to_csv(os.path.join(cfg["out"], "stage4_meta_nes_consistency.csv"), index=False)

    # forest-style plot of NES mean +/- std for top sets
    try:
        top = out.head(25).iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(top) + 1)))
        y = np.arange(len(top))
        ax.errorbar(top["NES1_mean"], y, xerr=top["NES1_std"].fillna(0),
                    fmt="o", color="steelblue", ecolor="gray", capsize=3)
        ax.axvline(0, color="black", lw=0.8)
        ax.axvline(1, color="red", ls="--", lw=0.7)
        ax.set_yticks(y); ax.set_yticklabels(top["Set1"], fontsize=7)
        ax.set_xlabel("Mean NES1 (± SD across ancestries)")
        ax.set_title("Transancestry meta-NES (Polycomb sets vs AntiPsy)")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage4_meta_nes_forest.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A1 plot failed: {e}")
    return out


# ===========================================================================
#  ANALYSIS 2 — cross-ancestry verdict concordance
# ===========================================================================
def a2_verdict(D, cfg):
    m = D["master"].copy()
    if m.empty or "Verdict" not in m.columns: return pd.DataFrame(), pd.DataFrame()
    grp_ok = m["Group1"].astype(str).isin(cfg["polycomb_group_names"]) \
             if "Group1" in m.columns else pd.Series(True, index=m.index)
    m = m[grp_ok]
    piv = m.pivot_table(index="Set1", columns="Trait",
                        values="Verdict", aggfunc="first")
    piv_num = piv.replace({"set1_beats_set2": 2, "not_better": 1}).fillna(0)
    concord = pd.DataFrame(index=piv.index)
    concord["n_wins"] = (piv == "set1_beats_set2").sum(axis=1)
    concord["n_tested"] = piv.notna().sum(axis=1)
    concord["win_fraction"] = (concord["n_wins"] / concord["n_tested"].replace(0, np.nan)).round(3)
    concord["class"] = np.where(concord["n_wins"] == concord["n_tested"], "WIN_ALL",
                        np.where(concord["n_wins"] == 0, "WIN_NONE", "PARTIAL"))
    concord = concord.sort_values(["n_wins", "win_fraction"], ascending=False)
    piv.to_csv(os.path.join(cfg["out"], "stage4_verdict_matrix.csv"))
    concord.to_csv(os.path.join(cfg["out"], "stage4_verdict_concordance.csv"))

    try:
        fig, ax = plt.subplots(figsize=(1.2 * piv.shape[1] + 3,
                                        max(4, 0.32 * piv.shape[0] + 1)))
        sns.heatmap(piv_num.astype(float), cmap="RdYlGn", vmin=0, vmax=2,
                    annot=piv.fillna("—"), fmt="", cbar=False,
                    linewidths=0.5, linecolor="white", ax=ax,
                    annot_kws={"fontsize": 6})
        ax.set_title("Polycomb-set verdict by ancestry\n(green=beats AntiPsy)")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage4_verdict_concordance.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A2 plot failed: {e}")
    return piv, concord


# ===========================================================================
#  ANALYSIS 3 — shared vs ancestry-specific leading-edge genes
# ===========================================================================
def a3_shared_genes(D, cfg):
    traits = cfg["traits"]
    # per-ancestry polycomb leading-edge gene sets from T3
    per_anc = {}
    for t in traits:
        t3 = D["T3"].get(t, pd.DataFrame())
        if t3.empty: per_anc[t] = set(); continue
        poly = t3[t3["Group"].astype(str).isin(cfg["polycomb_group_names"])] \
               if "Group" in t3.columns else t3
        per_anc[t] = set(poly["gene"].astype(str).str.upper())

    all_genes = set().union(*per_anc.values()) if per_anc else set()
    rows = []
    for g in sorted(all_genes):
        present = [t for t in traits if g in per_anc[t]]
        rows.append(dict(gene=g, n_ancestries=len(present),
                         ancestries=";".join(present),
                         is_hub=g in set(cfg["hub_genes"])))
    rec = pd.DataFrame(rows).sort_values(["n_ancestries", "gene"],
                                         ascending=[False, True])
    rec.to_csv(os.path.join(cfg["out"], "stage4_leading_edge_recurrence.csv"), index=False)

    core = rec[rec["n_ancestries"] == len(traits)]["gene"].tolist()
    shared2 = rec[rec["n_ancestries"] == 2]["gene"].tolist()
    with open(os.path.join(cfg["out"], "stage4_core_genes_all_ancestries.txt"), "w") as fh:
        fh.write("\n".join(core))

    try:
        counts = rec["n_ancestries"].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(counts.index.astype(str), counts.values,
               color="teal", edgecolor="black")
        for x, v in zip(counts.index.astype(str), counts.values):
            ax.text(x, v, str(v), ha="center", va="bottom", fontsize=9)
        ax.set_xlabel("# ancestries a gene appears in (leading edge)")
        ax.set_ylabel("# genes")
        ax.set_title("Leading-edge gene recurrence across ancestries")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage4_gene_recurrence.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A3 plot failed: {e}")
    return rec, core, shared2, per_anc


# ===========================================================================
#  ANALYSIS 4 — transancestry hub-gene prioritization
# ===========================================================================
def a4_hubs(D, cfg):
    traits = cfg["traits"]; hubs = cfg["hub_genes"]
    rows = []
    for t in traits:
        t3 = D["T3"].get(t, pd.DataFrame())
        inf = D["influence"].get(t, pd.DataFrame())
        for g in hubs:
            maxabs = np.nan; sgnZ = np.nan; le_hit = False
            if not t3.empty:
                sub = t3[t3["gene"].astype(str).str.upper() == g]
                if not sub.empty:
                    idx = sub["absZ"].astype(float).idxmax()
                    maxabs = float(sub.loc[idx, "absZ"])
                    sgnZ   = float(sub.loc[idx, "Z"])
                    le_hit = True
            infl = np.nan; n_infl = np.nan
            if not inf.empty and "Gene" in inf.columns:
                si = inf[inf["Gene"].astype(str).str.upper() == g]
                if not si.empty:
                    col = "Pct_Change_Stouffer" if "Pct_Change_Stouffer" in si else None
                    if col: infl = float(si[col].abs().max())
                    if "Influential" in si:
                        n_infl = int((si["Influential"] == True).sum())
            rows.append(dict(ancestry=t, gene=g, in_leading_edge=le_hit,
                             max_absZ=round(maxabs, 4) if maxabs == maxabs else np.nan,
                             signedZ=round(sgnZ, 4) if sgnZ == sgnZ else np.nan,
                             max_influence_pct=round(infl, 3) if infl == infl else np.nan,
                             n_influential=n_infl))
    long = pd.DataFrame(rows)
    # aggregate per hub
    agg = (long.groupby("gene")
           .agg(n_anc_in_LE=("in_leading_edge", "sum"),
                mean_max_absZ=("max_absZ", "mean"),
                max_max_absZ=("max_absZ", "max"),
                mean_signedZ=("signedZ", "mean"))
           .reset_index()
           .sort_values(["n_anc_in_LE", "mean_max_absZ"], ascending=[False, False]))
    long.to_csv(os.path.join(cfg["out"], "stage4_hub_long.csv"), index=False)
    agg.to_csv(os.path.join(cfg["out"], "stage4_hub_prioritization.csv"), index=False)

    try:
        piv = long.pivot_table(index="gene", columns="ancestry",
                               values="max_absZ")
        piv = piv.reindex(agg["gene"])
        fig, ax = plt.subplots(figsize=(1.4 * len(traits) + 3,
                                        max(4, 0.35 * len(piv) + 1)))
        sns.heatmap(piv.astype(float), cmap="viridis", annot=True, fmt=".2f",
                    linewidths=0.5, linecolor="white", ax=ax,
                    cbar_kws={"label": "max |Z|"})
        ax.set_title("Hub-gene max |Z| by ancestry")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage4_hub_heatmap.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A4 plot failed: {e}")
    return long, agg


# ===========================================================================
#  ANALYSIS 5 — ancestry x set interaction on |Z| distributions
# ===========================================================================
def a5_interaction(D, cfg):
    traits = cfg["traits"]
    # gather per-set polycomb absZ distributions across ancestries (from T3)
    setmap = {}  # {set : {ancestry : [absZ,...]}}
    for t in traits:
        t3 = D["T3"].get(t, pd.DataFrame())
        if t3.empty: continue
        poly = t3[t3["Group"].astype(str).isin(cfg["polycomb_group_names"])] \
               if "Group" in t3.columns else t3
        for s, sub in poly.groupby("Set"):
            setmap.setdefault(s, {})[t] = sub["absZ"].astype(float).dropna().tolist()

    rows = []
    for s, anc in setmap.items():
        groups = [v for v in anc.values() if len(v) >= 3]
        if len(groups) >= 2:
            try:
                H, p = kruskal(*groups)
            except Exception:
                H, p = np.nan, np.nan
        else:
            H, p = np.nan, np.nan
        med = {t: (round(np.median(anc[t]), 4) if t in anc and anc[t] else np.nan)
               for t in traits}
        rows.append(dict(Set=s, n_anc=len(anc), kruskal_H=round(H, 4) if H == H else np.nan,
                         kruskal_p=round(p, 6) if p == p else np.nan,
                         **{f"median_absZ_{t}": med[t] for t in traits}))
    out = pd.DataFrame(rows)
    if not out.empty:
        out["kruskal_FDR"] = bh_fdr(np.where(np.isfinite(out["kruskal_p"]),
                                             out["kruskal_p"], 1.0))
        out = out.sort_values("kruskal_p").reset_index(drop=True)
    out.to_csv(os.path.join(cfg["out"], "stage4_ancestry_set_interaction.csv"), index=False)
    return out


# ===========================================================================
#  ANALYSIS 6 — transancestry drug-repurposing overlap
# ===========================================================================
def a6_drugs(D, cfg):
    traits = cfg["traits"]
    frames = []
    for t in traits:
        dr = D["drugs"].get(t, pd.DataFrame())
        if dr.empty: continue
        dr = dr.copy(); dr["ancestry"] = t
        dr["drug_l"] = dr["drug"].astype(str).str.lower()
        frames.append(dr)
    if not frames:
        return pd.DataFrame(), pd.DataFrame()
    alld = pd.concat(frames, ignore_index=True)

    agg = (alld.groupby("drug_l")
           .agg(drug=("drug", "first"),
                n_ancestries=("ancestry", "nunique"),
                total_targets=("n_targets", "sum"),
                mean_targets=("n_targets", "mean"),
                mean_score=("mean_score", "mean"),
                ancestries=("ancestry", lambda s: ";".join(sorted(set(s)))))
           .reset_index(drop=True))
    known = set(x.lower() for x in cfg["known_drugs"])
    agg["is_known_psych_drug"] = agg["drug"].astype(str).str.lower().isin(known)
    agg = agg.sort_values(["n_ancestries", "mean_targets"],
                          ascending=[False, False]).reset_index(drop=True)
    agg.to_csv(os.path.join(cfg["out"], "stage4_drug_recurrence.csv"), index=False)

    try:
        top = agg.head(25).iloc[::-1]
        piv = (alld[alld["drug_l"].isin(top["drug"].str.lower())]
               .pivot_table(index="drug", columns="ancestry",
                            values="n_targets", aggfunc="sum"))
        piv = piv.reindex(top["drug"])
        fig, ax = plt.subplots(figsize=(1.4 * len(traits) + 3,
                                        max(4, 0.35 * len(piv) + 1)))
        sns.heatmap(piv.astype(float), cmap="Reds", annot=True, fmt=".0f",
                    linewidths=0.5, linecolor="white", ax=ax,
                    cbar_kws={"label": "# targets hit"})
        ax.set_title("Top repurposing drugs: targets hit by ancestry")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage4_drug_heatmap.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A6 plot failed: {e}")
    return alld, agg


# ===========================================================================
#  ANALYSIS 7 — signed-vs-abs / repression-model consistency
# ===========================================================================
def a7_repression(D, cfg):
    traits = cfg["traits"]
    # directionality of polycomb leading edges (T2), per ancestry
    rows = []
    for t in traits:
        t2 = D["T2"].get(t, pd.DataFrame())
        if t2.empty: continue
        poly = t2[t2["Group"].astype(str).isin(cfg["polycomb_group_names"])] \
               if "Group" in t2.columns else t2
        for _, r in poly.iterrows():
            rows.append(dict(ancestry=t, Set=r.get("Set"),
                             n_LE=r.get("n_LE"), pct_neg=r.get("pct_neg"),
                             mean_Z=r.get("mean_Z"),
                             sign_test_p=r.get("sign_test_p")))
    dirdf = pd.DataFrame(rows)

    summ = pd.DataFrame()
    if not dirdf.empty:
        summ = (dirdf.groupby("Set")
                .agg(n_anc=("ancestry", "nunique"),
                     mean_pct_neg=("pct_neg", "mean"),
                     mean_meanZ=("mean_Z", "mean"),
                     n_sig_neg=("sign_test_p",
                                lambda s: int(((pd.to_numeric(s, errors="coerce") < cfg["alpha"]).values &
                                               (dirdf.loc[s.index, "mean_Z"].astype(float) < 0).values).sum())),
                     combined_sign_p=("sign_test_p",
                                      lambda s: combine_p(pd.to_numeric(s, errors="coerce").tolist())))
                .reset_index())
        summ["repression_support"] = np.where(
            (summ["mean_meanZ"] < 0) & (summ["n_sig_neg"] >= 2), "STRONG",
            np.where((summ["mean_meanZ"] < 0) & (summ["n_sig_neg"] >= 1), "MODERATE",
                     "WEAK/NONE"))
        summ = summ.sort_values(["n_sig_neg", "mean_pct_neg"],
                                ascending=[False, False]).reset_index(drop=True)
    dirdf.to_csv(os.path.join(cfg["out"], "stage4_directionality_long.csv"), index=False)
    if not summ.empty:
        summ.to_csv(os.path.join(cfg["out"], "stage4_repression_model.csv"), index=False)

    # signed-vs-abs NES comparison merged across ancestries (if present)
    scmp = []
    for t in traits:
        c = D["signed_cmp"].get(t, pd.DataFrame())
        if not c.empty:
            c = c.copy(); c["ancestry"] = t; scmp.append(c)
    scmp = pd.concat(scmp, ignore_index=True) if scmp else pd.DataFrame()
    if not scmp.empty:
        scmp.to_csv(os.path.join(cfg["out"], "stage4_signed_vs_abs_all.csv"), index=False)
    return dirdf, summ, scmp


# ===========================================================================
#  ANALYSIS 8 (bonus) — transancestry pathway-enrichment consistency
# ===========================================================================
def a8_enrichment(D, cfg):
    traits = cfg["traits"]; alpha = cfg["alpha"]
    frames = []
    for t in traits:
        e = D["enrich"].get(t, pd.DataFrame())
        if e.empty or "Adjusted P-value" not in e.columns: continue
        sig = e[e["Adjusted P-value"] < alpha].copy()
        sig["ancestry"] = t
        frames.append(sig[["Term", "Gene_set", "Adjusted P-value", "ancestry"]]
                      if "Gene_set" in sig.columns else
                      sig[["Term", "Adjusted P-value", "ancestry"]])
    if not frames:
        return pd.DataFrame()
    allsig = pd.concat(frames, ignore_index=True)
    rec = (allsig.groupby("Term")
           .agg(n_ancestries=("ancestry", "nunique"),
                min_adjP=("Adjusted P-value", "min"),
                ancestries=("ancestry", lambda s: ";".join(sorted(set(s)))))
           .reset_index()
           .sort_values(["n_ancestries", "min_adjP"], ascending=[False, True]))
    rec.to_csv(os.path.join(cfg["out"], "stage4_enrichment_consistency.csv"), index=False)
    return rec


# ===========================================================================
#  DETAILED SUMMARY BUILDER
# ===========================================================================
def build_summary(cfg, R):
    a = cfg["alpha"]; traits = cfg["traits"]; TN = cfg["top_n"]
    L = []; add = L.append
    bar = "=" * 84

    add("#" * 84)
    add("#   096AvAp3 — STAGE 4 TRANSANCESTRY INTEGRATIVE SUMMARY")
    add(f"#   Generated : {datetime.now():%Y-%m-%d %H:%M:%S}")
    add(f"#   Ancestries: {', '.join(traits)}")
    add(f"#   Contrast  : Polycomb (group1) vs AntiPsy (group2)")
    add(f"#   Sig codes : *** p<0.001  ** p<0.01  * p<{a}")
    add("#" * 84); add("")

    # ---- A1 meta-NES ----
    add(bar); add("  [1]  TRANSANCESTRY META-NES & CONSISTENCY (Polycomb sets)"); add(bar)
    m = R.get("a1", pd.DataFrame())
    if isinstance(m, pd.DataFrame) and not m.empty:
        n_win_all = int((m["frac_beats"] == 1).sum())
        n_sig = int((m["combined_FDR"] < a).sum())
        add(f"  Polycomb sets evaluated                : {len(m)}")
        add(f"  Sets beating AntiPsy in ALL ancestries : {n_win_all}")
        add(f"  Sets sig. by combined FDR (<{a})       : {n_sig}")
        add("")
        add(f"  Top {TN} by consistency_score:")
        add(f"  {'Set':<34}{'NESmean':>9}{'NESsd':>8}{'dNESμ':>8}"
            f"{'frac_win':>9}{'combP':>10}{'FDR':>9}{'score':>8}")
        for _, r in m.head(TN).iterrows():
            add(f"  {str(r['Set1'])[:32]:<34}{f(r['NES1_mean']):>9}{f(r['NES1_std']):>8}"
                f"{f(r['dNES_mean']):>8}{f(r['frac_beats']):>9}"
                f"{f(r['combined_p_stouffer'],4):>10}{f(r['combined_FDR'],4):>9}"
                f"{f(r['consistency_score']):>8}{stars(r['combined_FDR'],a)}")
        core_hits = m[(m["frac_beats"] == 1) & (m["combined_FDR"] < a)]
        add("")
        if not core_hits.empty:
            add("  >> ROBUST WINNERS (win all ancestries AND combined FDR<{}):".format(a))
            for _, r in core_hits.iterrows():
                add(f"     - {r['Set1']}: meanNES={f(r['NES1_mean'])}, "
                    f"combined FDR={f(r['combined_FDR'],4)}")
        else:
            add("  >> No Polycomb set both wins in all ancestries and passes combined FDR.")
    else:
        add("  (MASTER_headtohead.csv not found or empty)")
    add("")

    # ---- A2 verdict ----
    add(bar); add("  [2]  CROSS-ANCESTRY VERDICT CONCORDANCE"); add(bar)
    conc = R.get("a2_conc", pd.DataFrame())
    if isinstance(conc, pd.DataFrame) and not conc.empty:
        for cls in ["WIN_ALL", "PARTIAL", "WIN_NONE"]:
            sets = conc[conc["class"] == cls].index.tolist()
            add(f"  {cls:<9} ({len(sets)}): "
                + (", ".join(map(str, sets[:20])) + (" ..." if len(sets) > 20 else "")
                   if sets else "none"))
        add("")
        add(f"  Detailed win counts (top {TN}):")
        add(f"  {'Set':<40}{'n_wins':>8}{'n_tested':>9}{'win_frac':>9}{'class':>10}")
        for s, r in conc.head(TN).iterrows():
            add(f"  {str(s)[:38]:<40}{int(r['n_wins']):>8}{int(r['n_tested']):>9}"
                f"{f(r['win_fraction']):>9}{str(r['class']):>10}")
    else:
        add("  (no verdict data)")
    add("")

    # ---- A3 shared genes ----
    add(bar); add("  [3]  SHARED vs ANCESTRY-SPECIFIC LEADING-EDGE GENES"); add(bar)
    rec = R.get("a3_rec", pd.DataFrame())
    core = R.get("a3_core", []); shared2 = R.get("a3_shared2", [])
    if isinstance(rec, pd.DataFrame) and not rec.empty:
        add(f"  Total unique leading-edge genes (Polycomb) : {len(rec)}")
        add(f"  Core (present in ALL {len(traits)} ancestries): {len(core)}")
        add(f"  Shared in exactly 2 ancestries             : {len(shared2)}")
        hub_core = [g for g in core if g in set(cfg['hub_genes'])]
        add(f"  Tracked hubs among CORE genes              : "
            f"{', '.join(hub_core) if hub_core else 'none'}")
        add("")
        add("  CORE genes (all ancestries):")
        for i in range(0, len(core), 12):
            add("    " + ", ".join(core[i:i+12]))
        add("")
        add(f"  Shared-in-2 genes (first {TN}):")
        add("    " + (", ".join(shared2[:TN]) if shared2 else "none"))
    else:
        add("  (no T3 gene-contribution data)")
    add("")

    # ---- A4 hubs ----
    add(bar); add("  [4]  TRANSANCESTRY HUB-GENE PRIORITIZATION"); add(bar)
    hub = R.get("a4_agg", pd.DataFrame())
    if isinstance(hub, pd.DataFrame) and not hub.empty:
        add(f"  {'Gene':<12}{'#anc_in_LE':>11}{'meanMax|Z|':>12}{'maxMax|Z|':>11}{'meanSignedZ':>13}")
        for _, r in hub.iterrows():
            add(f"  {str(r['gene']):<12}{int(r['n_anc_in_LE']):>11}"
                f"{f(r['mean_max_absZ']):>12}{f(r['max_max_absZ']):>11}"
                f"{f(r['mean_signedZ']):>13}")
        recur = hub[hub["n_anc_in_LE"] == len(traits)]["gene"].tolist()
        add("")
        add(f"  >> Hubs in leading edge of ALL ancestries: "
            f"{', '.join(recur) if recur else 'none'}")
    else:
        add("  (no hub data)")
    add("")

    # ---- A5 interaction ----
    add(bar); add("  [5]  ANCESTRY x SET INTERACTION ON |Z| (Kruskal-Wallis)"); add(bar)
    inter = R.get("a5", pd.DataFrame())
    if isinstance(inter, pd.DataFrame) and not inter.empty:
        n_sig = int((inter["kruskal_p"] < a).sum())
        add(f"  Sets testable                 : {len(inter)}")
        add(f"  Sets with sig ancestry diff   : {n_sig} (raw p<{a})")
        add("")
        medcols = [c for c in inter.columns if c.startswith("median_absZ_")]
        hdr = f"  {'Set':<34}{'H':>8}{'p':>10}{'FDR':>9}"
        for c in medcols: hdr += f"{c.replace('median_absZ_',''):>9}"
        add(hdr)
        for _, r in inter.head(TN).iterrows():
            line = (f"  {str(r['Set'])[:32]:<34}{f(r['kruskal_H']):>8}"
                    f"{f(r['kruskal_p'],4):>10}{f(r['kruskal_FDR'],4):>9}")
            for c in medcols: line += f"{f(r[c],2):>9}"
            add(line + stars(r['kruskal_p'], a))
        sig = inter[inter["kruskal_p"] < a]
        if not sig.empty:
            add("")
            add("  >> Sets with ancestry-specific |Z| magnitude:")
            for _, r in sig.iterrows():
                add(f"     - {r['Set']}: H={f(r['kruskal_H'])}, p={f(r['kruskal_p'],4)}"
                    f"{stars(r['kruskal_p'],a)}")
    else:
        add("  (insufficient per-set distributions)")
    add("")

    # ---- A6 drugs ----
    add(bar); add("  [6]  TRANSANCESTRY DRUG-REPURPOSING OVERLAP"); add(bar)
    drug = R.get("a6_agg", pd.DataFrame())
    if isinstance(drug, pd.DataFrame) and not drug.empty:
        in_all = drug[drug["n_ancestries"] == len(traits)]
        known_hits = drug[drug["is_known_psych_drug"]]
        add(f"  Distinct drugs hitting leading edge (any ancestry): {len(drug)}")
        add(f"  Drugs recurring in ALL {len(traits)} ancestries    : {len(in_all)}")
        add(f"  Known psych/BD drugs recovered                    : {len(known_hits)}")
        add("")
        add(f"  Top {TN} by recurrence & mean targets:")
        add(f"  {'Drug':<30}{'#anc':>6}{'totTgt':>8}{'meanTgt':>9}{'meanScore':>11}{'known':>7}")
        for _, r in drug.head(TN).iterrows():
            add(f"  {str(r['drug'])[:28]:<30}{int(r['n_ancestries']):>6}"
                f"{int(r['total_targets']):>8}{f(r['mean_targets'],2):>9}"
                f"{f(r['mean_score'],3):>11}{'YES' if r['is_known_psych_drug'] else '':>7}")
        if not in_all.empty:
            add("")
            add("  >> Cross-ancestry consistent candidates (all ancestries):")
            for _, r in in_all.head(TN).iterrows():
                tag = " [KNOWN]" if r["is_known_psych_drug"] else ""
                add(f"     - {r['drug']} (mean targets={f(r['mean_targets'],1)}){tag}")
        if not known_hits.empty:
            add("")
            add("  >> Known BD/psych drugs recovered from data:")
            for _, r in known_hits.iterrows():
                add(f"     - {r['drug']} in [{r['ancestries']}]")
    else:
        add("  (no DGIdb drug-ranking outputs found)")
    add("")

    # ---- A7 repression ----
    add(bar); add("  [7]  SIGNED-vs-ABS REPRESSION-MODEL CONSISTENCY"); add(bar)
    rep = R.get("a7_summ", pd.DataFrame())
    if isinstance(rep, pd.DataFrame) and not rep.empty:
        strong = rep[rep["repression_support"] == "STRONG"]["Set"].tolist()
        add(f"  Sets assessed                        : {len(rep)}")
        add(f"  STRONG repression support (neg-Z,   : {len(strong)}")
        add(f"        sig in >=2 ancestries)")
        add("")
        add(f"  {'Set':<34}{'n_anc':>6}{'%neg(μ)':>9}{'meanZ(μ)':>10}"
            f"{'n_sig_neg':>10}{'combP':>10}{'support':>10}")
        for _, r in rep.iterrows():
            add(f"  {str(r['Set'])[:32]:<34}{int(r['n_anc']):>6}"
                f"{f(r['mean_pct_neg'],1):>9}{f(r['mean_meanZ']):>10}"
                f"{int(r['n_sig_neg']):>10}{f(r['combined_sign_p'],4):>10}"
                f"{str(r['repression_support']):>10}")
        if strong:
            add("")
            add("  >> Sets supporting the Polycomb repression model across ancestries:")
            for s in strong:
                add(f"     - {s}")
    else:
        add("  (no T2 directionality data)")
    scmp = R.get("a7_scmp", pd.DataFrame())
    if isinstance(scmp, pd.DataFrame) and not scmp.empty:
        add("")
        add("  Signed-vs-abs NES (per ancestry, sign-flip = direction-driven signal):")
        cols = [c for c in ["ancestry","Trait","Set1","Set2","NES1_abs","NES1_signed",
                            "dNES_abs","dNES_signed"] if c in scmp.columns]
        add("  " + scmp[cols].head(40).to_string(index=False).replace("\n", "\n  "))
    add("")

    # ---- A8 enrichment ----
    add(bar); add("  [8]  BONUS: TRANSANCESTRY PATHWAY-ENRICHMENT CONSISTENCY"); add(bar)
    enr = R.get("a8", pd.DataFrame())
    if isinstance(enr, pd.DataFrame) and not enr.empty:
        multi = enr[enr["n_ancestries"] >= 2]
        add(f"  Significant terms (FDR<{a}) in >=1 ancestry : {len(enr)}")
        add(f"  Terms significant in >=2 ancestries         : {len(multi)}")
        add("")
        add(f"  Top {TN} recurrent terms:")
        add(f"  {'#anc':>5}  {'minAdjP':>10}  Term")
        for _, r in enr.head(TN).iterrows():
            add(f"  {int(r['n_ancestries']):>5}  {f(r['min_adjP'],2):>10}  "
                f"{str(r['Term'])[:60]}  [{r['ancestries']}]")
    else:
        add("  (Enrichr results not found across ancestries)")
    add("")

    # ---- HEADLINE ----
    add(bar); add("  TRANSANCESTRY HEADLINE TAKEAWAYS"); add(bar)
    try:
        if isinstance(m, pd.DataFrame) and not m.empty:
            ra = m[(m["frac_beats"] == 1)]
            add(f"  - {len(ra)} Polycomb set(s) beat AntiPsy in ALL ancestries; "
                f"top: {', '.join(map(str, ra['Set1'].head(5)))}"
                if not ra.empty else "  - No set wins in every ancestry.")
        if core:
            add(f"  - {len(core)} core leading-edge genes shared by all ancestries "
                f"(e.g. {', '.join(core[:8])}).")
        if isinstance(hub, pd.DataFrame) and not hub.empty:
            rec_h = hub[hub['n_anc_in_LE'] == len(traits)]['gene'].tolist()
            add(f"  - Hubs recovered in every ancestry: "
                f"{', '.join(rec_h) if rec_h else 'none'}.")
        if isinstance(drug, pd.DataFrame) and not drug.empty:
            ia = drug[drug['n_ancestries'] == len(traits)]['drug'].head(5).tolist()
            add(f"  - Consistent repurposing candidates: "
                f"{', '.join(map(str, ia)) if ia else 'none'}.")
        if isinstance(rep, pd.DataFrame) and not rep.empty:
            st = rep[rep['repression_support'] == 'STRONG']['Set'].tolist()
            add(f"  - Repression model strongly supported for: "
                f"{', '.join(st) if st else 'none'}.")
    except Exception as e:
        add(f"  (headline synthesis error: {e})")
    add(""); add(bar)

    summ = "\n".join(L)
    path = os.path.join(cfg["out"], "STAGE4_DETAILED_SUMMARY.txt")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(summ)
    print(f"\n[✓] Detailed summary → {path}")
    return summ


# ===========================================================================
#  MASTER DRIVER
# ===========================================================================
def run_stage4(cfg=CFG):
    os.makedirs(cfg["out"], exist_ok=True)
    print("=" * 72)
    print("  STAGE 4 — TRANSANCESTRY INTEGRATION")
    print(f"  Ancestries: {cfg['traits']}")
    print(f"  Output    : {cfg['out']}")
    print("=" * 72)

    print("\n[load] reading Stage 1-3 outputs ...")
    D = load_all(cfg)
    R = {}

    steps = [
        ("a1", "meta-NES & consistency",       lambda: a1_meta_nes(D, cfg)),
        ("a2", "verdict concordance",          lambda: a2_verdict(D, cfg)),
        ("a3", "shared leading-edge genes",    lambda: a3_shared_genes(D, cfg)),
        ("a4", "hub prioritization",           lambda: a4_hubs(D, cfg)),
        ("a5", "ancestry x set interaction",   lambda: a5_interaction(D, cfg)),
        ("a6", "drug-repurposing overlap",     lambda: a6_drugs(D, cfg)),
        ("a7", "repression-model consistency", lambda: a7_repression(D, cfg)),
        ("a8", "pathway enrichment (bonus)",   lambda: a8_enrichment(D, cfg)),
    ]

    for key, name, fn in steps:
        print(f"\n[{key}] {name} ...")
        try:
            res = fn()
            if key == "a2":
                R["a2_piv"], R["a2_conc"] = res
            elif key == "a3":
                R["a3_rec"], R["a3_core"], R["a3_shared2"], R["a3_peranc"] = res
            elif key == "a4":
                R["a4_long"], R["a4_agg"] = res
            elif key == "a6":
                R["a6_long"], R["a6_agg"] = res
            elif key == "a7":
                R["a7_dir"], R["a7_summ"], R["a7_scmp"] = res
            else:
                R[key] = res
            print(f"    done.")
        except Exception as e:
            print(f"    [ERROR] {name} failed: {e}")

    # detailed summary
    print("\n[summary] building detailed report ...")
    R["summary_text"] = build_summary(cfg, R)

    # manifest
    try:
        with open(os.path.join(cfg["out"], "stage4_manifest.json"), "w") as fh:
            json.dump({"generated_at": datetime.now().isoformat(timespec="seconds"),
                       "traits": cfg["traits"],
                       "outputs": sorted(os.listdir(cfg["out"]))}, fh, indent=2)
    except Exception as e:
        print(f"    [warn] manifest failed: {e}")

    print("\n" + "=" * 72)
    print(f"  STAGE 4 COMPLETE → {os.path.abspath(cfg['out'])}")
    print("=" * 72)
    print("\n" + R["summary_text"])
    return R


# ---------------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------------
stage4_results = run_stage4(CFG)

  STAGE 4 — TRANSANCESTRY INTEGRATION
  Ancestries: ['bip_afr', 'bip_eas', 'bip_eur']
  Output    : /content/096AvAp3_stage4

[load] reading Stage 1-3 outputs ...

[a1] meta-NES & consistency ...
    done.

[a2] verdict concordance ...
    done.

[a3] shared leading-edge genes ...
    done.

[a4] hub prioritization ...
    done.

[a5] ancestry x set interaction ...
    done.

[a6] drug-repurposing overlap ...
    done.

[a7] repression-model consistency ...
    done.

[a8] pathway enrichment (bonus) ...
    done.

[summary] building detailed report ...

[✓] Detailed summary → /content/096AvAp3_stage4/STAGE4_DETAILED_SUMMARY.txt

  STAGE 4 COMPLETE → /content/096AvAp3_stage4

####################################################################################
#   096AvAp3 — STAGE 4 TRANSANCESTRY INTEGRATIVE SUMMARY
#   Generated : 2026-07-04 15:35:30
#   Ancestries: bip_afr, bip_eas, bip_eur
#   Contrast  : Polycomb (group1) vs AntiPsy (group2)
#   Sig codes : *** p<0.001  ** p<0.01  * 

# Stage 5

In [19]:
# ============================================================================
#  096AvAp3 — STAGE 5  TRANSANCESTRY DIFFERENTIAL PIPELINE
#  Consumes ONLY Stage 1-4 outputs.  No raw TWAS re-processing.
#  Focus: DIVERGENCE (ancestry-specific signals), esp. EUR vs EAS/AFR.
#  Analyses:
#    [1] Ancestry-specific |Z| differential ranking (EUR vs non-EUR)
#    [2] Ancestry-unique leading-edge genes (EUR-only / EAS-only / AFR-only)
#    [3] Differential hub influence across ancestries
#    [4] Ancestry-stratified drug prioritization
#    [5] Differential repression signature (per-ancestry neg-Z skew)
#    [6] EUR-dominant vs shared set ranking
#    [7] Cross-ancestry dNES heterogeneity (variance decomposition)
#    [A] ACTIONABLE: core-gene validation prep (core list + universal hubs)
#    [B] ACTIONABLE: GABA / Glutamatergic / Ca subnetwork extraction
#  Ends with one very detailed STAGE5_DETAILED_SUMMARY.txt
# ============================================================================
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from itertools import combinations
warnings.filterwarnings("ignore")


# ===========================================================================
#  CONFIG  — edit paths / labels here if yours differ
# ===========================================================================
CFG = dict(
    traits        = ["bip_afr", "bip_eas", "bip_eur"],
    primary       = "bip_eur",                    # "advantage" reference ancestry
    compare_dir   = "/content/096AvAp3_compare",
    posthoc_tmpl  = "/content/096AvAp3_posthoc_{trait}",
    followup_tmpl = "/content/096AvAp3_followups_{trait}",
    stage4_dir    = "/content/096AvAp3_stage4",
    out           = "/content/096AvAp3_stage5",
    polycomb_group_names = ["polycomb", "ageing_polycomb"],
    antipsy_group_names  = ["AntiPsy", "AntiPsy_Genes"],
    hub_genes = ["GABRA6","RPTOR","MAPK3","GABRB2","GNB3","DRD2","GSK3B","ANK3",
                 "CYP2D6","CHRM5","HTR6","CACNA1B","MAPK1","SF3B1","FYN","LIMK1",
                 "CTNNB1","GLS2"],
    known_drugs = ["lithium","valproate","valproic acid","lamotrigine","quetiapine",
                   "olanzapine","carbamazepine","fluoxetine","risperidone",
                   "aripiprazole","clozapine","haloperidol","lithium carbonate"],
    # gene-family prefixes for the CGR GABA / Glutamatergic / Calcium subnetwork
    gaba_glut_ca_prefixes = ["GABR","GRIN","GRIA","GRIK","GRM","GAD","SLC1A",
                             "SLC17A","SLC32A","SLC38","CACNA","CACNB","CACNG",
                             "GLS","GLUL","DLG","SHANK","HOMER","NRXN","NLGN"],
    eur_zthresh = 5.0,        # |Z| threshold to call a hub "EUR-high"
    dnes_dom    = 0.15,       # EUR-dominant dNES cutoff
    alpha = 0.05,
    top_n = 30,
)


# ===========================================================================
#  small utilities
# ===========================================================================
def _read(path):
    try:
        if os.path.isfile(path):
            return pd.read_csv(path)
    except Exception as e:
        print(f"    [warn] read failed {path}: {e}")
    return pd.DataFrame()

def f(x, nd=3):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)): return "NA"
        if isinstance(x, (int, np.integer)): return str(int(x))
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def stars(p, alpha=0.05):
    if p != p or p is None: return ""
    if p < 0.001: return " ***"
    if p < 0.01:  return " **"
    if p < alpha: return " *"
    return ""

def _nonprimary(cfg):
    return [t for t in cfg["traits"] if t != cfg["primary"]]


# ===========================================================================
#  LOADERS — pull Stage 4 + per-trait tables
# ===========================================================================
def load_all(cfg):
    traits = cfg["traits"]
    s4 = cfg["stage4_dir"]
    D = dict(
        master        = _read(os.path.join(cfg["compare_dir"], "MASTER_headtohead.csv")),
        s4_interaction= _read(os.path.join(s4, "stage4_ancestry_set_interaction.csv")),
        s4_recurrence = _read(os.path.join(s4, "stage4_leading_edge_recurrence.csv")),
        s4_hub_long   = _read(os.path.join(s4, "stage4_hub_long.csv")),
        s4_hub_agg    = _read(os.path.join(s4, "stage4_hub_prioritization.csv")),
        s4_drug       = _read(os.path.join(s4, "stage4_drug_recurrence.csv")),
        s4_repression = _read(os.path.join(s4, "stage4_repression_model.csv")),
        s4_meta_nes   = _read(os.path.join(s4, "stage4_meta_nes_consistency.csv")),
        s4_dir_long   = _read(os.path.join(s4, "stage4_directionality_long.csv")),
        T2={}, T3={}, T6={}, drugs={}, influence={},
    )
    for t in traits:
        pd_dir = cfg["posthoc_tmpl"].format(trait=t)
        fu_dir = cfg["followup_tmpl"].format(trait=t)
        D["T2"][t] = _read(os.path.join(pd_dir, "T2_directionality.csv"))
        D["T3"][t] = _read(os.path.join(pd_dir, "T3_gene_contributions.csv"))
        D["T6"][t] = _read(os.path.join(pd_dir, "T6_Zdist_comparison.csv"))
        D["drugs"][t] = _read(os.path.join(fu_dir, "05_drugs", "dgidb_drug_ranking.csv"))
        inf = pd.DataFrame()
        for cand in (glob.glob(os.path.join(pd_dir, "*influence*.csv")) +
                     glob.glob(os.path.join(pd_dir, "**", "gene_influence*.csv"),
                               recursive=True)):
            inf = _read(cand)
            if not inf.empty: break
        D["influence"][t] = inf
    return D


# ===========================================================================
#  ANALYSIS 1 — ancestry-specific |Z| differential ranking (EUR vs non-EUR)
# ===========================================================================
def a1_zdiff(D, cfg):
    inter = D["s4_interaction"].copy()
    prim = cfg["primary"]; others = _nonprimary(cfg)
    col_p = f"median_absZ_{prim}"
    cols_o = [f"median_absZ_{t}" for t in others]
    if inter.empty or col_p not in inter.columns:
        # fall back: build from per-trait T3 medians
        rows = {}
        for t in cfg["traits"]:
            t3 = D["T3"].get(t, pd.DataFrame())
            if t3.empty: continue
            poly = (t3[t3["Group"].astype(str).isin(cfg["polycomb_group_names"])]
                    if "Group" in t3.columns else t3)
            med = poly.groupby("Set")["absZ"].median()
            for s, v in med.items():
                rows.setdefault(s, {})[f"median_absZ_{t}"] = round(float(v), 4)
        inter = pd.DataFrame([dict(Set=s, **v) for s, v in rows.items()])
        if inter.empty or col_p not in inter.columns:
            return pd.DataFrame()

    have_o = [c for c in cols_o if c in inter.columns]
    inter["nonEUR_mean_absZ"] = inter[have_o].mean(axis=1) if have_o else np.nan
    inter["eur_advantage"] = inter[col_p] - inter["nonEUR_mean_absZ"]
    keep = ["Set"] + [c for c in ["kruskal_H","kruskal_p","kruskal_FDR"] if c in inter.columns] \
           + [col_p] + have_o + ["nonEUR_mean_absZ", "eur_advantage"]
    out = inter[keep].sort_values("eur_advantage", ascending=False).reset_index(drop=True)
    out.to_csv(os.path.join(cfg["out"], "stage5_eur_advantage_ranking.csv"), index=False)

    try:
        top = out.head(20).iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(top) + 1)))
        colors = ["#c0392b" if v > 0 else "#2980b9" for v in top["eur_advantage"]]
        ax.barh(top["Set"].astype(str), top["eur_advantage"], color=colors,
                edgecolor="black", lw=0.3)
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel(f"EUR advantage in median |Z|  ({prim} − mean({','.join(others)}))")
        ax.set_title("Ancestry-specific |Z| differential (Polycomb sets)")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage5_eur_advantage.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A1 plot failed: {e}")
    return out


# ===========================================================================
#  ANALYSIS 2 — ancestry-unique leading-edge genes
# ===========================================================================
def a2_unique_genes(D, cfg):
    rec = D["s4_recurrence"].copy()
    if rec.empty:
        return pd.DataFrame(), {}
    hubset = set(cfg["hub_genes"])
    rec["is_hub"] = rec["gene"].astype(str).str.upper().isin(hubset)
    uniq = rec[rec["n_ancestries"] == 1].copy()
    per_anc = {}
    for t in cfg["traits"]:
        g = uniq[uniq["ancestries"].astype(str) == t]["gene"].tolist()
        per_anc[t] = g
        with open(os.path.join(cfg["out"], f"stage5_{t}_only_LE_genes.txt"), "w") as fh:
            fh.write("\n".join(g))
    uniq.to_csv(os.path.join(cfg["out"], "stage5_ancestry_unique_genes.csv"), index=False)

    try:
        counts = {t: len(per_anc[t]) for t in cfg["traits"]}
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(list(counts.keys()), list(counts.values()),
               color="slateblue", edgecolor="black")
        for i, (k, v) in enumerate(counts.items()):
            ax.text(i, v, str(v), ha="center", va="bottom")
        ax.set_ylabel("# ancestry-UNIQUE leading-edge genes")
        ax.set_title("Ancestry-specific (unique) leading-edge genes")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage5_unique_gene_counts.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A2 plot failed: {e}")
    return uniq, per_anc


# ===========================================================================
#  ANALYSIS 3 — differential hub influence across ancestries
# ===========================================================================
def a3_hub_diff(D, cfg):
    hl = D["s4_hub_long"].copy()
    prim = cfg["primary"]; others = _nonprimary(cfg)
    if hl.empty:
        return pd.DataFrame(), pd.DataFrame()

    piv = hl.pivot_table(index="gene", columns="ancestry", values="max_absZ")
    for t in cfg["traits"]:
        if t not in piv.columns: piv[t] = np.nan
    piv["nonEUR_mean"] = piv[others].mean(axis=1)
    piv["eur_advantage"] = piv[prim] - piv["nonEUR_mean"]
    piv["max_range"] = piv[cfg["traits"]].max(axis=1) - piv[cfg["traits"]].min(axis=1)
    piv = piv.sort_values("eur_advantage", ascending=False)
    piv.to_csv(os.path.join(cfg["out"], "stage5_hub_differential.csv"))

    # flag EUR-high hubs
    hl["is_eur_high"] = ((hl["ancestry"] == prim) &
                         (pd.to_numeric(hl["max_absZ"], errors="coerce") > cfg["eur_zthresh"]))
    eur_high = hl[hl["is_eur_high"]][["gene", "max_absZ"]].sort_values(
        "max_absZ", ascending=False)
    eur_high.to_csv(os.path.join(cfg["out"], "stage5_hub_eur_high.csv"), index=False)

    try:
        pm = piv[cfg["traits"]].astype(float)
        fig, ax = plt.subplots(figsize=(1.4 * len(cfg["traits"]) + 3,
                                        max(4, 0.35 * len(pm) + 1)))
        sns.heatmap(pm, cmap="magma", annot=True, fmt=".2f",
                    linewidths=0.5, linecolor="white", ax=ax,
                    cbar_kws={"label": "max |Z|"})
        ax.set_title("Hub-gene max |Z| by ancestry (differential)")
        plt.tight_layout()
        fig.savefig(os.path.join(cfg["out"], "stage5_hub_differential_heatmap.png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"    [warn] A3 plot failed: {e}")
    return piv.reset_index(), eur_high


# ===========================================================================
#  ANALYSIS 4 — ancestry-stratified drug prioritization
# ===========================================================================
def a4_drug_strat(D, cfg):
    prim = cfg["primary"]; others = _nonprimary(cfg)
    frames = []
    for t in cfg["traits"]:
        dr = D["drugs"].get(t, pd.DataFrame())
        if dr.empty: continue
        dr = dr[["drug", "n_targets"]].copy()
        dr = dr.rename(columns={"n_targets": f"n_targets_{t}"})
        dr["drug_l"] = dr["drug"].astype(str).str.lower()
        frames.append(dr.set_index("drug_l")[[f"n_targets_{t}"]])
    if not frames:
        return pd.DataFrame()
    wide = pd.concat(frames, axis=1).fillna(0)
    # bring back a display name
    names = {}
    for t in cfg["traits"]:
        dr = D["drugs"].get(t, pd.DataFrame())
        if not dr.empty:
            for _, r in dr.iterrows():
                names.setdefault(str(r["drug"]).lower(), r["drug"])
    wide["drug"] = [names.get(i, i) for i in wide.index]
    tcols = [f"n_targets_{t}" for t in cfg["traits"] if f"n_targets_{t}" in wide.columns]
    ocols = [f"n_targets_{t}" for t in others if f"n_targets_{t}" in wide.columns]
    pcol  = f"n_targets_{prim}"
    if pcol in wide.columns:
        wide["eur_targets_advantage"] = wide[pcol] - (wide[ocols].mean(axis=1) if ocols else 0)
    else:
        wide["eur_targets_advantage"] = np.nan
    known = set(x.lower() for x in cfg["known_drugs"])
    wide["is_known_psych_drug"] = wide["drug"].astype(str).str.lower().isin(known)
    out = wide.reset_index(drop=True)[["drug"] + tcols +
                                      ["eur_targets_advantage", "is_known_psych_drug"]]
    out = out.sort_values("eur_targets_advantage", ascending=False).reset_index(drop=True)
    out.to_csv(os.path.join(cfg["out"], "stage5_drug_ancestry_stratified.csv"), index=False)
    return out


# ===========================================================================
#  ANALYSIS 5 — differential repression signature (per-ancestry neg-Z skew)
# ===========================================================================
def a5_repression_diff(D, cfg):
    rows = []
    for t in cfg["traits"]:
        t2 = D["T2"].get(t, pd.DataFrame())
        if t2.empty: continue
        poly = (t2[t2["Group"].astype(str).isin(cfg["polycomb_group_names"])]
                if "Group" in t2.columns else t2)
        pn = pd.to_numeric(poly.get("pct_neg"), errors="coerce")
        mz = pd.to_numeric(poly.get("mean_Z"), errors="coerce")
        sp = pd.to_numeric(poly.get("sign_test_p"), errors="coerce")
        rows.append(dict(
            ancestry=t,
            n_sets=len(poly),
            mean_pct_neg=round(float(pn.mean()), 2) if pn.notna().any() else np.nan,
            median_pct_neg=round(float(pn.median()), 2) if pn.notna().any() else np.nan,
            mean_meanZ=round(float(mz.mean()), 4) if mz.notna().any() else np.nan,
            n_sig_neg=int(((sp < cfg["alpha"]) & (mz < 0)).sum()),
            n_sig_pos=int(((sp < cfg["alpha"]) & (mz > 0)).sum()),
        ))
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(os.path.join(cfg["out"], "stage5_repression_differential.csv"), index=False)
    # per-set ancestry contrast of pct_neg
    setrows = []
    for t in cfg["traits"]:
        t2 = D["T2"].get(t, pd.DataFrame())
        if t2.empty: continue
        poly = (t2[t2["Group"].astype(str).isin(cfg["polycomb_group_names"])]
                if "Group" in t2.columns else t2)
        for _, r in poly.iterrows():
            setrows.append(dict(Set=r.get("Set"), ancestry=t,
                                pct_neg=r.get("pct_neg"), mean_Z=r.get("mean_Z")))
    setdf = pd.DataFrame(setrows)
    contrast = pd.DataFrame()
    if not setdf.empty:
        contrast = setdf.pivot_table(index="Set", columns="ancestry", values="pct_neg")
        prim = cfg["primary"]; others = _nonprimary(cfg)
        have_o = [t for t in others if t in contrast.columns]
        if prim in contrast.columns and have_o:
            contrast["eur_pctneg_advantage"] = contrast[prim] - contrast[have_o].mean(axis=1)
            contrast = contrast.sort_values("eur_pctneg_advantage", ascending=False)
        contrast.to_csv(os.path.join(cfg["out"], "stage5_repression_pctneg_contrast.csv"))
    return out, contrast


# ===========================================================================
#  ANALYSIS 6 — EUR-dominant vs shared set ranking
# ===========================================================================
def a6_eur_dominant(D, cfg):
    m = D["master"].copy()
    if m.empty: return pd.DataFrame()
    grp_ok = (m["Group1"].astype(str).isin(cfg["polycomb_group_names"])
              if "Group1" in m.columns else pd.Series(True, index=m.index))
    m = m[grp_ok].copy()
    m["NES1"] = pd.to_numeric(m.get("NES1"), errors="coerce")
    m["dNES_1_vs_2"] = pd.to_numeric(m.get("dNES_1_vs_2"), errors="coerce")
    m["GSEA_p1"] = pd.to_numeric(m.get("GSEA_p1"), errors="coerce")
    prim = cfg["primary"]; others = _nonprimary(cfg)

    piv_d = m.pivot_table(index="Set1", columns="Trait", values="dNES_1_vs_2", aggfunc="mean")
    piv_p = m.pivot_table(index="Set1", columns="Trait", values="GSEA_p1", aggfunc="min")
    for t in cfg["traits"]:
        if t not in piv_d.columns: piv_d[t] = np.nan
        if t not in piv_p.columns: piv_p[t] = np.nan

    rows = []
    for s in piv_d.index:
        d_eur = piv_d.loc[s, prim]
        d_oth = piv_d.loc[s, others].mean()
        p_eur = piv_p.loc[s, prim]
        eur_dominant = (pd.notna(d_eur) and d_eur > cfg["dnes_dom"] and
                        pd.notna(p_eur) and p_eur < cfg["alpha"] and
                        (pd.isna(d_oth) or d_oth <= cfg["dnes_dom"]))
        rows.append(dict(
            Set1=s,
            dNES_eur=round(float(d_eur), 4) if pd.notna(d_eur) else np.nan,
            dNES_nonEUR_mean=round(float(d_oth), 4) if pd.notna(d_oth) else np.nan,
            eur_dNES_advantage=(round(float(d_eur - d_oth), 4)
                                if pd.notna(d_eur) and pd.notna(d_oth) else np.nan),
            GSEA_p1_eur=round(float(p_eur), 6) if pd.notna(p_eur) else np.nan,
            EUR_DOMINANT=eur_dominant,
        ))
    out = pd.DataFrame(rows).sort_values("eur_dNES_advantage",
                                         ascending=False).reset_index(drop=True)
    out.to_csv(os.path.join(cfg["out"], "stage5_eur_dominant_sets.csv"), index=False)
    return out


# ===========================================================================
#  ANALYSIS 7 — cross-ancestry dNES heterogeneity
# ===========================================================================
def a7_heterogeneity(D, cfg):
    m = D["master"].copy()
    if m.empty: return pd.DataFrame()
    grp_ok = (m["Group1"].astype(str).isin(cfg["polycomb_group_names"])
              if "Group1" in m.columns else pd.Series(True, index=m.index))
    m = m[grp_ok].copy()
    m["dNES_1_vs_2"] = pd.to_numeric(m.get("dNES_1_vs_2"), errors="coerce")
    rows = []
    for (s1, s2), sub in m.groupby(["Set1", "Set2"]):
        d = sub.set_index("Trait")["dNES_1_vs_2"]
        vals = d.reindex(cfg["traits"])
        rng = (vals.max() - vals.min()) if vals.notna().sum() >= 2 else np.nan
        rows.append(dict(
            Set1=s1, Set2=s2, n_anc=int(vals.notna().sum()),
            dNES_mean=round(float(vals.mean()), 4) if vals.notna().any() else np.nan,
            dNES_std=round(float(vals.std(ddof=0)), 4) if vals.notna().sum() > 1 else np.nan,
            dNES_range=round(float(rng), 4) if rng == rng else np.nan,
            **{f"dNES_{t}": (round(float(vals[t]), 4) if t in vals and pd.notna(vals[t]) else np.nan)
               for t in cfg["traits"]},
        ))
    out = pd.DataFrame(rows).sort_values("dNES_range", ascending=False).reset_index(drop=True)
    out.to_csv(os.path.join(cfg["out"], "stage5_dnes_heterogeneity.csv"), index=False)
    return out


# ===========================================================================
#  ACTIONABLE A — core-gene validation prep (core list + universal hubs)
# ===========================================================================
def actA_core(D, cfg):
    rec = D["s4_recurrence"].copy()
    if rec.empty:
        return [], [], pd.DataFrame()
    n_all = len(cfg["traits"])
    core = rec[rec["n_ancestries"] == n_all]["gene"].astype(str).str.upper().tolist()
    hubset = set(cfg["hub_genes"])
    core_hubs = [g for g in core if g in hubset]
    with open(os.path.join(cfg["out"], "stage5_core_genes_all_ancestries.txt"), "w") as fh:
        fh.write("\n".join(core))
    with open(os.path.join(cfg["out"], "stage5_universal_hubs.txt"), "w") as fh:
        fh.write("\n".join(core_hubs))

    # annotate core genes with per-ancestry max|Z| (from T3) for a validation table
    rows = []
    per_anc_maxabs = {}
    for t in cfg["traits"]:
        t3 = D["T3"].get(t, pd.DataFrame())
        if t3.empty: continue
        poly = (t3[t3["Group"].astype(str).isin(cfg["polycomb_group_names"])]
                if "Group" in t3.columns else t3)
        g = poly.copy()
        g["gene_u"] = g["gene"].astype(str).str.upper()
        per_anc_maxabs[t] = g.groupby("gene_u")["absZ"].max()
    for gsym in core:
        row = {"gene": gsym, "is_hub": gsym in hubset}
        vals = []
        for t in cfg["traits"]:
            v = per_anc_maxabs.get(t, pd.Series(dtype=float)).get(gsym, np.nan)
            row[f"maxabs_{t}"] = round(float(v), 4) if v == v else np.nan
            if v == v: vals.append(v)
        row["mean_maxabs"] = round(float(np.mean(vals)), 4) if vals else np.nan
        rows.append(row)
    core_tbl = pd.DataFrame(rows).sort_values("mean_maxabs", ascending=False).reset_index(drop=True)
    core_tbl.to_csv(os.path.join(cfg["out"], "stage5_core_gene_validation_table.csv"), index=False)
    return core, core_hubs, core_tbl


# ===========================================================================
#  ACTIONABLE B — GABA / Glutamatergic / Ca subnetwork extraction
# ===========================================================================
def actB_gabaglut(core, core_tbl, cfg):
    prefixes = tuple(cfg["gaba_glut_ca_prefixes"])
    subnet = [g for g in core if str(g).upper().startswith(prefixes)]
    with open(os.path.join(cfg["out"], "stage5_gaba_glut_ca_subnetwork.txt"), "w") as fh:
        fh.write("\n".join(subnet))
    sub_tbl = pd.DataFrame()
    if isinstance(core_tbl, pd.DataFrame) and not core_tbl.empty:
        sub_tbl = core_tbl[core_tbl["gene"].isin(subnet)].copy()
        # tag family
        def _fam(g):
            g = str(g).upper()
            if g.startswith(("GABR","GAD","SLC32A")): return "GABAergic"
            if g.startswith(("GRIN","GRIA","GRIK","GRM","SLC1A","SLC17A","GLS","GLUL",
                             "SLC38")): return "Glutamatergic"
            if g.startswith(("CACNA","CACNB","CACNG")): return "Calcium_channel"
            if g.startswith(("DLG","SHANK","HOMER","NRXN","NLGN")): return "Synaptic_scaffold"
            return "Other"
        sub_tbl["family"] = sub_tbl["gene"].map(_fam)
        sub_tbl = sub_tbl.sort_values(["family", "mean_maxabs"],
                                      ascending=[True, False]).reset_index(drop=True)
        sub_tbl.to_csv(os.path.join(cfg["out"],
                       "stage5_gaba_glut_ca_table.csv"), index=False)
        try:
            if not sub_tbl.empty:
                mcols = [c for c in sub_tbl.columns if c.startswith("maxabs_")]
                pm = sub_tbl.set_index("gene")[mcols].astype(float)
                fig, ax = plt.subplots(figsize=(1.4 * len(mcols) + 3,
                                                max(4, 0.35 * len(pm) + 1)))
                sns.heatmap(pm, cmap="YlGnBu", annot=True, fmt=".2f",
                            linewidths=0.5, linecolor="white", ax=ax,
                            cbar_kws={"label": "max |Z|"})
                ax.set_title("CGR GABA/Glut/Ca core subnetwork — max |Z| by ancestry")
                plt.tight_layout()
                fig.savefig(os.path.join(cfg["out"],
                            "stage5_gaba_glut_ca_heatmap.png"), dpi=150)
                plt.close(fig)
        except Exception as e:
            print(f"    [warn] actB plot failed: {e}")
    return subnet, sub_tbl


# ===========================================================================
#  DETAILED SUMMARY BUILDER
# ===========================================================================
def build_summary(cfg, R):
    a = cfg["alpha"]; traits = cfg["traits"]; TN = cfg["top_n"]
    prim = cfg["primary"]; others = _nonprimary(cfg)
    L = []; add = L.append
    bar = "=" * 84

    add("#" * 84)
    add("#   096AvAp3 — STAGE 5 TRANSANCESTRY DIFFERENTIAL SUMMARY")
    add(f"#   Generated  : {datetime.now():%Y-%m-%d %H:%M:%S}")
    add(f"#   Ancestries : {', '.join(traits)}   (primary/reference = {prim})")
    add(f"#   Contrast   : Polycomb (group1) vs AntiPsy (group2)")
    add(f"#   Focus      : DIVERGENCE — ancestry-specific / EUR-vs-nonEUR effects")
    add(f"#   Sig codes  : *** p<0.001  ** p<0.01  * p<{a}")
    add("#" * 84); add("")

    # ---- A1 ----
    add(bar); add(f"  [1]  ANCESTRY-SPECIFIC |Z| DIFFERENTIAL ({prim} vs mean(nonEUR))"); add(bar)
    d1 = R.get("a1", pd.DataFrame())
    if isinstance(d1, pd.DataFrame) and not d1.empty:
        n_pos = int((d1["eur_advantage"] > 0).sum())
        add(f"  Polycomb sets ranked            : {len(d1)}")
        add(f"  Sets with EUR |Z| advantage > 0 : {n_pos}")
        add("")
        medcols = [c for c in d1.columns if c.startswith("median_absZ_")]
        hdr = f"  {'Set':<32}"
        for c in medcols: hdr += f"{c.replace('median_absZ_',''):>9}"
        hdr += f"{'nonEURm':>9}{'EURadv':>9}"
        if "kruskal_p" in d1.columns: hdr += f"{'krusk_p':>9}"
        add(hdr)
        for _, r in d1.head(TN).iterrows():
            line = f"  {str(r['Set'])[:30]:<32}"
            for c in medcols: line += f"{f(r[c],2):>9}"
            line += f"{f(r['nonEUR_mean_absZ'],2):>9}{f(r['eur_advantage'],2):>9}"
            if "kruskal_p" in d1.columns:
                line += f"{f(r.get('kruskal_p'),4):>9}{stars(r.get('kruskal_p'),a)}"
            add(line)
        add("")
        topadv = d1.head(5)
        add("  >> Strongest EUR-advantaged sets:")
        for _, r in topadv.iterrows():
            add(f"     - {r['Set']}: EUR adv={f(r['eur_advantage'],3)} "
                f"(EUR={f(r.get(f'median_absZ_{prim}'),2)}, "
                f"nonEUR={f(r['nonEUR_mean_absZ'],2)})")
        botadv = d1.tail(3)
        add("  >> Strongest nonEUR-advantaged sets:")
        for _, r in botadv.iterrows():
            add(f"     - {r['Set']}: EUR adv={f(r['eur_advantage'],3)}")
    else:
        add("  (no interaction / T3 median data available)")
    add("")

    # ---- A2 ----
    add(bar); add("  [2]  ANCESTRY-UNIQUE LEADING-EDGE GENES"); add(bar)
    uniq = R.get("a2_uniq", pd.DataFrame()); per_anc = R.get("a2_peranc", {})
    if isinstance(uniq, pd.DataFrame) and not uniq.empty:
        add(f"  Total ancestry-UNIQUE genes (appear in exactly 1 ancestry): {len(uniq)}")
        add("")
        for t in traits:
            g = per_anc.get(t, [])
            hubs_here = [x for x in g if x in set(cfg["hub_genes"])]
            add(f"  {t}-ONLY ({len(g)} genes; hubs: "
                f"{', '.join(hubs_here) if hubs_here else 'none'}):")
            for i in range(0, min(len(g), 60), 12):
                add("    " + ", ".join(g[i:i+12]))
            if len(g) > 60:
                add(f"    ... (+{len(g)-60} more; full list in "
                    f"stage5_{t}_only_LE_genes.txt)")
            add("")
        prim_only = per_anc.get(prim, [])
        add(f"  >> {prim}-specific biology flagged for EUR-powered follow-up: "
            f"{len(prim_only)} genes.")
    else:
        add("  (no recurrence data)")
    add("")

    # ---- A3 ----
    add(bar); add("  [3]  DIFFERENTIAL HUB INFLUENCE ACROSS ANCESTRIES"); add(bar)
    hub = R.get("a3_piv", pd.DataFrame()); eur_high = R.get("a3_eurhigh", pd.DataFrame())
    if isinstance(hub, pd.DataFrame) and not hub.empty:
        tcols = [t for t in traits if t in hub.columns]
        add(f"  {'Gene':<12}" + "".join(f"{t:>10}" for t in tcols)
            + f"{'nonEURm':>10}{'EURadv':>9}{'range':>8}")
        for _, r in hub.iterrows():
            line = f"  {str(r['gene']):<12}"
            for t in tcols: line += f"{f(r[t],2):>10}"
            line += f"{f(r.get('nonEUR_mean'),2):>10}{f(r.get('eur_advantage'),2):>9}" \
                    f"{f(r.get('max_range'),2):>8}"
            add(line)
        add("")
        if isinstance(eur_high, pd.DataFrame) and not eur_high.empty:
            add(f"  >> EUR-HIGH hubs (|Z| > {cfg['eur_zthresh']} in {prim}):")
            for _, r in eur_high.iterrows():
                add(f"     - {r['gene']}: max|Z|={f(r['max_absZ'],3)}")
        else:
            add(f"  >> No hub exceeds |Z|>{cfg['eur_zthresh']} specifically in {prim}.")
        # most divergent hubs
        div = hub.sort_values("max_range", ascending=False).head(5)
        add("")
        add("  >> Most cross-ancestry-divergent hubs (by |Z| range):")
        for _, r in div.iterrows():
            add(f"     - {r['gene']}: range={f(r.get('max_range'),3)}")
    else:
        add("  (no hub_long data)")
    add("")

    # ---- A4 ----
    add(bar); add("  [4]  ANCESTRY-STRATIFIED DRUG PRIORITIZATION"); add(bar)
    dr = R.get("a4", pd.DataFrame())
    if isinstance(dr, pd.DataFrame) and not dr.empty:
        tcols = [c for c in dr.columns if c.startswith("n_targets_")]
        add(f"  Drugs compared across ancestries: {len(dr)}")
        n_known = int(dr["is_known_psych_drug"].sum())
        add(f"  Known psych/BD drugs in list    : {n_known}")
        add("")
        add(f"  Top {TN} by EUR target advantage:")
        add(f"  {'Drug':<28}" + "".join(f"{c.replace('n_targets_',''):>9}" for c in tcols)
            + f"{'EURadv':>9}{'known':>7}")
        for _, r in dr.head(TN).iterrows():
            line = f"  {str(r['drug'])[:26]:<28}"
            for c in tcols: line += f"{f(r[c],0):>9}"
            line += f"{f(r.get('eur_targets_advantage'),1):>9}" \
                    f"{'YES' if r.get('is_known_psych_drug') else '':>7}"
            add(line)
        eur_fav = dr[dr["eur_targets_advantage"] > 0]
        add("")
        add(f"  >> {len(eur_fav)} drug(s) hit more leading-edge targets in {prim}.")
        known_eur = eur_fav[eur_fav["is_known_psych_drug"]]["drug"].tolist()
        if known_eur:
            add(f"     Known psych drugs among EUR-favored: {', '.join(map(str, known_eur))}")
    else:
        add("  (no per-ancestry drug rankings found)")
    add("")

    # ---- A5 ----
    add(bar); add("  [5]  DIFFERENTIAL REPRESSION SIGNATURE (neg-Z skew by ancestry)"); add(bar)
    rep = R.get("a5", pd.DataFrame()); contrast = R.get("a5_contrast", pd.DataFrame())
    if isinstance(rep, pd.DataFrame) and not rep.empty:
        add(f"  {'Ancestry':<10}{'n_sets':>8}{'mean%neg':>10}{'med%neg':>9}"
            f"{'meanZ':>9}{'sigNeg':>8}{'sigPos':>8}")
        for _, r in rep.iterrows():
            add(f"  {str(r['ancestry']):<10}{int(r['n_sets']):>8}"
                f"{f(r['mean_pct_neg'],1):>10}{f(r['median_pct_neg'],1):>9}"
                f"{f(r['mean_meanZ']):>9}{int(r['n_sig_neg']):>8}{int(r['n_sig_pos']):>8}")
        # which ancestry most repressive
        try:
            most = rep.loc[rep["mean_pct_neg"].astype(float).idxmax()]
            add("")
            add(f"  >> Strongest overall repression (neg-Z) skew: {most['ancestry']} "
                f"(mean %neg={f(most['mean_pct_neg'],1)}, "
                f"{int(most['n_sig_neg'])} sig-neg sets).")
        except Exception:
            pass
    else:
        add("  (no T2 directionality data)")
    if isinstance(contrast, pd.DataFrame) and not contrast.empty \
            and "eur_pctneg_advantage" in contrast.columns:
        add("")
        add(f"  Top sets with EUR-specific repression (higher %neg in {prim}):")
        add(f"  {'Set':<34}" + "".join(f"{t:>9}" for t in traits if t in contrast.columns)
            + f"{'EURadv':>9}")
        for s, r in contrast.head(15).iterrows():
            line = f"  {str(s)[:32]:<34}"
            for t in traits:
                if t in contrast.columns: line += f"{f(r[t],1):>9}"
            line += f"{f(r['eur_pctneg_advantage'],1):>9}"
            add(line)
    add("")

    # ---- A6 ----
    add(bar); add(f"  [6]  {prim.upper()}-DOMINANT vs SHARED SET RANKING"); add(bar)
    ed = R.get("a6", pd.DataFrame())
    if isinstance(ed, pd.DataFrame) and not ed.empty:
        dom = ed[ed["EUR_DOMINANT"] == True]
        add(f"  Sets evaluated                    : {len(ed)}")
        add(f"  {prim}-DOMINANT sets (dNES>{cfg['dnes_dom']} & p<{a} in {prim}, "
            f"weak elsewhere): {len(dom)}")
        add("")
        add(f"  Top {TN} by {prim} dNES advantage:")
        add(f"  {'Set':<34}{'dNES_eur':>10}{'dNES_oth':>10}{'EURadv':>9}"
            f"{'p_eur':>9}{'DOM':>5}")
        for _, r in ed.head(TN).iterrows():
            add(f"  {str(r['Set1'])[:32]:<34}{f(r['dNES_eur']):>10}"
                f"{f(r['dNES_nonEUR_mean']):>10}{f(r['eur_dNES_advantage']):>9}"
                f"{f(r['GSEA_p1_eur'],4):>9}{'YES' if r['EUR_DOMINANT'] else '':>5}")
        if not dom.empty:
            add("")
            add(f"  >> {prim}-DOMINANT signals for EUR-first validation:")
            for _, r in dom.iterrows():
                add(f"     - {r['Set1']}: dNES_eur={f(r['dNES_eur'])}, "
                    f"p={f(r['GSEA_p1_eur'],4)}")
    else:
        add("  (no MASTER head-to-head data)")
    add("")

    # ---- A7 ----
    add(bar); add("  [7]  CROSS-ANCESTRY dNES HETEROGENEITY"); add(bar)
    het = R.get("a7", pd.DataFrame())
    if isinstance(het, pd.DataFrame) and not het.empty:
        multi = het[het["n_anc"] >= 2]
        add(f"  Pairwise comparisons (Set1 x Set2): {len(het)}")
        add(f"  Comparisons in >=2 ancestries      : {len(multi)}")
        add("")
        add(f"  Top {TN} MOST heterogeneous (largest dNES range across ancestries):")
        dcols = [c for c in het.columns if c.startswith("dNES_") and c not in
                 ("dNES_mean", "dNES_std", "dNES_range")]
        add(f"  {'Set1':<20}{'Set2':<16}" + "".join(f"{c.replace('dNES_',''):>9}" for c in dcols)
            + f"{'mean':>8}{'range':>8}")
        for _, r in het.head(TN).iterrows():
            line = f"  {str(r['Set1'])[:18]:<20}{str(r['Set2'])[:14]:<16}"
            for c in dcols: line += f"{f(r[c],2):>9}"
            line += f"{f(r['dNES_mean'],2):>8}{f(r['dNES_range'],2):>8}"
            add(line)
        add("")
        stable = het[het["dNES_range"].notna()].tail(5)
        add("  >> Most STABLE differential signals (smallest range):")
        for _, r in stable.iterrows():
            add(f"     - {r['Set1']} vs {r['Set2']}: range={f(r['dNES_range'],3)}, "
                f"mean={f(r['dNES_mean'],3)}")
    else:
        add("  (no MASTER head-to-head data)")
    add("")

    # ---- Actionable A ----
    add(bar); add("  [A]  ACTIONABLE — CORE-GENE VALIDATION PREP"); add(bar)
    core = R.get("A_core", []); core_hubs = R.get("A_hubs", [])
    core_tbl = R.get("A_tbl", pd.DataFrame())
    add(f"  Core leading-edge genes (present in ALL {len(traits)} ancestries): {len(core)}")
    add(f"  Universal hubs among core genes: {len(core_hubs)} "
        f"({', '.join(core_hubs) if core_hubs else 'none'})")
    add(f"  Files: stage5_core_genes_all_ancestries.txt, stage5_universal_hubs.txt")
    if isinstance(core_tbl, pd.DataFrame) and not core_tbl.empty:
        add("")
        add(f"  Top {TN} core genes by mean max|Z| (validation priority):")
        mcols = [c for c in core_tbl.columns if c.startswith("maxabs_")]
        add(f"  {'Gene':<12}{'hub':>5}" + "".join(f"{c.replace('maxabs_',''):>10}" for c in mcols)
            + f"{'meanMax':>9}")
        for _, r in core_tbl.head(TN).iterrows():
            line = f"  {str(r['gene']):<12}{'Y' if r['is_hub'] else '':>5}"
            for c in mcols: line += f"{f(r[c],2):>10}"
            line += f"{f(r['mean_maxabs'],2):>9}"
            add(line)
    add("")
    add("  Suggested external validation (no new pipeline run required):")
    add("    - Overlap core list with latest BIP GWAS loci (MAGMA/FUMA/PGC).")
    add("    - Check GTEx/BrainSpan expression + eQTL colocalization.")
    add("    - Re-run core list through Enrichr / g:Profiler.")
    add("    - Literature review on universal hubs + 'bipolar'/'Polycomb'.")
    add("")

    # ---- Actionable B ----
    add(bar); add("  [B]  ACTIONABLE — GABA / GLUTAMATERGIC / Ca SUBNETWORK (CGR)"); add(bar)
    subnet = R.get("B_subnet", []); sub_tbl = R.get("B_tbl", pd.DataFrame())
    add(f"  Core genes in GABA/Glut/Ca/synaptic subnetwork: {len(subnet)}")
    add(f"  File: stage5_gaba_glut_ca_subnetwork.txt")
    if isinstance(sub_tbl, pd.DataFrame) and not sub_tbl.empty:
        add("")
        for fam, g in sub_tbl.groupby("family"):
            add(f"  [{fam}] ({len(g)}): {', '.join(g['gene'].tolist())}")
        add("")
        add("  Per-gene ancestry |Z| (CGR priority subnetwork):")
        mcols = [c for c in sub_tbl.columns if c.startswith("maxabs_")]
        add(f"  {'Gene':<12}{'family':<18}" + "".join(f"{c.replace('maxabs_',''):>10}" for c in mcols)
            + f"{'meanMax':>9}")
        for _, r in sub_tbl.iterrows():
            line = f"  {str(r['gene']):<12}{str(r['family'])[:16]:<18}"
            for c in mcols: line += f"{f(r[c],2):>10}"
            line += f"{f(r['mean_maxabs'],2):>9}"
            add(line)
    add("")
    add("  CGR mechanistic angle: these core genes sit at the intersection of")
    add("  Polycomb-mediated synaptic identity maintenance and excitatory/inhibitory")
    add("  balance. GABRA6/GABRB2/CACNA1B are prime candidates for glutamatergic")
    add("  augmentation strategies in treatment-resistant BIP.")
    add("")

    # ---- HEADLINE ----
    add(bar); add("  STAGE 5 HEADLINE TAKEAWAYS (DIFFERENTIAL)"); add(bar)
    try:
        if isinstance(d1, pd.DataFrame) and not d1.empty:
            t5 = d1.head(3)["Set"].tolist()
            add(f"  - Most EUR-|Z|-advantaged Polycomb sets: {', '.join(map(str, t5))}.")
        if per_anc.get(prim):
            add(f"  - {len(per_anc[prim])} {prim}-unique leading-edge genes flagged.")
        if isinstance(eur_high, pd.DataFrame) and not eur_high.empty:
            add(f"  - EUR-high hubs: {', '.join(eur_high['gene'].tolist())}.")
        if isinstance(ed, pd.DataFrame) and not ed.empty:
            dom = ed[ed["EUR_DOMINANT"] == True]["Set1"].tolist()
            add(f"  - {prim}-dominant sets: "
                f"{', '.join(map(str, dom[:6])) if dom else 'none'}.")
        if core_hubs:
            add(f"  - Universal hubs for validation: {', '.join(core_hubs)}.")
        if subnet:
            add(f"  - CGR GABA/Glut/Ca subnetwork ({len(subnet)}): "
                f"{', '.join(subnet[:10])}{' ...' if len(subnet)>10 else ''}.")
    except Exception as e:
        add(f"  (headline synthesis error: {e})")
    add(""); add(bar)

    summ = "\n".join(L)
    path = os.path.join(cfg["out"], "STAGE5_DETAILED_SUMMARY.txt")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(summ)
    print(f"\n[✓] Detailed summary → {path}")
    return summ


# ===========================================================================
#  MASTER DRIVER
# ===========================================================================
def run_stage5(cfg=CFG):
    os.makedirs(cfg["out"], exist_ok=True)
    print("=" * 72)
    print("  STAGE 5 — TRANSANCESTRY DIFFERENTIAL INTEGRATION")
    print(f"  Ancestries: {cfg['traits']}  (primary={cfg['primary']})")
    print(f"  Output    : {cfg['out']}")
    print("=" * 72)

    print("\n[load] reading Stage 1-4 outputs ...")
    D = load_all(cfg)
    R = {}

    steps = [
        ("a1", "EUR-vs-nonEUR |Z| differential",  lambda: a1_zdiff(D, cfg)),
        ("a2", "ancestry-unique LE genes",        lambda: a2_unique_genes(D, cfg)),
        ("a3", "differential hub influence",      lambda: a3_hub_diff(D, cfg)),
        ("a4", "ancestry-stratified drugs",       lambda: a4_drug_strat(D, cfg)),
        ("a5", "differential repression",         lambda: a5_repression_diff(D, cfg)),
        ("a6", "EUR-dominant set ranking",        lambda: a6_eur_dominant(D, cfg)),
        ("a7", "dNES heterogeneity",              lambda: a7_heterogeneity(D, cfg)),
        ("A",  "core-gene validation prep",       lambda: actA_core(D, cfg)),
    ]

    for key, name, fn in steps:
        print(f"\n[{key}] {name} ...")
        try:
            res = fn()
            if key == "a2":
                R["a2_uniq"], R["a2_peranc"] = res
            elif key == "a3":
                R["a3_piv"], R["a3_eurhigh"] = res
            elif key == "a5":
                R["a5"], R["a5_contrast"] = res
            elif key == "A":
                R["A_core"], R["A_hubs"], R["A_tbl"] = res
            else:
                R[key] = res
            print("    done.")
        except Exception as e:
            print(f"    [ERROR] {name} failed: {e}")

    # actionable B depends on A
    print("\n[B] GABA/Glut/Ca subnetwork extraction ...")
    try:
        R["B_subnet"], R["B_tbl"] = actB_gabaglut(
            R.get("A_core", []), R.get("A_tbl", pd.DataFrame()), cfg)
        print("    done.")
    except Exception as e:
        print(f"    [ERROR] subnetwork extraction failed: {e}")
        R["B_subnet"], R["B_tbl"] = [], pd.DataFrame()

    print("\n[summary] building detailed report ...")
    R["summary_text"] = build_summary(cfg, R)

    try:
        with open(os.path.join(cfg["out"], "stage5_manifest.json"), "w") as fh:
            json.dump({"generated_at": datetime.now().isoformat(timespec="seconds"),
                       "traits": cfg["traits"], "primary": cfg["primary"],
                       "outputs": sorted(os.listdir(cfg["out"]))}, fh, indent=2)
    except Exception as e:
        print(f"    [warn] manifest failed: {e}")

    print("\n" + "=" * 72)
    print(f"  STAGE 5 COMPLETE → {os.path.abspath(cfg['out'])}")
    print("=" * 72)
    print("\n" + R["summary_text"])
    return R


# ---------------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------------
stage5_results = run_stage5(CFG)

  STAGE 5 — TRANSANCESTRY DIFFERENTIAL INTEGRATION
  Ancestries: ['bip_afr', 'bip_eas', 'bip_eur']  (primary=bip_eur)
  Output    : /content/096AvAp3_stage5

[load] reading Stage 1-4 outputs ...

[a1] EUR-vs-nonEUR |Z| differential ...
    done.

[a2] ancestry-unique LE genes ...
    done.

[a3] differential hub influence ...
    done.

[a4] ancestry-stratified drugs ...
    done.

[a5] differential repression ...
    done.

[a6] EUR-dominant set ranking ...
    done.

[a7] dNES heterogeneity ...
    done.

[A] core-gene validation prep ...
    done.

[B] GABA/Glut/Ca subnetwork extraction ...
    done.

[summary] building detailed report ...

[✓] Detailed summary → /content/096AvAp3_stage5/STAGE5_DETAILED_SUMMARY.txt

  STAGE 5 COMPLETE → /content/096AvAp3_stage5

####################################################################################
#   096AvAp3 — STAGE 5 TRANSANCESTRY DIFFERENTIAL SUMMARY
#   Generated  : 2026-07-04 15:46:36
#   Ancestries : bip_afr, bip_eas, bip_eur  

# WRANGLING


In [16]:
!apt-get install tree -y


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (908 kB/s)
Selecting previously unselected package tree.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [20]:
!tree /content -I "drive"

/content
├── 096AvAp3_compare
│   ├── bip_afr
│   │   ├── dNES_heatmap_bip_afr.png
│   │   ├── headtohead_bip_afr.csv
│   │   ├── linearmodel_bip_afr_HSA03020_RNA_POLYMERASE_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA03040_SPLICEOSOME_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04080_NEUROACTIVE_LIGAND_RECEPTOR_INTERACTION_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04142_LYSOSOME_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04216_FERROPTOSIS_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04360_AXON_GUIDANCE_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04514_CELL_ADHESION_MOLECULES_CAMS_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04520_ADHERENS_JUNCTION_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04614_RENIN_ANGIOTENSIN_SYSTEM_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04720_LONG_TERM_POTENTIATION_vs_AntiPsy_Genes.txt
│   │   ├── linearmodel_bip_afr_HSA04721_SYNAPTIC_VESICLE_CYCLE_v

### grouping

In [21]:
%%bash
# ============================================================
#  Group all 096AvAp3 pipeline results into one folder
# ============================================================

# Name of the master results folder
RESULTS_DIR="/content/096AvAp3_ALL_RESULTS"
mkdir -p "$RESULTS_DIR"

# List of all result folders produced by the pipeline stages
RESULT_FOLDERS=(
    "096AvAp3_compare"
    "096AvAp3_posthoc_bip_afr"
    "096AvAp3_posthoc_bip_eas"
    "096AvAp3_posthoc_bip_eur"
    "096AvAp3_followups_bip_afr"
    "096AvAp3_followups_bip_eas"
    "096AvAp3_followups_bip_eur"
    "096AvAp3_stage4"
    "096AvAp3_stage5"
)

echo "Collecting results into: $RESULTS_DIR"
echo "============================================================"

for folder in "${RESULT_FOLDERS[@]}"; do
    src="/content/$folder"
    if [ -d "$src" ]; then
        cp -r "$src" "$RESULTS_DIR/"
        echo "  ✓ copied  $folder"
    else
        echo "  ⚠ missing $folder (skipped)"
    fi
done

echo "============================================================"
echo "Folder counts:"
echo "  directories : $(find "$RESULTS_DIR" -type d | wc -l)"
echo "  files       : $(find "$RESULTS_DIR" -type f | wc -l)"
echo "============================================================"

# (Optional) Create a single zip archive for easy download / backup
cd /content
zip -r -q "096AvAp3_ALL_RESULTS.zip" "096AvAp3_ALL_RESULTS"
echo "Zipped → /content/096AvAp3_ALL_RESULTS.zip ($(du -h 096AvAp3_ALL_RESULTS.zip | cut -f1))"

  ✓ copied  096AvAp3_compare
  ✓ copied  096AvAp3_posthoc_bip_afr
  ✓ copied  096AvAp3_posthoc_bip_eas
  ✓ copied  096AvAp3_posthoc_bip_eur
  ✓ copied  096AvAp3_followups_bip_afr
  ✓ copied  096AvAp3_followups_bip_eas
  ✓ copied  096AvAp3_followups_bip_eur
  ✓ copied  096AvAp3_stage4
  ✓ copied  096AvAp3_stage5
Folder counts:
  directories : 37
  files       : 221
Zipped → /content/096AvAp3_ALL_RESULTS.zip (4.4M)


# Saving

In [22]:
!cp "/content/096AvAp3_ALL_RESULTS" "/content/drive/MyDrive/Dr Uccello/00_Studies/096_polycomb" -r


### The End